# IR HW1：Document Indexing（Positional Inverted Index）

**目標**：以 ClueWeb09 樣本 `07.warc.gz`（WARC 格式，31,480 篇網頁）為輸入，建立**positional inverted index**（投影片 p.2–p.5）。

## 處理流程

```text
07.warc.gz
  ↓ 1. Parser       WARC record → HTTP/HTML → (title, body) 純文字
  ↓ 2. Tokenizer    純文字 → [(token, position), ...]
  ↓ 3. Linguistic   case folding → stopword removal → (lemmatization) → stemming
  ↓ 5. Indexer      SPIMI 區塊 + K-way merge
dictionary.txt / dictionary_terms_only.txt / postings.txt / doc_map.txt
```

## Notebook 結構

| 章節 | 內容 |
|---|---|
| 0 | 匯入套件、`Config` 開關、檔案路徑 |
| 1 | Document Parser（WARC、HTTP、編碼、HTML 轉文字） |
| 2 | Tokenizer |
| 3 | Linguistic Module |
| 4 | 給 Indexer 用的 API 與中間檔格式 |
| 5 | Positional Inverted Indexer 與加分功能（trie、skip pointers、查詢、命令列開關） |
| 6 | 單元測試：只用內建的小型測試資料，不需要 ClueWeb09，數秒內完成 |
| 7 | 執行：小型範例索引、單篇文件檢視、處理完整 ClueWeb09、建立 Enhanced / Baseline 索引 |
| 8 | 輸出驗證：檢查第 7 節產生的索引檔，並與原始語料重新計算的結果比對；最後印出作業要求對照表 |

第 1–5 節只定義函式與類別；第 6 節測試它們；第 7 節才真正讀寫檔案；第 8 節驗證輸出檔。

**執行方式**：在第 0 節確認 `BASE_DIR` 與檔案路徑後，由上往下依序執行即可。第 6 節的測試若有失敗，會停在那一格，不會繼續處理整個語料。最花時間的是 7.3（約 3 分鐘）、7.4（約 3 分鐘）與 8.2（重新解析整個 WARC，約 4 分鐘），實際時間依機器而定。

## 主要設計

| 項目 | 決定 | 原因 / 可調整方式 |
|---|---|---|
| docID | 依 `response` record 在檔案中的順序編號 `1..N`；抽不出文字的頁面也照樣給 ID | docID 與 TREC-ID 永遠一一對應，對照表寫在 `doc_map.txt` |
| 索引範圍 | 預設索引 **title + 可見內文**，position 從 title 的第一個 token 起算 | title 也是頁面內容；`include_title=False` 只索引內文（投影片 p.6 "positions in the body"） |
| position | **token 序號**（不是字元位置），預設從 0 開始 | phrase query / proximity search 需要 token 距離；`pos_base=1` 可對齊投影片 p.5 的 1-based 範例 |
| 編碼 | HTTP header 的 `charset` → HTML 前 4 KB 的 `<meta charset>` → UTF-8；UTF-8 頁面中不合法的 byte 改用 cp1252 解讀（`caf\xe9` → `café`），其他無法解碼的 byte 換成 `U+FFFD` | ClueWeb09 非英文頁面保留原網站編碼；很多標成 UTF-8 的頁面其實夾雜 cp1252 字元；壞 byte 不讓程式中斷 |
| HTML 文字 | 刪除 NUL 等控制字元；沒有 `;` 的舊式 entity（`&nbsp`）照瀏覽器規則解碼；`&#146;` 這類 C1 字元依 cp1252 解讀（→ `’`）；只剩低位元組的引號亂碼（`don\x19t`）還原成 `’`；**每個 tag 都當成字的分隔** | `nbsp` 不會變成索引詞，NUL 不會截斷頁面，`don't` 不會變成 `dont`；tag 一律分隔的取捨見下方「已知限制」 |
| 數字 | 千分位去逗號（`1,000,000` → `1000000`）；小數、版本號、IP 整體保留；英數混合（`mp3`）保留；數字直接接單位時分開（`3.5mm` → `3.5`、`mm`）；型號 `v2.0.1`、`PM2.5` 整體保留 | 投影片 p.6 要求說明 digits 的處理 |
| hyphen | 預設 `split`：`state-of-the-art` → 4 個 token、各佔一個 position；可選 `join` / `keep` | 拆開後查單字也能命中，phrase query 仍成立 |
| 標點 | 一律當分隔符號丟掉；縮寫去點（`U.S.A.` → `USA`）；所有格 `'s` 去掉，其餘 apostrophe 保留（`don't`）；當成 apostrophe 用的 `` ` `` `´` `ʼ` 視同 `'`；看不見的 zero-width space、soft hyphen 直接刪除 | 投影片 p.6 要求說明 punctuation 的處理 |
| 中日文 | 每個字一個 token（character unigram），不使用斷詞器 | 不需額外字典，行為可預測 |
| 其他文字 | 字母後面的組合符號（印地文的母音符號、virama 等）屬於同一個字：`हिन्दी` 是一個 token | Python `re` 不把組合符號當成字元，否則字會被拆散 |
| Case folding | 預設開啟，全部轉小寫 | 投影片 p.7；可關閉 |
| Stopword removal | 預設開啟，使用 NLTK 英文停用字表；**在 stemming 之前**、對原始 token（不分大小寫）比對一次；移除後**不重新編 position**（保留 gap） | stemming 後才比對會誤刪內容字（`others`→`other`、`owned`→`own`、`DOS`→`do`）；保留 gap 讓 phrase / proximity 距離等於原文距離 |
| Stemming / Lemmatization | 預設 Porter stemming；WordNet lemmatization 為選用（預設關）；只處理純英文字，數字與中文不動 | 投影片 p.7；兩者都可獨立開關 |
| 排序 | dictionary 依 Python 字串比較（Unicode code point）排序；postings 依 docID 遞增；positions 遞增 | 排序後的 postings 才能做線性合併（AND、phrase） |
| 輸出格式 | `dictionary.txt`：`term, df`；`dictionary_terms_only.txt`：每行一個 term；`postings.txt`：每個 term 一行，格式同投影片 p.5；全部 UTF-8、無 BOM、LF 換行 | 投影片 p.5–p.6；單行格式方便逐行讀取與 byte offset 定位 |
| 大語料 | SPIMI：每累積固定數量的 `(term, pos)` 就排序寫成暫存區塊，最後用 K-way merge 串流寫出 | 不必把整份 postings 放進記憶體；區塊大小由 `spimi_block_limit` 控制 |
| 查詢 | 查詢詞一律經過與文件相同的 tokenizer 與 linguistic 處理（`Information` → `inform`），AND、phrase、proximity 都一樣 | 原始詞若剛好也是某個罕見的 stem（`Informationes` → `information`），直接查會得到錯誤結果 |
| 加分功能 | Trie 字典、skip pointers、phrase query、proximity search、命令列開關（`run_cli`） | 投影片 p.7 |

**已知限制與取捨**（刻意保留，不是 bug）

- `100,200,300` 符合千分位格式，會變成 `100200300`；逗號分隔的數字清單無法和千分位數字區分。
- 縮寫去點後轉小寫，`U.S.` 與代名詞 `us` 是同一個 term（大小寫折疊的一般代價）。
- 每個 tag 都分隔字，所以字中間有標記的寫法會被拆開：`<big>C</big>ORPORATE` → `C`、`ORPORATE`。反過來讓 inline tag 相連，會把 CSS 顯示成分開的選單連結黏成 `HomeAbout`，錯得更多（見 1.2 與 8.5）。
- 小數兩邊都黏著字母時維持原本的切法：`x3.5mm` → `x3`、`5mm`（只處理「數字＋單位」與「型號＋版本」兩種常見形式）。
- 沒有 token 長度上限：base64 之類的長字串（最長 751 字元）也會被索引，對索引正確性沒有影響。
- 標記嚴重損壞的頁面（例如屬性引號沒有關閉，整段 HTML 被當成屬性值）仍可能混入少量 script 文字。

另外會建立兩套索引：**Enhanced**（上表預設設定，主要輸出）與 **Baseline**（關閉所有語言處理，保留 `to`、`retrieval` 這類原始詞，方便對照投影片 p.5 的範例）。

## 給 Indexer 的介面

`iter_processed_docs(path, cfg)` 逐篇 yield `(doc_id, trec_id, terms)`：

- `doc_id`：從 1 開始的整數 docID
- `trec_id`：ClueWeb09 原始 ID，例如 `clueweb09-en0000-07-00000`
- `terms`：`[(term, pos), ...]`，`pos` 是該 token 在文件中的 token 位置（預設從 0 開始）

## 0. 匯入套件與設定

**相依套件**：Python 標準函式庫，加上

- `lxml`：解析 HTML。
- `nltk`：Porter stemmer、英文 stopword 表、WordNet lemmatizer。只有在本機還沒有 `stopwords`、`wordnet`、`omw-1.4` 資料時才會下載，之後可離線執行；沒有 WordNet 時，lemmatization（預設關閉）的測試會標成 `SKIP`。
- 版本固定為 `lxml==6.1.3`、`nltk==3.10.3`（HTML 解析結果與 lxml / libxml2 版本有關）。

**`Config`**：所有可開關的選項都集中在這個 dataclass（投影片 p.7 "able to be turned on/off by a parameter trigger"）。要改變行為時，建立新的 `Config(...)` 傳進各函式即可，例如 `Config(stemming=False)`。

| 欄位 | 預設 | 作用 |
|---|---|---|
| `include_title` | `True` | 把 `<title>` 文字放在內文前一起索引 |
| `hyphen_mode` | `"split"` | 連字號處理：`split` / `join` / `keep` |
| `pos_base` | `0` | 第一個 token 的 position（`1` 對應投影片 p.5 的範例） |
| `case_folding` | `True` | 轉小寫 |
| `stopword_removal` | `True` | 移除 NLTK 英文 stopword |
| `lemmatization` | `False` | WordNet lemmatization |
| `stemming` | `True` | Porter stemming |
| `use_trie` | `True` | 用 trie 存字典，支援精確查詢與前綴查詢 |
| `use_skip_pointers` | `True` | 在 postings list 上建 skip pointers |
| `spimi_block_limit` | `2_500_000` | SPIMI 每個區塊最多累積幾個 `(term, pos)` 才寫入磁碟 |

`Config.baseline()` 會關閉 case folding、stopword removal、lemmatization、stemming，用來建立保留原始詞的 Baseline index。

**路徑**：所有輸入輸出檔名都在這一格設定，執行前只需要確認 `BASE_DIR`（預設是 notebook 的工作目錄）。

| 變數 | 用途 |
|---|---|
| `SAMPLE_PATH` | 5 篇的小型測試檔 `test.warc.gz`（不存在時由 7.1 用第 6 節的測試資料產生） |
| `WARC_PATH` | ClueWeb09 語料 `07.warc.gz` |
| `TOKEN_OUT` / `LING_OUT` | tokenizer 與 linguistic module 的中間輸出 |
| `INDEX_OUT_DIR` 與 `DICT_OUT` 等 | 小型測試檔的索引輸出（`sample_index/`） |
| `CLUEWEB_*` | ClueWeb09 Enhanced index 的輸出 |
| `BASELINE_*` | ClueWeb09 Baseline index 的輸出 |
| `MAX_DOCS` | 只處理前幾篇（`None` = 全部） |
| `CFG` | 整份 notebook 使用的設定 |
| `USE_SPIMI` | 7.4 是否使用串流 SPIMI（`False` 時改用記憶體內建索引） |

In [1]:
%pip install lxml==6.1.3 nltk==3.10.3

Note: you may need to restart the kernel to use updated packages.


In [2]:
import argparse
import codecs
import gzip
import zlib
import hashlib
import heapq
import html
from html.entities import html5 as HTML5_ENTITIES
import math
import os
import re
import shutil
import tempfile
import time
import tracemalloc
import unicodedata
from collections import Counter, defaultdict
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional, Tuple

import lxml.html
from lxml import etree
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer



def nltk_has(resource: str) -> bool:
    """True if NLTK data `resource` (e.g. "corpora/wordnet") is installed, unpacked or as a .zip."""
    for name in (resource, resource + ".zip"):
        try:
            nltk.data.find(name)
            return True
        except LookupError:
            pass
    return False


# NLTK data used by the linguistic module; downloaded only if missing, so later runs work offline
for _resource, _package in (("corpora/stopwords", "stopwords"), ("corpora/wordnet", "wordnet"),
                            ("corpora/omw-1.4", "omw-1.4")):
    if not nltk_has(_resource):
        nltk.download(_package, quiet=True)
# lemmatization (optional, off by default) needs WordNet; without it the lemmatization tests are skipped
HAVE_WORDNET = nltk_has("corpora/wordnet")


@dataclass
class Config:
    """Every switch of the pipeline in one place (Slide 7: optional features can be turned on/off)."""

    # ---------- Parser ----------
    # prepend the <title> text to the body, so title words are indexed and counted in positions
    include_title: bool = True

    # ---------- Tokenizer ----------
    # hyphenated words such as "state-of-the-art":
    #   "split" -> state / of / the / art   (4 tokens, 4 positions)
    #   "join"  -> stateoftheart            (1 token)
    #   "keep"  -> state-of-the-art         (1 token, hyphens kept)
    hyphen_mode: str = "split"
    # position of the first token: 0 (default) or 1 (as in the Slide 5 example)
    pos_base: int = 0

    # ---------- Linguistic module (Slide 7) ----------
    # lower-case every token
    case_folding: bool = True
    # drop NLTK English stopwords (matched on the surface token, before stemming)
    stopword_removal: bool = True
    # WordNet lemma, e.g. "children" -> "child", "mice" -> "mouse"
    lemmatization: bool = False
    # Porter stem, e.g. "running" -> "run"
    stemming: bool = True

    # ---------- Indexer (Slide 7 efficiency features) ----------
    # keep the vocabulary in a trie (exact and prefix lookups)
    use_trie: bool = True
    # attach skip pointers every floor(sqrt(df)) postings
    use_skip_pointers: bool = True
    # SPIMI: (term, pos) pairs collected in memory before a block is sorted and written to disk
    spimi_block_limit: int = 2_500_000

    @classmethod
    def baseline(cls, **overrides) -> "Config":
        """Config with every linguistic step OFF (keeps stopwords such as 'to' and unstemmed words
        such as 'retrieval', as in the Slide 5 example). Keyword arguments override single fields."""
        params = dict(
            case_folding=False,
            stopword_removal=False,
            lemmatization=False,
            stemming=False,
        )
        params.update(overrides)
        return cls(**params)


# ---------- Paths: check BASE_DIR before running ----------
BASE_DIR = Path.cwd()
print(f"BASE_DIR: {BASE_DIR}")

# input corpora and intermediate outputs
SAMPLE_PATH = str(BASE_DIR / "test.warc.gz")
WARC_PATH = str(BASE_DIR / "07.warc.gz")
TOKEN_OUT = str(BASE_DIR / "tokenizer_output.txt")
LING_OUT = str(BASE_DIR / "linguistic_output.txt")

# index of the small sample file (Section 7.1)
INDEX_OUT_DIR = BASE_DIR / "sample_index"
DICT_OUT = str(INDEX_OUT_DIR / "dictionary.txt")
DICT_TERMS_OUT = str(INDEX_OUT_DIR / "dictionary_terms_only.txt")
POSTINGS_OUT = str(INDEX_OUT_DIR / "postings.txt")
DOC_MAP_OUT = str(INDEX_OUT_DIR / "doc_map.txt")

# Enhanced ClueWeb09 index (Section 7.4); its inputs default to the files above
CLUEWEB_TOK_PATH = TOKEN_OUT
CLUEWEB_LING_PATH = LING_OUT
CLUEWEB_WARC_PATH = WARC_PATH
CLUEWEB_DICT_OUT = str(BASE_DIR / "dictionary.txt")
CLUEWEB_DICT_TERMS_OUT = str(BASE_DIR / "dictionary_terms_only.txt")
CLUEWEB_POST_OUT = str(BASE_DIR / "postings.txt")
CLUEWEB_MAP_OUT = str(BASE_DIR / "doc_map.txt")

# Baseline ClueWeb09 index: raw tokens, no linguistic processing (shares doc_map.txt)
BASELINE_DICT_OUT = str(BASE_DIR / "baseline_dictionary.txt")
BASELINE_DICT_TERMS_OUT = str(BASE_DIR / "baseline_dictionary_terms_only.txt")
BASELINE_POST_OUT = str(BASE_DIR / "baseline_postings.txt")

# None = whole file; a small number gives a quick trial run
MAX_DOCS = None
# configuration used by every cell below
CFG = Config()
# True = streaming SPIMI to disk; False = build in memory, then save
USE_SPIMI = True

BASE_DIR: e:\ir_hw\claude_hw1


## 1. Document Parser

輸入是 ClueWeb09 的 WARC 檔（投影片 p.4）。這一節把它轉成一篇篇 `Document(doc_id, trec_id, url, title, body)`。

### 1.1 WARC reader

ClueWeb09 使用 WARC/0.18 格式，整個檔案就是一連串 record 接在一起：

```text
WARC/0.18                      ← record 的第一行
WARC-Type: response            ← WARC header（每行一個 key: value）
WARC-TREC-ID: clueweb09-en0000-07-00000
Content-Length: 1096           ← 後面 content 的 byte 數
                               ← 空行：header 結束
HTTP/1.1 200 OK ... <html>...  ← content，長度剛好是 Content-Length 個 bytes
```

`iter_warc_records(path)` 是一個 generator，一次只讀一個 record，所以不論檔案多大，記憶體裡只有一個 record：

1. 讀檔案開頭 2 個 bytes 判斷是否為 gzip（gzip 檔固定以 `1f 8b` 開頭），所以 `.warc` 與 `.warc.gz` 都能直接讀。
2. 逐行讀取，以 `WARC/` 開頭的行代表新 record 開始；record 之間多餘的空行會被跳過。
3. 繼續讀 header，直到空行，存成 dict。
4. 依 `Content-Length` 一次讀出 content。

**容錯設計**

| 情況 | 處理方式 |
|---|---|
| 缺少 `Content-Length`，或值不是數字、是負數 | 當成 0（content 為空），下一輪再從下一個 `WARC/` 行繼續。負數若直接傳給 `read()` 會把檔案剩下的內容全部讀掉 |
| gzip 檔被截斷或損壞 | 已經讀完的完整 record 照常輸出，然後安靜地停止，不拋出例外 |

In [3]:
@dataclass
class WarcRecord:
    """One WARC record: its header fields and its raw content bytes."""
    # WARC header fields, e.g. {"WARC-Type": "response", "WARC-TREC-ID": "...", ...}
    headers: dict
    # raw content; for a "response" record this is the HTTP header + HTML
    content: bytes


def iter_warc_records(path: str) -> Iterator[WarcRecord]:
    """Stream WARC records one at a time from a .warc or .warc.gz file (gzip is auto-detected)."""
    # gzip files always start with the magic bytes 1f 8b
    with open(path, "rb") as raw:
        is_gzip = raw.read(2) == b"\x1f\x8b"

    opener = gzip.open if is_gzip else open
    with opener(path, "rb") as f:
        try:
            # readline() keeps the newline and returns b"" only at end of file
            line = f.readline()
            while line:
                # look for the first line of the next record; blank lines between records are skipped
                if not line.startswith(b"WARC/"):
                    line = f.readline()
                    continue
                # header lines "Key: Value" until the first blank line (or EOF)
                headers = {}
                while True:
                    line = f.readline()
                    if not line or line in (b"\r\n", b"\n"):
                        break
                    k, _, v = line.decode("utf-8", "replace").partition(":")
                    headers[k.strip()] = v.strip()
                # A missing, non-numeric or negative Content-Length counts as 0 (empty content):
                # int() would raise ValueError, and f.read(-n) would swallow the rest of the file.
                # The outer loop then resyncs on the next "WARC/" line.
                try:
                    length = max(0, int(headers.get("Content-Length", "0") or 0))
                except ValueError:
                    length = 0
                content = f.read(length)
                yield WarcRecord(headers, content)
                line = f.readline()
        except (EOFError, zlib.error):
            # truncated or corrupted gzip stream: every complete record was already yielded, so just stop
            return

### 1.2 HTTP、編碼、HTML 轉純文字

`response` record 的 content 是「HTTP header + 空行 + HTML」，依序經過三個步驟：

1. **`split_http`：切開 HTTP header 與 HTML。** ClueWeb09 的 HTTP header 用 `\n` 換行，但 HTML 內文常用 `\r\n`。如果只找 `\r\n\r\n`，可能在 HTML 中間才找到，導致大半頁被當成 header。因此兩種空行都找，**取最早出現的那一個**。找不到空行時，整段都當成 HTML。

2. **`_detect_charset`：決定編碼。** 依序查看 HTTP header 的 `charset=`、HTML 前 4 KB 的 `<meta charset>`，都沒有就用 UTF-8（ClueWeb09 的非英文頁面保留原網站的編碼）。
   - 常見編碼換成可解更多字的超集：`gb2312`/`gbk` → `gb18030`、`big5` → `cp950`、`iso-8859-1`/`ascii` → `cp1252`（瀏覽器實際上也這樣處理）。
   - 寫錯或 Python 不認得的編碼名稱會被略過，繼續找下一個來源。
   - 編碼是 UTF-8 時（宣告的或預設的），不合法的 byte 逐一改用 cp1252 解讀：很多頁面標成 UTF-8，內容卻夾雜 cp1252 的 `é`、`ü`、不換行空白。合法的 UTF-8 序列不受影響。
   - 其他編碼使用 `errors="replace"`：無法解碼的 byte 會變成 `U+FFFD`，不會讓程式中斷。

3. **`html_to_text`：HTML → `(title, body)`。**
   - 解析前先清理原始碼（`_clean_source`）：刪除 NUL 等控制字元（libxml2 可能在 NUL 停止讀取，整頁只剩開頭）；把沒有 `;` 的舊式 entity 補上 `;`（`&nbsp<input>` 在瀏覽器裡是空白，否則 `nbsp` 會被當成一個字）。
   - 用 `lxml` 解析，刪除 `<script>`、`<style>`、`<noscript>` 與 HTML 註解，這些都不是可見文字。
   - 取第一個 `<title>` 的文字當 title。
   - body = **整份文件扣掉 `<head>`**，而不是只讀 `<body>`：有些網頁 `</body>` 很早就出現，後面的內容 lxml 不會放進 `<body>`。因為 `<head>` 已經刪除，body 不會重複包含 title。
   - lxml 解析失敗，或抽出的文字是空的（例如 `</title>` 寫錯，整頁被吃進 title），就改用 `_regex_text`：直接用 regex 去掉註解、script/style、`<head>` 與所有 tag，再解碼 HTML entity。
   - **每個 tag 都當成字的分隔**，inline tag 也一樣。曾試過讓 `<b>`、`<a>` 等 inline tag 內的文字直接相連（`<big>C</big>ORPORATE` → `CORPORATE`），但網頁常用 CSS 把相鄰的連結、標籤顯示成分開的字，相連反而產生 `HomeAbout`、`Hotelsbeta` 這類錯字，數量遠多於修好的字（8.5 用獨立的解析器量測），所以保留分隔。
   - 被破壞的 Word 引號只剩 Unicode 的低位元組，成了控制字元（`don\x19t`、`\x1c…\x1d`）。刪除控制字元之前先把它們還原成 `’ “ ” ‘ – —`，否則 `don't` 會變成 `dont`（8.5 找到的問題）。
   - U+0080–U+009F 的 C1 控制字元（`&#146;` 解析後就是 U+0092）依 cp1252 換成實際的字元（U+2019 `’`），所以 `client&#146;s` 不會被拆成 `client` 與 `s`。
   - 最後把連續空白壓成一個空格。

In [4]:
# captures the encoding name after "charset=", e.g. "big5" in 'charset="big5"'
_CHARSET_RE = re.compile(rb"""charset\s*=\s*["']?\s*([A-Za-z0-9_\-:.]+)""", re.I)
# declared charset -> codec that decodes a superset of it (fewer garbled characters)
_CHARSET_ALIASES = {
    # Simplified Chinese: gb18030 covers gb2312 and gbk
    "gb2312": "gb18030", "gbk": "gb18030",
    # Traditional Chinese: Microsoft's Big5 superset
    "big5": "cp950",
    # non-standard name for Shift-JIS
    "x-sjis": "shift_jis",
    # browsers decode latin-1 / ascii pages as cp1252, and such pages often contain cp1252 bytes
    "iso-8859-1": "cp1252", "latin1": "cp1252",
    "us-ascii": "cp1252", "ascii": "cp1252",
}


def split_http(content: bytes) -> Tuple[bytes, bytes]:
    """Split an HTTP response into (http_header, payload) at the EARLIEST blank line.

    Both b"\\r\\n\\r\\n" and b"\\n\\n" are searched: ClueWeb09 headers end with "\\n\\n" while the HTML
    often contains "\\r\\n\\r\\n", so searching only one form could cut the page in the middle.
    """
    # (start index, separator) for every blank-line form that occurs
    hits = [(i, sep)
            for sep in (b"\r\n\r\n", b"\n\n")
            if (i := content.find(sep)) != -1]
    # no blank line at all -> no header, everything is payload
    if not hits:
        return b"", content
    i, sep = min(hits)
    return content[:i], content[i + len(sep):]


def _detect_charset(http_header: bytes, html_bytes: bytes) -> str:
    """Return a Python codec name: HTTP header charset, else <meta> charset in the first 4 KB, else utf-8."""
    for blob in (http_header, html_bytes[:4096]):
        m = _CHARSET_RE.search(blob)
        if m:
            cs = m.group(1).decode("ascii", "ignore").lower()
            cs = _CHARSET_ALIASES.get(cs, cs)
            # accept the name only if Python knows the codec (pages contain typos); otherwise keep looking
            try:
                "".encode(cs)
                return cs
            except LookupError:
                pass
    return "utf-8"


# leading <?xml ... ?> declaration (lxml rejects a str input that carries an encoding declaration)
_XML_DECL_RE = re.compile(r"^\s*<\?xml[^>]*\?>", re.I)
# control characters that are never visible text (tab, LF, FF, CR are kept); libxml2 may stop reading at NUL
_CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0e-\x1f]")
# HTML5 "legacy" entities that browsers decode even without the closing ';' ("Email: &nbsp<input>");
# libxml2 does not, so the name would be indexed as a word ("nbsp"). Not touched when followed by a letter,
# digit, '=' or ';' (e.g. "&copyright", "?a=1&amp=2").
_LEGACY_ENTITY_RE = re.compile(
    r"&(" + "|".join(sorted((n for n in HTML5_ENTITIES if not n.endswith(";")), key=len, reverse=True))
    + r")(?![A-Za-z0-9;=])")
# C1 control characters U+0080..U+009F -> the cp1252 character a browser shows instead (U+0092 -> U+2019 '’').
# They come from raw cp1252 bytes (libxml2 silently drops them: "don\x92t" -> "dont") and from references
# such as "&#146;" (lxml keeps U+0092, which splits "client&#146;s" into "client" + "s"). The 5 code points
# cp1252 leaves undefined stay unchanged (they are not word characters).
_C1_TO_CP1252 = {}
for _b in range(0x80, 0xA0):
    try:
        _C1_TO_CP1252[_b] = bytes([_b]).decode("cp1252")
    except UnicodeDecodeError:
        pass
# Mojibake from broken Word -> HTML converters: only the low byte of U+2018/2019/201C/201D/2013/2014 survives
# as a control character ("don\x19t"). Restored before control characters are dropped; otherwise the
# apostrophe vanishes and "don't" becomes "dont".
_LOW_BYTE_PUNCT = {0x13: "\u2013", 0x14: "\u2014", 0x18: "\u2018", 0x19: "\u2019", 0x1C: "\u201c", 0x1D: "\u201d"}
# patterns used only by the regex fallback
_TAG_RE = re.compile(r"<[^>]+>")
_SCRIPT_RE = re.compile(r"<(script|style|noscript)\b.*?</\1\s*>", re.I | re.S)
_TITLE_RE = re.compile(r"<title[^>]*>(.*?)</title\s*>", re.I | re.S)
_HEAD_RE = re.compile(r"<head\b.*?</head\s*>", re.I | re.S)
_COMMENT_RE = re.compile(r"<!--.*?-->", re.S)


def _clean_source(html_str: str) -> str:
    """Before parsing: drop the <?xml?> declaration, restore low-byte punctuation mojibake, drop the other control
    characters, map raw C1 characters through cp1252, close legacy entities ("&nbsp" -> "&nbsp;")."""
    html_str = _XML_DECL_RE.sub("", html_str).translate(_LOW_BYTE_PUNCT)
    html_str = _CONTROL_RE.sub("", html_str).translate(_C1_TO_CP1252)
    return _LEGACY_ENTITY_RE.sub(r"&\1;", html_str)


def _regex_text(html_str: str) -> Tuple[str, str]:
    """Fallback for pages lxml cannot handle: strip comments, script/style, <head> and tags with regex."""
    m = _TITLE_RE.search(html_str)
    title = html.unescape(_TAG_RE.sub(" ", m.group(1))) if m else ""
    body = _COMMENT_RE.sub(" ", html_str)
    body = _SCRIPT_RE.sub(" ", body)
    body = _HEAD_RE.sub(" ", body)
    return title, html.unescape(_TAG_RE.sub(" ", body))


def html_to_text(html_str: str) -> Tuple[str, str]:
    """Return (title, body_text) with whitespace collapsed.

    body = visible text of the whole document minus <head> (not only <body>), because some pages close
    </body> early and lxml then places the remaining content outside <body>. Every tag separates words:
    joining text across inline tags would fix "<big>C</big>ORPORATE" but glue far more words that CSS shows
    apart ("Home</a><a>About" -> "HomeAbout"); Section 8.5 measures both effects.
    """
    html_str = _clean_source(html_str)
    try:
        root = lxml.html.document_fromstring(html_str)
        # drop non-visible nodes; tail text after them is kept by drop_tree()
        for el in root.xpath("//script|//style|//noscript|//comment()"):
            # comments outside <html> have no parent and cannot be dropped
            if el.getparent() is not None:
                el.drop_tree()
        title = root.findtext(".//title") or ""
        # remove <head>, so the title is not repeated in the body
        for head in root.xpath("//head"):
            if head.getparent() is not None:
                head.drop_tree()
        text = " ".join(root.itertext())
        # nothing extracted (e.g. a broken </title> swallowed the page): use the regex fallback,
        # but keep lxml's title if it found one
        if not text.strip():
            title2, text = _regex_text(html_str)
            title = title or title2
    except (etree.ParserError, ValueError):
        # lxml gave up on this document entirely
        title, text = _regex_text(html_str)
    # C1 characters that lxml created from references ("&#146;" -> U+0092); before collapsing whitespace,
    # because U+0085 would otherwise count as whitespace
    title, text = title.translate(_C1_TO_CP1252), text.translate(_C1_TO_CP1252)
    return " ".join(title.split()), " ".join(text.split())

### 1.3 `parse_warc`：把 record 轉成 `Document`

- 只處理 `WARC-Type: response`；`warcinfo` 這類描述檔案本身的 record 直接跳過。
- 每個 response 依出現順序給一個整數 `doc_id`（1, 2, 3, …）。**頁面抽不出文字也照樣給 ID**，因此 docID 與 TREC-ID 永遠一一對應，`doc_map.txt` 的行數等於檔案中 response record 的數量。
- 每個 record 依序執行：`split_http` → `_detect_charset` → 解碼成 str（`errors="replace"`）→ `html_to_text`。
- `parse_warc(path)` 是 generator，每次 yield 一個 `Document`。

In [5]:
@dataclass
class Document:
    """One web page after parsing."""
    # sequential integer id, starting at 1
    doc_id: int
    # original ClueWeb09 id, e.g. "clueweb09-en0000-07-00000"
    trec_id: str
    # WARC-Target-URI of the page
    url: str
    # text inside <title>
    title: str
    # visible text of the page (everything except <head>)
    body: str


def _cp1252_fallback(err: UnicodeDecodeError) -> Tuple[str, int]:
    """Decode error handler: bytes that are not valid UTF-8 are read as cp1252 (U+FFFD where cp1252 is undefined)."""
    return err.object[err.start:err.end].decode("cp1252", errors="replace"), err.end


# Many pages labelled (or defaulted to) UTF-8 contain cp1252 bytes ("caf\xe9"). Only the invalid bytes are read
# as cp1252, so valid UTF-8 sequences stay correct and "café" is not cut into "caf" + U+FFFD.
codecs.register_error("cp1252_fallback", _cp1252_fallback)


def parse_warc(path: str) -> Iterator[Document]:
    """Yield one Document per WARC 'response' record, numbered 1, 2, 3, ... in file order."""
    doc_id = 0
    for rec in iter_warc_records(path):
        # warcinfo and other record types are not web pages
        if rec.headers.get("WARC-Type") != "response":
            continue

        # every response gets an id, even if no text can be extracted, so ids stay aligned with TREC ids
        doc_id += 1
        http_header, payload = split_http(rec.content)
        charset = _detect_charset(http_header, payload)
        # utf-8: invalid bytes are read as cp1252; other charsets: undecodable bytes become U+FFFD instead of raising
        errors = "cp1252_fallback" if codecs.lookup(charset).name == "utf-8" else "replace"
        try:
            html_str = payload.decode(charset, errors=errors)
        except LookupError:
            # cannot happen after _detect_charset validated the codec, kept as a safety net
            html_str = payload.decode("utf-8", errors="replace")
        # an empty payload gives an empty document
        title, body = html_to_text(html_str) if html_str.strip() else ("", "")
        yield Document(doc_id=doc_id,
                       trec_id=rec.headers.get("WARC-TREC-ID", ""),
                       url=rec.headers.get("WARC-Target-URI", ""),
                       title=title, body=body)

## 2. Tokenizer

投影片 p.6 要求說明 tokenizer 如何處理數字、連字號與標點符號，規則如下。

**步驟 1：Unicode 正規化（`_normalize`）**

- 刪除瀏覽器不會顯示的字元：zero-width space、word joiner、BOM、soft hyphen（`Shuff\u200ble` 顯示為 `Shuffle`，否則會被拆成 `Shuff`、`le`）。
- 夾在兩個字母之間、被當成 apostrophe 用的 `` ` ``、`´`、`ʼ` 換成 `'`（``Let`s`` → `Let's`、`don´t` → `don't`），否則會拆出單獨的 `s`、`t`。這一步在 NFKC 之前，因為 NFKC 會把 `´` 變成空白加組合符號。
- NFKC：全形字母、數字轉半形（`ＩＲ` → `IR`），連字拆開（`ﬁ` → `fi`）。
- 彎引號 `’` `‘` 換成 `'`，Unicode 連字號 `‐` `‑` 換成 `-`，後面的規則只需處理一種寫法。

**步驟 2：用一條 regex 由左到右掃描**，依比對到的 named group 套用規則：

| 規則 | 範例 | 輸出 |
|---|---|---|
| `ACRONYM` 縮寫（2 個以上「單一字母 + `.`」） | `U.S.A.`、`e.g.` | `USA`、`eg`（去掉點） |
| `NUMBER` 千分位數字 | `1,000,000` | `1000000`（去掉逗號） |
| `NUMBER` 小數 / 版本 / IP | `3.14`、`2.0.1`、`192.168.0.1` | 原樣保留，不拆開 |
| `NUMUNIT` 數字後面直接接字母 | `3.5mm`、`2.4GHz`、`1,000th` | `3.5` + `mm`、`2.4` + `GHz`、`1000` + `th`（數字與單位分開，數字本身不被切斷） |
| `VERSION` 字母 + 數字 + `.數字` | `v2.0.1`、`PM2.5`、`web2.0` | 原樣保留；`.` 前面一定要有數字（`No.1` → `No`、`1`），後面不能再黏字（`v2.0.1beta` 改用其他規則） |
| `WORD` 字母、數字或英數混合 | `mp3`、`covid19`、`2009`、`café` | 原樣保留 |
| `WORD` 含連字號 | `state-of-the-art`、`555-1234` | 依 `hyphen_mode`：`split`（預設）/ `join` / `keep` |
| `WORD` 含 apostrophe | `don't`、`O'Neill`、`John's` | `don't`、`O'Neill`、`John`（只去掉所有格 `'s`） |
| `WORD` 含組合符號 | `हिन्दी` | `हिन्दी`（字母後面的組合符號，Unicode 類別 M*，屬於同一個字） |
| `CJK` 中日文字 | `資訊檢索` | `資`、`訊`、`檢`、`索`（character unigram，不用斷詞器） |
| 其他標點與符號 | `, . ! ? ( ) / & _ …` | 不會被比對到，等於當成分隔符號丟棄 |

- alternation 由左到右嘗試，所以 `ACRONYM`、`NUMBER`、`NUMUNIT`、`VERSION` 放在 `WORD` 前面，`3.14` 才不會被切成 `3` 和 `14`，`3.5mm` 才不會變成 `3` 和 `5mm`。
- `NUMUNIT` 只在數字後面緊接**字母**時成立，所以逗號分隔的數字清單 `8,1995` 仍是 `8`、`1995`，不會被併成一個數。
- 句尾的點不屬於數字：`in 2009.` → `in`、`2009`。
- 底線 `_` 也是分隔符號：`under_score` → `under`、`score`。

**步驟 3：position**

每輸出一個 token，position 加 1，從 `cfg.pos_base`（預設 0）開始。連字號 `split` 出來的每一段各佔一個 position，所以 `a b-c d` 的 position 是 `0, 1, 2, 3`。position 是 token 序號而不是字元位置，phrase query 與 proximity search 直接用它計算距離（投影片 p.6）。

`trace=True` 時另外回傳每次比對的明細（原始字串、使用的規則、輸出的 token、分到的 position），供 7.2 的單篇檢視使用。

In [6]:
# kana + CJK ideograph ranges (used inside [...])
_CJK = r"\u3040-\u30ff\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff"
# one "word character": any Unicode letter or digit, except '_' and CJK (CJK is handled one char at a time)
_W = rf"[^\W_{_CJK}]"
# one letter (a word character that is not a digit)
_L = rf"[^\W\d_{_CJK}]"


def _char_ranges(pred) -> str:
    """Regex character-class body (\\uXXXX-\\uYYYY...) of every BMP character c with pred(c)."""
    out, start = [], None
    for cp in range(0x10001):
        hit = cp < 0x10000 and pred(chr(cp))
        if hit and start is None:
            start = cp
        elif not hit and start is not None:
            out.append(f"\\u{start:04x}" + (f"-\\u{cp - 1:04x}" if cp - 1 > start else ""))
            start = None
    return "".join(out)


# combining marks (Unicode category M*), e.g. Devanagari vowel signs and virama, Arabic / Hebrew points.
# re does not count them as word characters, so without them "हिन्दी" would fall apart into "ह न द".
_MARKS = _char_ranges(lambda c: unicodedata.category(c).startswith("M"))
# a word character or a combining mark (used after the first character of a word)
_WM = rf"(?:{_W}|[{_MARKS}])"
# Alternatives are tried left to right, so ACRONYM and NUMBER must come before WORD.
# re.X (verbose mode) ignores the whitespace and comments inside the pattern.
_TOKEN_RE = re.compile(rf"""
    # 2+ single letters each followed by '.', e.g. U.S.A.
    (?P<ACRONYM>(?:[A-Za-z]\.){{2,}})
  # NUMBER group: either of the two number forms below
  | (?P<NUMBER>
        # thousands-grouped number: 1,000,000 or 1,234.5
        \d{{1,3}}(?:,\d{{3}})+(?:\.\d+)?(?!{_W})
      # dotted number: 3.14, 2.0.1, 192.168.0.1
      | \d+(?:\.\d+)+(?!{_W})
    # end of the NUMBER group
    )
  # a number glued to letters ("3.5mm", "1,000th"): only the number is matched, the letters become the next
  # WORD. The next character must be a letter, so a list such as "8,1995" is never joined into one number.
  | (?P<NUMUNIT>
        \d{{1,3}}(?:,\d{{3}})+(?:\.\d+)?(?={_L})
      | \d+(?:\.\d+)+(?={_L})
    )
  # version / model number: letters, digits, then .digits ("v2.0.1", "PM2.5", "web2.0"). Digits are required
  # before the dot ("No.1" stays No + 1), and nothing may be glued after it ("v2.0.1beta" uses the other rules).
  | (?P<VERSION>{_L}+\d+(?:\.\d+)+(?![.']?{_W}))
  # alphanumeric run (combining marks allowed after the first character), optionally joined by - or '
  | (?P<WORD>{_W}{_WM}*(?:[-']{_WM}+)*)
  # a single CJK / kana character
  | (?P<CJK>[{_CJK}])
""", re.X)
# whole-string check for thousands grouping
_THOUSANDS_RE = re.compile(r"^\d{1,3}(?:,\d{3})+(?:\.\d+)?$")


@dataclass
class Token:
    """A token string and its position (token index, starting at cfg.pos_base)."""
    text: str
    pos: int


# characters a browser does not show: zero-width space, word joiner, BOM, soft hyphen ("Shuff\u200ble" is "Shuffle")
_INVISIBLE_RE = re.compile(r"[\u200b\u2060\ufeff\u00ad]")
# apostrophe look-alikes between two letters ("Let`s", "don\u00b4t", "don\u02bct") -> "'"
_APOS_LIKE_RE = re.compile(r"(?<=[^\W\d_])[\u0060\u00b4\u02bc](?=[^\W\d_])")


def _normalize(text: str) -> str:
    """Unicode clean-up before tokenizing: invisible characters dropped, apostrophe look-alikes -> ', NFKC,
    curly apostrophes -> ', Unicode hyphens -> -."""
    text = _INVISIBLE_RE.sub("", text)
    # before NFKC, which turns U+00B4 into a space + combining accent
    text = _APOS_LIKE_RE.sub("'", text)
    # full-width -> half-width, ligatures split, etc.
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u2010", "-").replace("\u2011", "-")
    return text


def _rule_outputs(kind: str, raw: str, cfg: Config) -> List[str]:
    """Turn one regex match (rule `kind`, matched text `raw`) into 0..n token strings."""
    # "U.S.A." -> "USA"
    if kind == "ACRONYM":
        return [raw.replace(".", "")]
    if kind == "NUMBER":
        # "1,000,000" -> "1000000"
        if _THOUSANDS_RE.match(raw):
            return [raw.replace(",", "")]
        # decimals, versions and IP addresses stay whole
        return [raw]
    # number in front of letters: "1,000" (before "th") -> "1000", "3.5" (before "mm") unchanged
    if kind == "NUMUNIT":
        return [raw.replace(",", "")]
    # version / model numbers stay whole: "v2.0.1", "PM2.5"
    if kind == "VERSION":
        return [raw]
    # one CJK character = one token
    if kind == "CJK":
        return [raw]
    # kind == "WORD" from here on
    # possessive: "John's" -> "John" (other apostrophes such as "don't" are kept)
    if raw.lower().endswith("'s"):
        raw = raw[:-2]
    if "-" in raw:
        # one token per part, each with its own position
        if cfg.hyphen_mode == "split":
            return [p for p in raw.split("-") if p]
        # one token without hyphens
        if cfg.hyphen_mode == "join":
            return [raw.replace("-", "")]
        # "keep" falls through and returns the word unchanged
    return [raw] if raw else []


def tokenize(text: str, cfg: Config = Config(), trace: bool = False):
    """Return list[Token]; with trace=True return (tokens, rows), one row (raw, rule, tokens, positions) per match."""
    tokens = []
    rows = []
    # next position to hand out
    pos = cfg.pos_base
    for m in _TOKEN_RE.finditer(_normalize(text)):
        # name of the matching group: ACRONYM / NUMBER / NUMUNIT / VERSION / WORD / CJK
        kind = m.lastgroup
        raw = m.group()
        outs = _rule_outputs(kind, raw, cfg)
        start = pos
        # every output token takes the next position
        for o in outs:
            tokens.append(Token(o, pos))
            pos += 1
        if trace:
            rows.append((raw, kind, outs, list(range(start, pos))))
    return (tokens, rows) if trace else tokens

## 3. Linguistic Module

輸入是 tokenizer 產生的 `Token` list，依下列**固定順序**處理。每一步都能用 `Config` 單獨開關（投影片 p.7）：

| 順序 | 步驟 | 預設 | 作法 |
|---|---|---|---|
| 1 | Case folding | 開 | 全部轉小寫 |
| 2 | Stopword removal | 開 | NLTK 英文停用字表（`nltk.corpus.stopwords.words("english")`），比對時一律轉小寫，所以關掉 case folding 時 `The` 也會被移除 |
| 3 | Lemmatization | 關 | `WordNetLemmatizer`：先當名詞還原，沒有變化再當動詞還原（`children` → `child`、`went` → `go`） |
| 4 | Stemming | 開 | `PorterStemmer`（`running` → `run`、`retrieval` → `retriev`） |

**設計決定**

- **Stopword 在 stemming 之前、只比對一次。** 停用字表列的是原文的字形，所以要拿原始 token 比對。如果在 stemming 之後再比對一次，會誤刪 stem 剛好等於停用字的內容字，例如 `others` → `other`、`owned` → `own`、`willing` → `will`、`DOS` → `do`，這些字就再也查不到。這也是 Lucene `EnglishAnalyzer` 的順序（lowercase → stop filter → stemmer）。代價是字典裡會出現少數「看起來像停用字」的 stem（例如 `AED` → `a`），查詢時 query 經過同一套處理，結果仍然一致。
- **移除 token 後不重新編 position**，會留下跳號（gap）。這樣 phrase query 與 proximity search 算出的距離就是原文中的真實距離。
- **Lemmatization 與 stemming 只處理純英文字**（字母，可含 apostrophe）。數字、中文、`mp3` 這類英數混合的 token 原樣保留。
- **關掉 case folding 時保留大小寫樣式**：先用小寫處理，再把原本的樣式套回去（`Running` → `Run`、`MICE` → `MOUSE`），全大寫縮寫如 `USA` 不會被改壞。
- 同一個字會反覆出現，所以 stemmer 與 lemmatizer 的結果用 `lru_cache` 快取。

`trace=True` 時另外回傳一個 dict，記錄每個步驟的輸出，供 7.2 的單篇檢視使用。

In [7]:
# NLTK English stopword list; frozenset gives O(1) membership tests
STOPWORDS = frozenset(stopwords.words("english"))

# shared NLTK instances
_stemmer = PorterStemmer()
_lemmatizer = WordNetLemmatizer()
# words the stemmer / lemmatizer may touch: ASCII letters, apostrophes allowed (numbers, CJK, "mp3" are skipped)
_ASCII_ALPHA_RE = re.compile(r"^[A-Za-z][A-Za-z']*$")


def _restore_case(word: str, out: str) -> str:
    """Copy the casing pattern of `word` (lower / UPPER / Title) onto the lowercase result `out`.
    Only matters when case_folding=False, e.g. "Running" -> "Run", "MICE" -> "MOUSE"."""
    if word.islower():
        return out
    if word.isupper():
        return out.upper()
    if word[0].isupper():
        return out.capitalize()
    return out


@lru_cache(maxsize=500_000)
def _lemmatize(word: str) -> str:
    """WordNet lemma of an English word (noun first, then verb), keeping its casing pattern."""
    if not _ASCII_ALPHA_RE.match(word):
        return word
    low = word.lower()
    lemma = _lemmatizer.lemmatize(low, pos="n")
    if lemma == low:
        lemma = _lemmatizer.lemmatize(low, pos="v")
    return _restore_case(word, lemma)


@lru_cache(maxsize=500_000)
def _stem(word: str) -> str:
    """Porter stem of an English word, keeping its casing pattern; numbers, CJK and mixed tokens are unchanged."""
    if not _ASCII_ALPHA_RE.match(word):
        return word
    return _restore_case(word, _stemmer.stem(word.lower()))


def linguistic(tokens: List[Token], cfg: Config = Config(), trace: bool = False):
    """Apply case folding -> stopword removal -> lemmatization -> stemming (each switchable in cfg).

    Returns list[(term, pos)]; with trace=True returns (terms, stages), where stages maps each step name
    to its output. Positions are never renumbered, so removed tokens leave gaps.
    """
    stages = {"input": [(t.text, t.pos) for t in tokens]}
    cur = stages["input"]
    if cfg.case_folding:
        cur = [(w.lower(), p) for w, p in cur]
        stages["case_folding"] = cur
    if cfg.stopword_removal:
        # matched case-insensitively on the surface token; positions are kept, so removed words leave gaps
        cur = [(w, p) for w, p in cur if w.lower() not in STOPWORDS]
        stages["stopword_removal"] = cur
    # Stopwords are removed only once, on the surface tokens above. A content word whose lemma / stem
    # happens to equal a stopword string ("others" -> "other", "owned" -> "own", "DOS" -> "do") is kept,
    # so it stays searchable (same order as Lucene's EnglishAnalyzer: lowercase -> stop filter -> stemmer).
    if cfg.lemmatization:
        cur = [(_lemmatize(w), p) for w, p in cur]
        stages["lemmatization"] = cur
    if cfg.stemming:
        cur = [(_stem(w), p) for w, p in cur]
        stages["stemming"] = cur
    return (cur, stages) if trace else cur

## 4. 給 Indexer 用的 API

| 函式 | 用途 |
|---|---|
| `process_document(doc, cfg)` | 一篇 `Document` 經過 tokenizer 與 linguistic module，回傳 `[(term, pos), ...]`。`include_title=True` 時索引的文字是 `title + " " + body` |
| `iter_processed_docs(path, cfg, max_docs)` | 串流讀取 WARC 檔，逐篇 yield `(doc_id, trec_id, terms)`；`max_docs` 可只跑前幾篇 |
| `iter_processed_file(path, cfg, max_docs, raw_tokens)` | 讀取 7.3 存下的文字檔，逐篇 yield 同樣的 tuple，讓 indexer 不必重新解析 WARC |

**中間檔格式**（`tokenizer_output.txt`、`linguistic_output.txt`）：每篇文件一行，

```text
doc_id<TAB>trec_id<TAB>term:pos term:pos ...
```

`_fmt(pairs)` 把 `[(text, pos), ...]` 轉成第三欄 `text:pos text:pos ...`（7.3 寫檔與第 6 節的測試共用）。

**`iter_processed_file` 的兩種模式**

- `cfg=None`（7.4 使用）：照檔案內容原樣輸出，不做任何處理。
- 有 `cfg`（命令列 `run_cli` 使用）：
  - 套用 `max_docs`；若檔案的 position 起點（0 或 1）與 `cfg.pos_base` 不同，整體平移。
  - 判斷檔案內是原始 token 還是已處理的 term：`raw_tokens=True` / `False` 直接指定（命令列 `--txt-input tokens` / `terms`）；`None`（預設）時，檔名含 `tok` 視為原始、含 `ling` 視為已處理，其他檔名則看第一篇文件，有大寫字母或停用字比例 ≥ 20% 就視為原始。檔名猜測可能出錯（`stokes_ling.txt` 含 `tok`），所以提供明確的開關。原始 token 會再經過 linguistic module，已處理的不會（避免重複 stemming）。
  - 不符合上述格式的行當成純文字，每行是一篇文件（docID = 行號）。

```python
cfg = Config(stemming=False)
for doc_id, trec_id, terms in iter_processed_docs(WARC_PATH, cfg):
    # terms: [("information", 0), ("retrieval", 1), ...]
    for term, pos in terms:
        ...  # add pos to postings[term][doc_id]

for doc_id, trec_id, terms in iter_processed_file(LING_OUT):
    for term, pos in terms:
        ...
```

In [8]:
def _fmt(pairs) -> str:
    """[(text, pos), ...] -> "text:pos text:pos ..." (one line of the intermediate files)."""
    return " ".join(f"{w}:{p}" for w, p in pairs)


def process_document(doc: Document, cfg: Config = Config()) -> List[Tuple[str, int]]:
    """Tokenize + linguistic-process one document; returns [(term, pos), ...]."""
    # title words come first, so they take the first positions
    text = f"{doc.title} {doc.body}" if cfg.include_title else doc.body
    return linguistic(tokenize(text, cfg), cfg)


def iter_processed_docs(path: str, cfg: Config = Config(),
                        max_docs: Optional[int] = None
                        ) -> Iterator[Tuple[int, str, List[Tuple[str, int]]]]:
    """Yield (doc_id, trec_id, [(term, pos), ...]) for every document in a WARC file.
    max_docs=None processes the whole file; a number stops after that doc_id."""
    for doc in parse_warc(path):
        if max_docs is not None and doc.doc_id > max_docs:
            break
        yield doc.doc_id, doc.trec_id, process_document(doc, cfg)


def iter_processed_file(
    path: str,
    cfg: Optional[Config] = None,
    max_docs: Optional[int] = None,
    raw_tokens: Optional[bool] = None,
) -> Iterator[Tuple[int, str, List[Tuple[str, int]]]]:
    """Yield (doc_id, trec_id, [(term, pos), ...]) from a `doc_id<TAB>trec_id<TAB>term:pos ...` text file.

    cfg is None and max_docs is None: lines are streamed verbatim (Section 7.4 reads
    linguistic_output.txt and tokenizer_output.txt this way).
    cfg given (run_cli): max_docs is honored, positions are shifted to cfg.pos_base, and linguistic()
    is applied if the file holds raw tokens. raw_tokens=True / False says so explicitly; None (default) decides
    by the file name ("tok" / "ling"), otherwise by the first document: capitals or a stopword share of at
    least 20% mean raw tokens.
    Processed files are never processed again, so terms are not stemmed twice.
    Lines that are not in the tab-separated format are treated as plain text, one document per line.
    """
    # None = not decided yet (file name first, then the first document)
    is_raw_tokens: Optional[bool] = raw_tokens
    pos_shift: Optional[int] = None
    fname_lower = os.path.basename(path).lower()
    if is_raw_tokens is None and "tok" in fname_lower:
        is_raw_tokens = True
    elif is_raw_tokens is None and "ling" in fname_lower:
        is_raw_tokens = False

    with open(path, encoding="utf-8") as f:
        for line_idx, line in enumerate(f, start=1):
            raw_line = line.rstrip("\r\n")
            parts = raw_line.split("\t", 2)
            # plain-text fallback: the whole line is one document, doc_id = line number
            if len(parts) < 2 or not parts[0].isdigit():
                if max_docs is not None and line_idx > max_docs:
                    break
                active_cfg = cfg or Config()
                doc = Document(doc_id=line_idx, trec_id=f"txt-{line_idx:05d}", url="", title="", body=raw_line)
                yield line_idx, doc.trec_id, process_document(doc, active_cfg)
                continue

            doc_id = int(parts[0])
            if max_docs is not None and doc_id > max_docs:
                break
            trec_id = parts[1]
            rest = parts[2] if len(parts) > 2 else ""

            # "term:pos" pairs; rpartition keeps any ':' inside the term
            terms: List[Tuple[str, int]] = []
            valid_pairs = True
            for item in rest.split():
                w, sep, p_str = item.rpartition(":")
                if not sep or not p_str.lstrip("-").isdigit():
                    valid_pairs = False
                    break
                terms.append((w, int(p_str)))

            # third column is free text rather than term:pos pairs -> process it as a document body
            if not valid_pairs:
                active_cfg = cfg or Config()
                doc = Document(doc_id=doc_id, trec_id=trec_id, url="", title="", body=rest)
                yield doc_id, trec_id, process_document(doc, active_cfg)
                continue

            if cfg is not None and terms:
                # shift 0-based files to pos_base=1 (or the reverse), decided once from the first document
                if pos_shift is None:
                    min_p = min(p for _, p in terms)
                    pos_shift = (cfg.pos_base - min_p) if min_p in (0, 1) and min_p != cfg.pos_base else 0
                if pos_shift:
                    terms = [(w, p + pos_shift) for w, p in terms]

                if is_raw_tokens is None:
                    # raw tokenizer output has capitals or a high stopword share (~40% of English text);
                    # processed output may still hold a few stems equal to a stopword ("others" -> "other"),
                    # so a single stopword is not proof of raw input
                    n_sw = sum(w.lower() in STOPWORDS for w, _ in terms)
                    is_raw_tokens = any(w != w.lower() for w, _ in terms) or n_sw >= 0.2 * len(terms)
                if is_raw_tokens:
                    tokens = [Token(w, p) for w, p in terms]
                    terms = linguistic(tokens, cfg)

            yield doc_id, trec_id, terms

## 5. Positional Inverted Indexer 與加分功能

輸入是第 4 節的 `(doc_id, trec_id, [(term, pos), ...])` 串流，輸出投影片 p.5 規定的 positional index。

| 小節 | 內容 |
|---|---|
| 5.1 | 資料結構：`PostingEntry`、trie 字典、skip pointers、postings 一行的寫出與讀回 |
| 5.2 | `PositionalIndex`：三種建索引方式（記憶體內、SPIMI、串流 SPIMI）、寫檔、AND / phrase / proximity 查詢 |
| 5.3 | 命令列開關 `run_cli` |

### 輸出檔案（UTF-8、無 BOM、LF 換行）

| 檔案 | 格式 | 範例 |
|---|---|---|
| `dictionary.txt` | 每行 `term, df`，依 term 排序 | `inform, 2` |
| `dictionary_terms_only.txt` | 每行一個 term（投影片 p.5 "a sorted list of vocabularies"） | `inform` |
| `postings.txt` | 每個 term 一行，順序與 dictionary 相同，格式同投影片 p.5 / 教科書 Fig. 2.11 | `inform, 2: <1, 2: <0, 2>; 2, 1: <12>>` |
| `doc_map.txt` | 每行 `docID<TAB>TREC-ID` | `1	clueweb09-en0000-07-00000` |

- 排序使用 Python 字串比較（Unicode code point 順序），所以數字開頭的 term 排在英文字母前面，中文等非拉丁文字的 term 排在英文之後。
- postings 預設每個 term 寫成一行，方便逐行讀取，也讓 trie 能記錄每行的 byte offset；`multiline_postings=True` 可改成投影片的多行排版。

### Baseline 與 Enhanced 兩套索引

- **Enhanced**（主要輸出）：`Config()` 預設設定，或讀取 `linguistic_output.txt`。開啟 case folding、stopword removal、Porter stemming。
- **Baseline**：`Config.baseline()`，或讀取 `tokenizer_output.txt`。不做任何語言處理，保留投影片 p.5 範例中的 `to` 與未 stem 的 `retrieval`，方便直接對照。

### 5.1 資料結構與 postings 格式

| 名稱 | 說明 |
|---|---|
| `PostingEntry(doc_id, tf, positions, skip)` | 一個 term 在一篇文件中的資訊。`positions` 嚴格遞增，保留 stopword 移除後的 gap |
| `VocabularyTrie` | 字典的 trie：精確查詢 `search` 與前綴查詢 `starts_with`。節點存 df 與 `meta`（記憶體內索引存 postings list，串流索引存該 term 在 `postings.txt` 的 byte offset）。前綴查詢使用迭代式 DFS，很長的 term 也不會觸發 `RecursionError` |
| `compute_skip_pointers` | 每隔 ⌊√L⌋ 個 entry 放一個 skip pointer（L = postings 長度），L < 4 時不建 |
| `format_posting_standard` | 把一個 term 的 postings 寫成投影片 p.5 的格式（單行或多行） |
| `parse_posting_line` | 把一筆 postings 讀回 `PostingEntry` list；skip pointer 不寫進檔案，在這裡重新計算 |

In [9]:
# ==============================================================================
# Data structures: postings, vocabulary trie, skip pointers, postings line format
# ==============================================================================
@dataclass
class PostingEntry:
    """One document's entry in a term's postings list: doc_id, term frequency (tf), positions, skip pointer."""
    doc_id: int
    tf: int
    positions: List[int]
    # skip pointer (index of the target entry in the postings list, its doc_id); None = no skip here
    skip: Optional[Tuple[int, int]] = None


class VocabularyTrieNode:
    """Trie node; a node that ends a term stores the term's df and `meta` (its postings or a byte offset)."""
    __slots__ = ("children", "is_end", "df", "meta")

    def __init__(self):
        self.children: Dict[str, "VocabularyTrieNode"] = {}
        self.is_end: bool = False
        self.df: int = 0
        # List[PostingEntry] for an in-memory index, or the int byte offset of the term's line in postings.txt
        self.meta: Any = None


class VocabularyTrie:
    """Vocabulary stored as a trie: exact lookup and prefix lookup (Slide 7 efficiency feature)."""
    def __init__(self):
        self.root = VocabularyTrieNode()
        self.size = 0

    def insert(self, term: str, df: int, meta: Any = None) -> None:
        """Add `term` (or overwrite its df / meta if it already exists)."""
        node = self.root
        for ch in term:
            if ch not in node.children:
                node.children[ch] = VocabularyTrieNode()
            node = node.children[ch]
        if not node.is_end:
            self.size += 1
        node.is_end = True
        node.df = df
        node.meta = meta

    def search(self, term: str) -> Optional[Tuple[int, Any]]:
        """Return (df, meta) if `term` is in the vocabulary, else None (a prefix of a term is not a term)."""
        node = self.root
        for ch in term:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return (node.df, node.meta) if node.is_end else None

    def starts_with(self, prefix: str) -> List[Tuple[str, int, Any]]:
        """Return [(term, df, meta), ...] for every term starting with `prefix`, in lexicographical order."""
        node = self.root
        for ch in prefix:
            if ch not in node.children:
                return []
            node = node.children[ch]
        results: List[Tuple[str, int, Any]] = []
        # iterative pre-order DFS with an explicit stack (no RecursionError on very long terms);
        # children are pushed in reverse order so they are popped in lexicographical order
        stack: List[Tuple[VocabularyTrieNode, str]] = [(node, prefix)]
        while stack:
            cur, word = stack.pop()
            if cur.is_end:
                results.append((word, cur.df, cur.meta))
            for ch in sorted(cur.children, reverse=True):
                stack.append((cur.children[ch], word + ch))
        return results


def compute_skip_pointers(postings: List[PostingEntry]) -> List[PostingEntry]:
    """Attach skip pointers every floor(sqrt(L)) entries, L = len(postings); none when L < 4 (Slide 7).
    Entry i (i a multiple of the step) points to entry i + step. Existing pointers are reset."""
    L = len(postings)
    if L < 4:
        for p in postings:
            p.skip = None
        return postings
    step = int(math.isqrt(L))
    for i in range(L):
        if i % step == 0 and i + step < L:
            postings[i].skip = (i + step, postings[i + step].doc_id)
        else:
            postings[i].skip = None
    return postings


def format_posting_standard(term: str, postings: List[PostingEntry], multiline: bool = False) -> str:
    """Format one postings record as in Slide 5 (textbook Fig. 2.11, Sec. 2.4.2):
        term, df: <doc1, tf1: <pos1, pos2, ...>; doc2, tf2: <pos1, pos2, ...>>
    multiline=False (default) writes the whole record on one line, so a file can be read line by line and
    a byte offset identifies a term; multiline=True breaks lines after "df:" and after each ";" as on the slide.
    """
    df = len(postings)
    doc_parts = []
    for p in postings:
        pos_str = ", ".join(str(pos) for pos in p.positions)
        doc_parts.append(f"{p.doc_id}, {p.tf}: <{pos_str}>")
    if multiline:
        sep = ";\n"
        return f"{term}, {df}:\n<{sep.join(doc_parts)}>"
    return f"{term}, {df}: <{'; '.join(doc_parts)}>"


def parse_posting_line(line: str, attach_skips: bool = True) -> Tuple[str, int, List[PostingEntry]]:
    """Parse one postings.txt record (single-line or multi-line) back into (term, df, [PostingEntry, ...]).
    Skip pointers are not stored in the file; they are recomputed here when attach_skips=True."""
    raw = (
        line.strip()
        .replace(":\r\n<", ": <")
        .replace(":\n<", ": <")
        .replace(";\r\n", "; ")
        .replace(";\n", "; ")
    )
    head_part, body_part = raw.split(": <", 1)
    term, df_str = head_part.rsplit(", ", 1)
    df = int(df_str)
    # drop the record's closing '>'
    body = body_part[:-1]
    postings: List[PostingEntry] = []
    for chunk in body.split("; "):
        doc_tf_part, pos_part = chunk.split(": <", 1)
        doc_id_str, tf_str = doc_tf_part.split(", ", 1)
        positions = [int(x) for x in pos_part[:-1].split(", ")]
        postings.append(PostingEntry(doc_id=int(doc_id_str), tf=int(tf_str), positions=positions))
    if attach_skips:
        compute_skip_pointers(postings)
    return term, df, postings

### 5.2 `PositionalIndex`：建索引與查詢

`PositionalIndex` 記錄 term → postings list（依 `doc_id` 遞增）、`doc_map`（docID → TREC-ID）、文件數與 token 總數。

**建索引的三種方式**

| 方式 | 函式 | 說明 |
|---|---|---|
| 記憶體內 | `build_from_iter(..., use_spimi=False)` | 全部 postings 放在記憶體，適合小資料與測試；`save_files` 寫出與串流版相同的檔案 |
| SPIMI（結果放記憶體） | `build_from_iter(..., use_spimi=True)` | 分區塊寫入暫存檔再合併，最後仍載入記憶體；用來驗證合併邏輯 |
| **串流 SPIMI（預設）** | `build_and_save_streaming(...)` | 合併結果直接寫檔，不把整份 postings 放進記憶體 |

**串流 SPIMI 的作法**

1. 逐篇讀取文件，把 `(term, doc, positions)` 累積在記憶體中的區塊。一篇文件處理完後，若累積的 `(term, pos)` 數量達到 `cfg.spimi_block_limit`，就把區塊依 term 排序寫成暫存檔。同一篇文件一定落在同一個區塊。
2. 用 `heapq` 對所有區塊做 **K-way merge**：每次取出字典序最小的 term。因為文件依 docID 遞增讀入，較早的區塊 docID 一定較小，所以同一個 term 的 postings 依區塊順序串接後仍然是 docID 遞增。
3. 每合併完一個 term，就立刻寫出 `dictionary.txt` 與 `postings.txt` 的一行，然後釋放這個 term 的資料。記憶體中只有一個區塊、每個區塊檔的一行、`doc_map`，以及（若開啟）trie 字典。
4. 結束後刪除暫存區塊檔。

**加分功能（投影片 p.7）**

| 功能 | 設計 |
|---|---|
| Trie 字典（`use_trie`） | 記憶體內索引的 trie 節點直接存 postings list；串流索引只存 **byte offset**，查詢時 `seek` 過去讀一行 |
| 不使用 trie（`use_trie=False`） | 不在記憶體保留字典；查詢時依序掃描已排序的 `postings.txt`，超過目標 term 就停止 |
| Skip pointers（`use_skip_pointers`） | 間隔 ⌊√L⌋，在 postings 載入記憶體時計算 |
| AND 查詢 | `intersect_with_skips(t1, t2)`：兩個 postings 線性合併，能跳就跳，並回報比較次數與跳躍次數 |
| Phrase query | `phrase_query("...")`：query 經過同一套 tokenizer + linguistic 處理，再比對每個 term 的相對 position。因為 stopword 留下 gap，`state-of-the-art information` 這類含停用字的片語也能正確比對 |
| Proximity search | `proximity_query(t1, t2, k)`：找出 `|pos1 − pos2| ≤ k` 的位置對，文件層級用 skip pointers，文件內用雙指標滑動視窗 |
| 查詢詞正規化 | AND 與 proximity 的查詢詞一律先用索引的 `Config` 處理（`Inverted` → `invert`），與 phrase query 相同；處理後被刪掉（停用字）或變成多個 token 的詞沒有結果。不先拿原始詞查字典：`information` 本身也是一個罕見的 term（`Informationes` 的 stem，df 1），直接查會蓋掉 `inform` 的 postings |
| 輸入順序檢查 | 三種建法都要求 docID 嚴格遞增，否則丟出 `ValueError`（postings 依讀入順序串接，順序錯了索引就不再排序） |

In [10]:
# ==============================================================================
# Positional inverted index: construction (in-memory / SPIMI / streaming SPIMI) and queries
# ==============================================================================
class PositionalIndex:
    """Positional inverted index.

    Built in memory, with SPIMI, or with streaming SPIMI straight to disk. The vocabulary is a dict and,
    optionally, a trie (holding postings in memory or byte offsets into postings.txt). Supports skip
    pointers, AND queries, phrase queries and proximity search."""

    def __init__(self, cfg: Config = Config()):
        self.cfg = cfg
        self.postings: Dict[str, List[PostingEntry]] = {}
        self.doc_map: Dict[int, str] = {}
        self.trie: Optional[VocabularyTrie] = VocabularyTrie() if cfg.use_trie else None
        self.num_docs: int = 0
        self.total_tokens: int = 0
        self._streamed_vocab_size: int = 0
        self._streamed_postings_path: Optional[str] = None

    @property
    def vocab_size(self) -> int:
        """Number of distinct terms (for a streamed index, counted while writing the files)."""
        if self._streamed_vocab_size > 0 and not self.postings:
            return self._streamed_vocab_size
        return len(self.postings)

    @staticmethod
    def _read_posting_record_from_handle(fp) -> Optional[str]:
        """Read one postings record (single-line or multi-line, ends with '>') from a binary file handle."""
        lines: List[str] = []
        for raw_line in fp:
            s = raw_line.decode("utf-8").rstrip("\r\n")
            if not lines and not s.strip():
                continue
            lines.append(s)
            if s.endswith(">"):
                return "\n".join(lines)
        return "\n".join(lines) if lines else None

    def get_postings(self, term: str) -> List[PostingEntry]:
        """Return the postings list of `term` ([] if absent).

        In-memory index: trie or dict lookup. Streamed index with a trie: seek to the stored byte offset and
        parse one record. Streamed index without a trie: scan the sorted postings.txt and stop once the
        current term passes `term` (no vocabulary kept in memory)."""
        if self.trie is not None:
            res = self.trie.search(term)
            if res is None:
                return []
            meta = res[1]
            if isinstance(meta, int) and self._streamed_postings_path:
                with open(self._streamed_postings_path, "rb") as fp:
                    fp.seek(meta)
                    rec_str = self._read_posting_record_from_handle(fp)
                if not rec_str:
                    return []
                _, _, plist = parse_posting_line(rec_str, attach_skips=self.cfg.use_skip_pointers)
                return plist
            return meta if isinstance(meta, list) else []
        if self.postings:
            return self.postings.get(term, [])
        if self._streamed_postings_path and os.path.exists(self._streamed_postings_path):
            with open(self._streamed_postings_path, "rb") as fp:
                while True:
                    rec_str = self._read_posting_record_from_handle(fp)
                    if rec_str is None:
                        break
                    head_line = rec_str.split("\n", 1)[0]
                    head_part = head_line.split(": <", 1)[0] if ": <" in head_line else head_line.rstrip(":")
                    rec_term, _ = head_part.rsplit(", ", 1)
                    if rec_term == term:
                        _, _, plist = parse_posting_line(rec_str, attach_skips=self.cfg.use_skip_pointers)
                        return plist
                    if rec_term > term:
                        break
        return []

    def _finalize_structure(self) -> None:
        """Sort the vocabulary, attach skip pointers (if enabled) and fill the trie (if enabled)."""
        sorted_terms = sorted(self.postings.keys())
        sorted_postings: Dict[str, List[PostingEntry]] = {}
        self.trie = VocabularyTrie() if self.cfg.use_trie else None

        for term in sorted_terms:
            plist = self.postings[term]
            if self.cfg.use_skip_pointers:
                compute_skip_pointers(plist)
            sorted_postings[term] = plist
            if self.trie is not None:
                self.trie.insert(term, len(plist), plist)
        self.postings = sorted_postings

    @classmethod
    def _flush_spimi_blocks(
        cls,
        doc_iter: Iterator[Tuple[int, str, List[Tuple[str, int]]]],
        cfg: Config,
        spimi_temp_dir: str,
        idx: "PositionalIndex",
    ) -> List[str]:
        """Read the document stream and write sorted SPIMI block files to spimi_temp_dir; return their paths.

        A block is flushed only after a whole document, once it holds >= cfg.spimi_block_limit (term, pos)
        pairs, so a document never spans two blocks. Block line format: term<TAB>doc:p1,p2 doc:p1,... ."""
        block_files: List[str] = []
        block_index: Dict[str, List[PostingEntry]] = defaultdict(list)
        block_pairs = 0
        last_doc_id: Optional[int] = None

        def flush_block(block_id: int) -> str:
            bpath = os.path.join(spimi_temp_dir, f"block_{block_id:04d}.txt")
            with open(bpath, "w", encoding="utf-8", newline="\n") as bf:
                for term in sorted(block_index.keys()):
                    entries_str = " ".join(
                        f"{p.doc_id}:{','.join(map(str, p.positions))}"
                        for p in block_index[term]
                    )
                    bf.write(f"{term}\t{entries_str}\n")
            block_index.clear()
            return bpath

        for doc_id, trec_id, terms in doc_iter:
            # postings are concatenated in arrival order, so they are sorted only if doc_ids increase
            if last_doc_id is not None and doc_id <= last_doc_id:
                raise ValueError(f"doc_ids must be strictly increasing: got {doc_id} after {last_doc_id}")
            last_doc_id = doc_id
            idx.doc_map[doc_id] = trec_id
            idx.num_docs += 1
            idx.total_tokens += len(terms)
            if terms:
                # group this document's positions by term -> one PostingEntry per term
                doc_term_pos: Dict[str, List[int]] = defaultdict(list)
                for term, pos in terms:
                    doc_term_pos[term].append(pos)
                for term, pos_list in doc_term_pos.items():
                    block_index[term].append(
                        PostingEntry(doc_id=doc_id, tf=len(pos_list), positions=pos_list)
                    )
                block_pairs += len(terms)

            # flush only between documents, so a document never spans two blocks
            if block_pairs >= cfg.spimi_block_limit:
                block_files.append(flush_block(len(block_files)))
                block_pairs = 0

        if block_index:
            block_files.append(flush_block(len(block_files)))
        return block_files

    @staticmethod
    def _iter_merged_blocks(block_files: List[str]) -> Iterator[Tuple[str, List[PostingEntry]]]:
        """K-way merge of sorted SPIMI block files; yields (term, complete postings list) in term order.

        Only one line per block file is in memory at a time. The heap orders by (term, block index), so a
        term's entries are concatenated in block order; blocks were written in doc_id order, so the
        merged postings list is already sorted by doc_id."""
        handles = [open(bp, "r", encoding="utf-8") for bp in block_files]
        heap: List[Tuple[str, int, List[PostingEntry]]] = []

        def parse_block_line(line: str) -> Tuple[str, List[PostingEntry]]:
            term, rest = line.rstrip("\n").split("\t", 1)
            entries = []
            for item in rest.split(" "):
                if not item:
                    continue
                d_str, p_str = item.split(":", 1)
                pos_list = [int(x) for x in p_str.split(",")]
                entries.append(PostingEntry(doc_id=int(d_str), tf=len(pos_list), positions=pos_list))
            return term, entries

        try:
            for b_idx, fh in enumerate(handles):
                line = fh.readline()
                if line:
                    t, ents = parse_block_line(line)
                    heapq.heappush(heap, (t, b_idx, ents))

            cur_term: Optional[str] = None
            cur_postings: List[PostingEntry] = []

            while heap:
                term, b_idx, ents = heapq.heappop(heap)
                if cur_term is None:
                    cur_term = term
                    cur_postings = list(ents)
                elif term == cur_term:
                    cur_postings.extend(ents)
                else:
                    yield cur_term, cur_postings
                    cur_term = term
                    cur_postings = list(ents)

                next_line = handles[b_idx].readline()
                if next_line:
                    nt, nents = parse_block_line(next_line)
                    heapq.heappush(heap, (nt, b_idx, nents))

            if cur_term is not None:
                yield cur_term, cur_postings
        finally:
            for fh in handles:
                fh.close()

    @classmethod
    def build_and_save_streaming(
        cls,
        doc_iter: Iterator[Tuple[int, str, List[Tuple[str, int]]]],
        dict_path: str,
        postings_path: str,
        doc_map_path: Optional[str] = None,
        dict_terms_only_path: Optional[str] = None,
        cfg: Config = Config(),
        dict_include_df: bool = True,
        multiline_postings: bool = False,
        spimi_temp_dir: Optional[str] = None,
    ) -> Tuple["PositionalIndex", Dict[str, Any]]:
        """Streaming SPIMI: write blocks, then K-way merge them directly into dictionary.txt, postings.txt
        (and the optional terms-only dictionary and doc map), one term at a time; returns (index, stats).

        The full postings never sit in memory. With cfg.use_trie the trie stores each term's byte offset in
        postings.txt, so get_postings() can seek straight to it. Temporary block files are deleted at the end."""
        idx = cls(cfg)
        cleanup_dir = False
        if spimi_temp_dir is None:
            spimi_temp_dir = tempfile.mkdtemp(prefix="spimi_stream_")
            cleanup_dir = True
        else:
            os.makedirs(spimi_temp_dir, exist_ok=True)

        os.makedirs(os.path.dirname(os.path.abspath(dict_path)) or ".", exist_ok=True)
        os.makedirs(os.path.dirname(os.path.abspath(postings_path)) or ".", exist_ok=True)
        if dict_terms_only_path:
            os.makedirs(os.path.dirname(os.path.abspath(dict_terms_only_path)) or ".", exist_ok=True)

        try:
            block_files = cls._flush_spimi_blocks(doc_iter, cfg, spimi_temp_dir, idx)
            vocab_count = 0
            f_terms_only = open(dict_terms_only_path, "w", encoding="utf-8", newline="\n") if dict_terms_only_path else None
            try:
                with open(dict_path, "w", encoding="utf-8", newline="\n") as f_dict, \
                     open(postings_path, "wb") as f_post:
                    for term, plist in cls._iter_merged_blocks(block_files):
                        vocab_count += 1
                        df = len(plist)
                        if dict_include_df:
                            f_dict.write(f"{term}, {df}\n")
                        else:
                            f_dict.write(f"{term}\n")
                        if f_terms_only is not None:
                            f_terms_only.write(f"{term}\n")

                        # postings.txt is opened in binary mode so tell() is an exact byte offset for the trie
                        byte_offset = f_post.tell()
                        line_bytes = (format_posting_standard(term, plist, multiline=multiline_postings) + "\n").encode("utf-8")
                        f_post.write(line_bytes)

                        if idx.trie is not None:
                            idx.trie.insert(term, df, byte_offset)
            finally:
                if f_terms_only is not None:
                    f_terms_only.close()

            if doc_map_path:
                os.makedirs(os.path.dirname(os.path.abspath(doc_map_path)) or ".", exist_ok=True)
                with open(doc_map_path, "w", encoding="utf-8", newline="\n") as f_map:
                    for doc_id in sorted(idx.doc_map.keys()):
                        f_map.write(f"{doc_id}\t{idx.doc_map[doc_id]}\n")

            idx._streamed_vocab_size = vocab_count
            idx._streamed_postings_path = postings_path
            stats = {
                "num_docs": idx.num_docs,
                "total_tokens": idx.total_tokens,
                "vocab_size": vocab_count,
                "dict_path": dict_path,
                "dict_terms_only_path": dict_terms_only_path,
                "postings_path": postings_path,
                "doc_map_path": doc_map_path,
                "num_blocks": len(block_files),
            }
            return idx, stats
        finally:
            if cleanup_dir and os.path.exists(spimi_temp_dir):
                shutil.rmtree(spimi_temp_dir, ignore_errors=True)

    @classmethod
    def build_from_iter(
        cls,
        doc_iter: Iterator[Tuple[int, str, List[Tuple[str, int]]]],
        cfg: Config = Config(),
        use_spimi: bool = False,
        spimi_temp_dir: Optional[str] = None,
    ) -> "PositionalIndex":
        """Build an in-memory index from (doc_id, trec_id, [(term, pos), ...]); use_spimi=True merges via
        SPIMI block files first (same result). doc_ids must be strictly increasing (ValueError otherwise)."""
        if use_spimi:
            return cls._build_spimi(doc_iter, cfg, spimi_temp_dir=spimi_temp_dir)

        idx = cls(cfg)
        raw_index: Dict[str, List[PostingEntry]] = defaultdict(list)
        last_doc_id: Optional[int] = None

        for doc_id, trec_id, terms in doc_iter:
            # postings are appended in arrival order, so they are sorted only if doc_ids increase
            if last_doc_id is not None and doc_id <= last_doc_id:
                raise ValueError(f"doc_ids must be strictly increasing: got {doc_id} after {last_doc_id}")
            last_doc_id = doc_id
            idx.doc_map[doc_id] = trec_id
            idx.num_docs += 1
            idx.total_tokens += len(terms)
            if not terms:
                continue

            doc_term_pos: Dict[str, List[int]] = defaultdict(list)
            for term, pos in terms:
                doc_term_pos[term].append(pos)

            for term, pos_list in doc_term_pos.items():
                raw_index[term].append(PostingEntry(doc_id=doc_id, tf=len(pos_list), positions=pos_list))

        idx.postings = dict(raw_index)
        idx._finalize_structure()
        return idx

    @classmethod
    def _build_spimi(
        cls,
        doc_iter: Iterator[Tuple[int, str, List[Tuple[str, int]]]],
        cfg: Config = Config(),
        spimi_temp_dir: Optional[str] = None,
    ) -> "PositionalIndex":
        """SPIMI blocks + K-way merge, with the merged postings kept in memory (used to cross-check streaming)."""
        idx = cls(cfg)
        cleanup_dir = False
        if spimi_temp_dir is None:
            spimi_temp_dir = tempfile.mkdtemp(prefix="spimi_blocks_")
            cleanup_dir = True
        else:
            os.makedirs(spimi_temp_dir, exist_ok=True)

        try:
            block_files = cls._flush_spimi_blocks(doc_iter, cfg, spimi_temp_dir, idx)
            idx.postings = {term: plist for term, plist in cls._iter_merged_blocks(block_files)}
            idx._finalize_structure()
            return idx
        finally:
            if cleanup_dir and os.path.exists(spimi_temp_dir):
                shutil.rmtree(spimi_temp_dir, ignore_errors=True)

    def save_files(
        self,
        dict_path: str,
        postings_path: str,
        doc_map_path: Optional[str] = None,
        dict_terms_only_path: Optional[str] = None,
        multiline_postings: bool = False,
        dict_include_df: bool = True,
    ) -> Dict[str, Any]:
        """Write an in-memory index to the same files, in the same format, as build_and_save_streaming."""
        os.makedirs(os.path.dirname(os.path.abspath(dict_path)) or ".", exist_ok=True)
        os.makedirs(os.path.dirname(os.path.abspath(postings_path)) or ".", exist_ok=True)
        if dict_terms_only_path:
            os.makedirs(os.path.dirname(os.path.abspath(dict_terms_only_path)) or ".", exist_ok=True)

        f_terms_only = open(dict_terms_only_path, "w", encoding="utf-8", newline="\n") if dict_terms_only_path else None
        try:
            with open(dict_path, "w", encoding="utf-8", newline="\n") as f_dict, \
                 open(postings_path, "w", encoding="utf-8", newline="\n") as f_post:
                for term in sorted(self.postings.keys()):
                    plist = self.postings[term]
                    df = len(plist)
                    if dict_include_df:
                        f_dict.write(f"{term}, {df}\n")
                    else:
                        f_dict.write(f"{term}\n")
                    if f_terms_only is not None:
                        f_terms_only.write(f"{term}\n")
                    f_post.write(format_posting_standard(term, plist, multiline=multiline_postings) + "\n")
        finally:
            if f_terms_only is not None:
                f_terms_only.close()

        if doc_map_path:
            os.makedirs(os.path.dirname(os.path.abspath(doc_map_path)) or ".", exist_ok=True)
            with open(doc_map_path, "w", encoding="utf-8", newline="\n") as f_map:
                for doc_id in sorted(self.doc_map.keys()):
                    f_map.write(f"{doc_id}\t{self.doc_map[doc_id]}\n")

        return {
            "num_docs": self.num_docs,
            "total_tokens": self.total_tokens,
            "vocab_size": self.vocab_size,
            "dict_path": dict_path,
            "dict_terms_only_path": dict_terms_only_path,
            "postings_path": postings_path,
            "doc_map_path": doc_map_path,
        }

    # --------------------------------------------------------------------------
    # Query processing: AND with skip pointers, phrase query, proximity search
    # --------------------------------------------------------------------------

    def _resolve_query_postings(self, word: str) -> List[PostingEntry]:
        """Postings of one query word. The word goes through this index's tokenizer and linguistic module, like
        document text and like phrase_query ("Information" -> "inform"). The raw word is never looked up as is:
        it may also exist as a rare term of its own ("Informationes" is stemmed to "information", df 1), which
        would hide the real postings of "inform". A word that is removed (stopword) or splits into several
        tokens has no postings."""
        processed = linguistic(tokenize(word, self.cfg), self.cfg)
        if len(processed) != 1:
            return []
        return self.get_postings(processed[0][0])

    def intersect_with_skips(self, term1: str, term2: str) -> Tuple[List[int], Dict[str, int]]:
        """term1 AND term2 by a linear merge that follows skip pointers when they do not overshoot (Slide 7).
        term1 / term2 are query words, normalized like document text. Returns (doc_ids, {"comparisons": ...,
        "skips_used": ...})."""
        p1 = self._resolve_query_postings(term1)
        p2 = self._resolve_query_postings(term2)
        answer: List[int] = []
        i, j = 0, 0
        comparisons = 0
        skips_used = 0

        while i < len(p1) and j < len(p2):
            comparisons += 1
            # advance the list with the smaller doc_id; follow skips only while they do not pass the other doc_id
            d1, d2 = p1[i].doc_id, p2[j].doc_id
            if d1 == d2:
                answer.append(d1)
                i += 1
                j += 1
            elif d1 < d2:
                if self.cfg.use_skip_pointers and p1[i].skip is not None and p1[i].skip[1] <= d2:
                    while p1[i].skip is not None and p1[i].skip[1] <= d2:
                        i = p1[i].skip[0]
                        skips_used += 1
                        comparisons += 1
                else:
                    i += 1
            else:
                if self.cfg.use_skip_pointers and p2[j].skip is not None and p2[j].skip[1] <= d1:
                    while p2[j].skip is not None and p2[j].skip[1] <= d1:
                        j = p2[j].skip[0]
                        skips_used += 1
                        comparisons += 1
                else:
                    j += 1

        return answer, {"comparisons": comparisons, "skips_used": skips_used}

    def phrase_query(self, query_str: str) -> List[Tuple[int, List[int]]]:
        """Exact phrase query using positions (Slide 6); returns [(doc_id, [start positions]), ...].

        The query goes through the same tokenizer and linguistic module, so its terms keep their relative
        positions, including gaps left by removed stopwords."""
        q_terms = linguistic(tokenize(query_str, self.cfg), self.cfg)
        if not q_terms:
            return []

        first_term, first_pos = q_terms[0]
        # q_terms are already processed, so they are looked up directly
        first_postings = self.get_postings(first_term)
        if not first_postings:
            return []

        # doc_id -> candidate phrase start positions (aligned to query position 0)
        candidates: Dict[int, List[int]] = {
            p.doc_id: [pos - first_pos for pos in p.positions]
            for p in first_postings
        }

        # keep a start s only if every later query term occurs at s + (its query position)
        for term, q_pos in q_terms[1:]:
            t_postings = {
                p.doc_id: set(p.positions)
                for p in self.get_postings(term)
                if p.doc_id in candidates
            }
            next_candidates: Dict[int, List[int]] = {}
            for doc_id, start_positions in candidates.items():
                if doc_id not in t_postings:
                    continue
                doc_pos_set = t_postings[doc_id]
                matched_starts = [s for s in start_positions if (s + q_pos) in doc_pos_set]
                if matched_starts:
                    next_candidates[doc_id] = matched_starts
            candidates = next_candidates
            if not candidates:
                break

        return [
            (doc_id, [s + first_pos for s in sorted(starts)])
            for doc_id, starts in sorted(candidates.items())
        ]

    def proximity_query(self, term1: str, term2: str, max_distance: int) -> List[Tuple[int, List[Tuple[int, int]]]]:
        """Proximity search: [(doc_id, [(pos1, pos2), ...]), ...] for every pair with |pos1 - pos2| <= max_distance.
        term1 / term2 are query words, normalized like document text. Skip pointers are used across documents
        and a two-pointer sliding window inside a document."""
        p1 = self._resolve_query_postings(term1)
        p2 = self._resolve_query_postings(term2)
        results: List[Tuple[int, List[Tuple[int, int]]]] = []
        i, j = 0, 0
        while i < len(p1) and j < len(p2):
            d1, d2 = p1[i].doc_id, p2[j].doc_id
            if d1 == d2:
                matches: List[Tuple[int, int]] = []
                pos_a, pos_b = p1[i].positions, p2[j].positions
                # sliding window over pos_b: lo = first position >= pos1 - max_distance (only moves forward)
                lo, n_b = 0, len(pos_b)
                for pos1 in pos_a:
                    while lo < n_b and pos_b[lo] < pos1 - max_distance:
                        lo += 1
                    k_idx = lo
                    while k_idx < n_b and pos_b[k_idx] <= pos1 + max_distance:
                        matches.append((pos1, pos_b[k_idx]))
                        k_idx += 1
                if matches:
                    results.append((d1, matches))
                i += 1
                j += 1
            elif d1 < d2:
                if self.cfg.use_skip_pointers and p1[i].skip is not None and p1[i].skip[1] <= d2:
                    while p1[i].skip is not None and p1[i].skip[1] <= d2:
                        i = p1[i].skip[0]
                else:
                    i += 1
            else:
                if self.cfg.use_skip_pointers and p2[j].skip is not None and p2[j].skip[1] <= d1:
                    while p2[j].skip is not None and p2[j].skip[1] <= d1:
                        j = p2[j].skip[0]
                else:
                    j += 1
        return results

### 5.3 命令列開關 `run_cli`

投影片 p.7 要求選用功能能「用參數開關」。`run_cli([...])` 接受與命令列相同的參數，建立串流索引並回傳統計數字；`.txt` 輸入用 `iter_processed_file` 讀取，其他副檔名當成 WARC。

| 參數 | 作用 |
|---|---|
| `--input`, `-i` | 輸入檔：`.warc`、`.warc.gz`、中間檔 `.txt`，或每行一篇的純文字 `.txt` |
| `--output-dir`, `-o` | 輸出目錄（預設 `sample_index/`） |
| `--baseline` | 關閉 case folding、stopword removal、stemming；仍可再加 `--lemmatize` |
| `--no-case-folding` / `--no-stopwords` / `--no-stem` / `--lemmatize` | 個別開關語言處理步驟 |
| `--no-title` | 不索引 `<title>` |
| `--hyphen-mode {split,join,keep}` | 連字號處理方式 |
| `--pos-base {0,1}` | position 起始值 |
| `--no-trie` / `--no-skip-pointers` | 關閉加分資料結構 |
| `--dict-terms-only` | `dictionary.txt` 只寫 term，不寫 df |
| `--multiline-postings` | postings 使用投影片的多行排版 |
| `--max-docs N` | 只索引前 N 篇 |
| `--block-limit N` | SPIMI 區塊大小 |
| `--txt-input {auto,tokens,terms}` | `.txt` 輸入是原始 token 還是已處理的 term（預設 `auto`：依檔名、再依第一篇文件判斷） |

例如：`run_cli(["--input", "07.warc.gz", "--output-dir", "out", "--no-stem"])`。每個參數在 6.7 都有對應的測試。

In [11]:
# ==============================================================================
# Command-line switches (Slide 7: features turned on/off by a parameter)
# ==============================================================================
def build_cli_parser() -> argparse.ArgumentParser:
    """Argument parser for run_cli(); every Config switch has a flag."""
    parser = argparse.ArgumentParser(
        description="IR HW1: Positional Inverted Indexer for WARC / ClueWeb09 collections."
    )
    parser.add_argument("--input", "-i", type=str, default=None,
                        help="Path to input .warc, .warc.gz, pre-tokenized .txt, or plain-text .txt file.")
    parser.add_argument("--output-dir", "-o", type=str, default=str(INDEX_OUT_DIR),
                        help="Directory where dictionary.txt, postings.txt, and doc_map.txt will be written.")
    parser.add_argument("--baseline", action="store_true",
                        help="Turn OFF case folding, stopword removal and stemming (--lemmatize can still be added).")
    parser.add_argument("--no-title", action="store_true", help="Exclude <title> text from document body.")
    parser.add_argument("--no-case-folding", action="store_true", help="Disable case folding.")
    parser.add_argument("--no-stopwords", action="store_true", help="Disable stopword removal.")
    parser.add_argument("--no-stem", action="store_true", help="Disable Porter stemming.")
    parser.add_argument("--lemmatize", action="store_true", help="Enable English lemmatization.")
    parser.add_argument("--no-trie", action="store_true", help="Disable VocabularyTrie construction.")
    parser.add_argument("--no-skip-pointers", action="store_true", help="Disable skip pointers on postings lists.")
    parser.add_argument("--hyphen-mode", choices=["split", "join", "keep"], default="split",
                        help="How to handle hyphenated words (default: split).")
    parser.add_argument("--pos-base", type=int, choices=[0, 1], default=0,
                        help="Starting token position offset: 0 (default) or 1 (Slide 5 example).")
    parser.add_argument("--dict-terms-only", action="store_true",
                        help="Write only 'term' per line in dictionary.txt instead of 'term, df'.")
    parser.add_argument("--multiline-postings", action="store_true",
                        help="Write postings in Slide 5 multi-line wrapped format.")
    parser.add_argument("--max-docs", type=int, default=None,
                        help="Optional limit on number of documents to index.")
    parser.add_argument("--block-limit", type=int, default=2_500_000,
                        help="Max (term, pos) pairs per SPIMI block before disk flush.")
    parser.add_argument("--txt-input", choices=["auto", "tokens", "terms"], default="auto",
                        help=".txt input holds raw tokens (run the linguistic module), processed terms (index as "
                             "they are), or auto: decide from the file name ('tok' / 'ling'), then the first document.")
    return parser


def run_cli(argv: Optional[List[str]] = None) -> Dict[str, Any]:
    """Build an index from the command-line style arguments `argv` and return the build statistics.
    Inputs ending in .txt are read with iter_processed_file, anything else as a WARC file."""
    parser = build_cli_parser()
    args = parser.parse_args(argv)
    if not args.input:
        raise ValueError("Please provide --input <path_to_warc_or_txt>")

    if args.baseline:
        cfg = Config.baseline(
            include_title=not args.no_title,
            hyphen_mode=args.hyphen_mode,
            pos_base=args.pos_base,
            lemmatization=args.lemmatize,
            use_trie=not args.no_trie,
            use_skip_pointers=not args.no_skip_pointers,
            spimi_block_limit=args.block_limit,
        )
    else:
        cfg = Config(
            include_title=not args.no_title,
            hyphen_mode=args.hyphen_mode,
            pos_base=args.pos_base,
            case_folding=not args.no_case_folding,
            stopword_removal=not args.no_stopwords,
            lemmatization=args.lemmatize,
            stemming=not args.no_stem,
            use_trie=not args.no_trie,
            use_skip_pointers=not args.no_skip_pointers,
            spimi_block_limit=args.block_limit,
        )

    if args.input.endswith(".txt"):
        raw_tokens = {"auto": None, "tokens": True, "terms": False}[args.txt_input]
        stream = iter_processed_file(args.input, cfg=cfg, max_docs=args.max_docs, raw_tokens=raw_tokens)
    else:
        stream = iter_processed_docs(args.input, cfg, max_docs=args.max_docs)

    _, stats = PositionalIndex.build_and_save_streaming(
        stream,
        dict_path=os.path.join(args.output_dir, "dictionary.txt"),
        postings_path=os.path.join(args.output_dir, "postings.txt"),
        doc_map_path=os.path.join(args.output_dir, "doc_map.txt"),
        dict_terms_only_path=os.path.join(args.output_dir, "dictionary_terms_only.txt"),
        cfg=cfg,
        dict_include_df=not args.dict_terms_only,
        multiline_postings=args.multiline_postings,
    )
    return stats

## 6. 單元測試

這一節只測試第 1–5 節的程式本身，不需要 ClueWeb09，全部在數秒內完成。放在處理完整語料（第 7 節）之前：若有錯誤會在這裡停下來，不必先花幾分鐘處理整個語料才發現。

**設計原則**

- **不依賴、也不修改磁碟上的檔案**：測試用的 WARC（與 `test.warc.gz` 相同的 5 篇文件，加上 1 個 `warcinfo` record）由程式碼產生；所有測試輸出都寫到系統暫存目錄，kernel 結束時自動刪除，不會動到 `sample_index/` 或 ClueWeb09 的輸出檔。
- **預期值是人工推導的，不是拿程式的輸出當答案**：例如整篇測試文件每個 token 的 position、投影片 p.5 的 postings 範例字串、查詢結果。索引的正確性則和一個另外寫的暴力法（`brute_force_index`）比對，並用與 `parse_posting_line` 無關的 regex parser 讀檔。
- **對照作業要求**：每項檢查都有編號，字母代表類別。下表列出每條作業要求由哪些檢查負責；8.6 會依這張表印出實際的執行結果。
- **一致的輸出**：每項檢查印一行 `[PASS]`、`[WARN]` 或 `[FAIL]`，每格最後印出統計；只要有 `FAIL` 就丟出 `AssertionError`，`Run All` 會停在那一格。`WARN` 只是提醒，不算失敗；`[SKIP]` 表示這個環境無法執行（沒有安裝 NLTK WordNet 時的 lemmatization 測試）。

| 字母 | 測試對象 | 小節 |
|---|---|---|
| P | Parser：WARC、HTTP、編碼、HTML | 6.1 |
| T | Tokenizer | 6.2 |
| L | Linguistic module | 6.3 |
| A | Pipeline API 與中間檔 | 6.4 |
| I | Indexer：建索引、輸出格式 | 6.5 |
| B | 加分功能：trie、skip pointers、查詢 | 6.6 |
| C | 命令列開關 `run_cli` | 6.7 |
| O、G、D、M | ClueWeb09 輸出驗證 | 8.1–8.4 |
| X | 與獨立的 HTML 解析器比對抽出的文字 | 8.5 |

**作業要求 → 檢查**

| 作業要求 | 投影片 | 單元測試（第 6 節） | 輸出驗證（第 8 節） |
|---|---|---|---|
| 輸入：ClueWeb09 WARC record | p.2、p.4 | P01–P08、P17、P18 | G01、G11 |
| 前處理：HTTP header、編碼、HTML 轉文字 | p.6 | P09–P16、P19–P27 | G02–G06、G10、X01–X05 |
| Tokenizer：數字、連字號、標點 | p.6 | T01–T25、A05、A06 | — |
| pos = token 在文件內文中的位置 | p.6 | T09、T18、L04、A03–A05 | G07、G08 |
| Linguistic：case folding、stopword、lemmatization、stemming | p.7 | L01–L17、A07 | O30–O36、G09 |
| 選用功能可用參數開關 | p.7 | A01、A02、A23、L09、L10、L12、C01–C22 | — |
| Dictionary：排序的 vocabulary，每行一個，含 df | p.5、p.6 | I13、I14、I16 | O04–O12 |
| Postings：`term, df: <doc, tf: <pos, ...>; ...>` | p.5、p.6 | I01–I03、I11、I13、I19 | O13–O29 |
| 索引內容正確 | p.9 | A08–A22、I04–I07、I12、I15、I17、I20 | G01–G08、D03、M01–M03 |
| 大語料：SPIMI 區塊、串流寫檔 | p.8 | I05–I09、I18 | D01–D05 |
| 加分：Trie 字典 | p.7 | I10、B01–B05 | M04 |
| 加分：Skip pointers | p.7 | B06–B10 | M05 |
| 加分：Phrase query、proximity search | p.6 | B11–B19 | M06 |

### 6.0 測試工具與測試資料

這一格只定義東西、不做檢查：

| 名稱 | 用途 |
|---|---|
| `TestSuite` | 每個測試格一個：`ok`、`eq`、`raises` 記錄一項檢查，`skip` 記錄無法執行的檢查，`done()` 印統計並在有 `FAIL` 時丟出例外 |
| `ErrBook` | 大量資料的檢查：依種類計數違規，各保留前 3 個例子 |
| `brute_force_index` | 參考答案：用最直接的 dict 算出 `{term: [(doc, tf, positions), ...]}` |
| `iter_records` / `parse_post_file` | 獨立的 postings 檔 parser（regex），單行與多行格式都能讀 |
| `SAMPLE_WARC_RECORDS` / `write_sample_warc` / `FIXTURE_WARC` | 5 篇的測試 WARC（與 `test.warc.gz` 相同），寫在暫存目錄 |
| `QUERY_DOCS` | 兩篇短文件，用於人工推導的查詢測試 |
| `make_random_corpus` | 80 篇的隨機語料（10 個字、position 之間有隨機 gap、含空文件），用於與暴力法比對 |
| `build_index_variants` | 同一份資料用六種方式建索引：記憶體內、無 skip、SPIMI、串流 + trie、串流無 trie、多行 postings |
| `REQUIREMENTS` | 上面「作業要求 → 檢查」表的程式版本，8.6 用它印出結果 |

In [12]:
# ==============================================================================
# Test harness, test data and reference implementations (used by Sections 6, 7.1 and 8)
# ==============================================================================
import atexit
import contextlib
import io
import random

# ---------- harness ----------
# every check run so far: (suite, check id, check name, status); Section 8.5 prints the summary
TEST_RESULTS: List[Tuple[str, str, str, str]] = []
# check ids look like "P01", "T12", "O05"
_CHECK_ID_RE = re.compile(r"\b([A-Z]\d{2})\b")


def _short(x: Any, limit: int = 300) -> str:
    """repr() cut to `limit` characters, so a failing check on a large value stays readable."""
    s = repr(x)
    return s if len(s) <= limit else s[:limit] + " ..."


class TestSuite:
    """The checks of one test cell.

    Every check prints one [PASS] / [WARN] / [FAIL] (or [SKIP]) line. done() prints the counts and raises AssertionError
    if any FAIL was recorded, so "Run All" stops at the first broken suite. WARN marks something worth
    knowing that is not a bug and never fails a suite. Re-running a cell replaces its earlier results."""

    def __init__(self, name: str):
        self.name = name
        self.counts: Counter = Counter()
        self.failures: List[str] = []
        TEST_RESULTS[:] = [r for r in TEST_RESULTS if r[0] != name]
        print(f"===== {name} =====")

    def section(self, title: str) -> None:
        print(f"\n--- {title} ---")

    def note(self, msg: str) -> None:
        print(f"[INFO] {msg}")

    def ok(self, name: str, cond: Any, detail: str = "", level: str = "FAIL", info: str = "") -> bool:
        """PASS if cond is truthy, else `level` (FAIL or WARN). `detail` is shown on failure, `info` on success."""
        status = "PASS" if cond else level
        self.counts[status] += 1
        if status == "FAIL":
            self.failures.append(f"{name}: {detail}")
        m = _CHECK_ID_RE.search(name)
        TEST_RESULTS.append((self.name, m.group(1) if m else "", name, status))
        shown = info if cond else detail
        print(f"[{status}] {name}" + (f"  -> {shown}" if shown else ""))
        return bool(cond)

    def eq(self, name: str, got: Any, expected: Any, level: str = "FAIL") -> bool:
        """Check got == expected; on failure both values are shown."""
        return self.ok(name, got == expected, f"got {_short(got)}, expected {_short(expected)}", level)

    def raises(self, name: str, exc_type, fn, *args, **kwargs) -> bool:
        """Check that fn(*args, **kwargs) raises exc_type."""
        try:
            fn(*args, **kwargs)
        except exc_type:
            return self.ok(name, True)
        except Exception as e:
            return self.ok(name, False, f"raised {type(e).__name__}: {e}")
        return self.ok(name, False, f"{exc_type.__name__} was not raised")

    def skip(self, name: str, reason: str) -> None:
        """Record a check that cannot run here (e.g. WordNet is not installed); never fails the suite."""
        self.counts["SKIP"] += 1
        m = _CHECK_ID_RE.search(name)
        TEST_RESULTS.append((self.name, m.group(1) if m else "", name, "SKIP"))
        print(f"[SKIP] {name}  -> {reason}")

    def done(self) -> None:
        c = self.counts
        print(f"\n{self.name}: PASS={c['PASS']}  WARN={c['WARN']}  FAIL={c['FAIL']}"
              + (f"  SKIP={c['SKIP']}" if c["SKIP"] else ""))
        if self.failures:
            raise AssertionError(f"{len(self.failures)} check(s) failed:\n  " + "\n  ".join(self.failures))


class ErrBook:
    """Counts violations per kind and keeps the first 3 examples of each (for checks over many items)."""
    def __init__(self):
        self.n, self.ex = Counter(), defaultdict(list)

    def add(self, kind: str, msg: str) -> None:
        self.n[kind] += 1
        if len(self.ex[kind]) < 3:
            self.ex[kind].append(msg)

    def ok(self, kind: str) -> bool:
        return self.n[kind] == 0

    def detail(self, kind: str) -> str:
        return f"{self.n[kind]} violations, e.g. {self.ex[kind]}" if self.n[kind] else ""


# ---------- small helpers ----------
def toks(text: str, **kw) -> List[str]:
    """Token strings of tokenize(text, Config(**kw))."""
    return [x.text for x in tokenize(text, Config(**kw))]


def ling(text: str, **kw) -> List[Tuple[str, int]]:
    """tokenize + linguistic with the same Config(**kw); returns [(term, pos), ...]."""
    cfg = Config(**kw)
    return linguistic(tokenize(text, cfg), cfg)


def words(pairs) -> List[str]:
    """[(term, pos), ...] -> [term, ...]"""
    return [w for w, _ in pairs]


def entries(plist) -> List[Tuple[int, int, List[int]]]:
    """[PostingEntry, ...] -> [(doc_id, tf, positions), ...] for direct comparison."""
    return [(p.doc_id, p.tf, p.positions) for p in plist]


def flat(ix) -> Dict[str, List[Tuple[int, int, List[int]]]]:
    """In-memory PositionalIndex -> {term: [(doc_id, tf, positions), ...]}."""
    return {t: entries(pl) for t, pl in ix.postings.items()}


def sha256_file(path: str) -> str:
    """SHA-256 of a file, read in 64 KB chunks."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 16), b""):
            h.update(chunk)
    return h.hexdigest()


def read_lines(path: str) -> List[str]:
    """Lines of a UTF-8 file without '\\n'; newline="" keeps any '\\r', so CRLF files would not compare equal."""
    with open(path, encoding="utf-8", newline="") as f:
        text = f.read()
    return text.split("\n")[:-1] if text.endswith("\n") else text.split("\n")


# ---------- reference implementations (share no code with the indexer) ----------
def brute_force_index(docs) -> Dict[str, List[Tuple[int, int, List[int]]]]:
    """Oracle: {term: [(doc_id, tf, sorted positions), ...] sorted by doc_id}, computed with plain dicts
    from a (doc_id, trec_id, [(term, pos), ...]) stream."""
    ref = defaultdict(lambda: defaultdict(list))
    for d, _, terms in docs:
        for w, p in terms:
            ref[w][d].append(p)
    return {t: [(d, len(ps), sorted(ps)) for d, ps in sorted(dd.items())] for t, dd in ref.items()}


# independent postings-file parser (regex based, does not use parse_posting_line):
#   term, df: <doc, tf: <pos, ...>; ...>
REC_RE = re.compile(r"(\S+), (\d+):\s*<(.*)>", re.S)
BODY_RE = re.compile(r"\d+, \d+: <\d+(?:, \d+)*>(?:;\s*\d+, \d+: <\d+(?:, \d+)*>)*")
ENT_RE = re.compile(r"(\d+), (\d+): <([^>]*)>")


def iter_records(path: str) -> Iterator[Tuple[int, str]]:
    """Yield (byte_offset_of_record, text). Accepts the single-line layout and the slide's multi-line layout."""
    buf, start, off = [], 0, 0
    with open(path, "rb") as f:
        for raw in f:
            # UnicodeDecodeError propagates, so the caller can report it
            s = raw.decode("utf-8").rstrip("\n")
            if not buf:
                if not s.strip():
                    off += len(raw)
                    continue
                start = off
            buf.append(s)
            off += len(raw)
            if s.rstrip().endswith(">"):
                yield start, "\n".join(buf)
                buf = []
    if buf:
        yield start, "\n".join(buf)


def parse_post_file(path: str) -> Dict[str, List[Tuple[int, int, List[int]]]]:
    """Whole postings file -> {term: [(doc_id, tf, positions), ...]} (for small files)."""
    out = {}
    for _, rec in iter_records(path):
        m = REC_RE.fullmatch(rec)
        out[m.group(1)] = [(int(d), int(tf), [int(x) for x in ps.split(", ")])
                           for d, tf, ps in ENT_RE.findall(m.group(3))]
    return out


# ---------- test data ----------
def _warc_record(warc_type: str, content: bytes, trec_id: str = "", uri: str = "") -> bytes:
    """One WARC/0.18 record with LF line endings (as in ClueWeb09)."""
    head = ["WARC/0.18", f"WARC-Type: {warc_type}"]
    if uri:
        head.append(f"WARC-Target-URI: {uri}")
    if trec_id:
        head.append(f"WARC-TREC-ID: {trec_id}")
        head.append("Content-Type: application/http;msgtype=response")
    head.append(f"Content-Length: {len(content)}")
    return ("\n".join(head) + "\n\n").encode("utf-8") + content + b"\n\n"


def _http_response(html_bytes: bytes, content_type: str = "text/html") -> bytes:
    """HTTP header (LF line endings) + payload."""
    return f"HTTP/1.1 200 OK\nContent-Type: {content_type}\nServer: Apache\n\n".encode("ascii") + html_bytes


# The sample collection: the same 5 documents as test.warc.gz, each aimed at a parser / tokenizer feature.
SAMPLE_WARC_RECORDS: List[bytes] = [
    # describes the file itself; must be skipped
    _warc_record("warcinfo", b"software: Nutch 1.0-dev\nformat: WARC file version 0.18\n"),
    # doc 1: plain English page, the title is repeated in <h1>
    _warc_record("response", _http_response(
        b"<html><head><title>Information Retrieval</title></head><body><h1>Information Retrieval</h1>"
        b"<p>Information retrieval (IR) is finding material of an unstructured nature that satisfies an "
        b"information need from within large collections.</p></body></html>"),
        "clueweb09-en0000-00-00000", "http://example.com/ir"),
    # doc 2: <style>, <script>, comment, entities, and every tokenizer rule (digits, hyphens, acronyms, ...)
    _warc_record("response", _http_response(
        ("<html><head><title>Messy &amp; Tricky</title><style>p{color:red}</style>"
         "<script>var x = 'should not appear';</script></head><body><!-- hidden comment -->"
         "<p>The U.S.A. spent $1,000,000 on state-of-the-art e-mail systems in 2009.</p>"
         "<p>John's mp3 player doesn't support version 2.0.1; pi is 3.14 &amp; IP is 192.168.0.1.</p>"
         "<p>Call 555-1234 (e.g. after 5pm). Caf&eacute; na&iuml;ve résumé</p></body></html>").encode("utf-8"),
        "text/html; charset=utf-8"),
        "clueweb09-en0000-00-00001", "http://example.com/messy"),
    # doc 3: Big5 page; the charset is declared only in <meta>, not in the HTTP header
    _warc_record("response", _http_response(
        ("<html><head><meta http-equiv='Content-Type' content='text/html; charset=big5'><title>資訊檢索</title>"
         "</head><body><p>資訊檢索（IR）課程 HW1：建立 inverted index！</p></body></html>").encode("cp950")),
        "clueweb09-zh0000-00-00000", "http://example.com.tw/"),
    # doc 4: XHTML with an <?xml?> declaration and no <title>
    _warc_record("response", _http_response(
        b"<?xml version='1.0' encoding='utf-8'?><html><body><p>XHTML pages also work.</p></body></html>",
        "application/xhtml+xml"),
        "clueweb09-en0000-00-00002", "http://example.com/xhtml"),
    # doc 5: HTTP header only, no HTML
    _warc_record("response", _http_response(b""), "clueweb09-en0000-00-00003", "http://example.com/empty"),
]


def write_sample_warc(path: str) -> str:
    """Write SAMPLE_WARC_RECORDS as a gzip WARC file and return its path."""
    with gzip.open(path, "wb") as f:
        for rec in SAMPLE_WARC_RECORDS:
            f.write(rec)
    return path


# every test file goes to a private temporary directory, deleted when the kernel exits;
# the tests never write into BASE_DIR
TEST_TMP = tempfile.mkdtemp(prefix="ir_hw1_tests_")
atexit.register(shutil.rmtree, TEST_TMP, True)


def tmp_path(*parts: str) -> str:
    """Path inside TEST_TMP; parent directories are created."""
    p = os.path.join(TEST_TMP, *parts)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    return p


FIXTURE_WARC = write_sample_warc(tmp_path("sample.warc.gz"))

# two short documents for the query tests; their positions are worked out in Section 6.6
QUERY_DOCS = [
    Document(1, "doc-1", "http://example.com/1", "Information Retrieval",
             "Information retrieval system builds an inverted index to search engines. Please come home."),
    Document(2, "doc-2", "http://example.com/2", "Messy & Tricky",
             "Café naïve résumé with U.S.A. and state-of-the-art information retrieval. Please stay home."),
]

# random corpus for brute-force comparisons: 80 documents over 10 words with random gaps between
# positions (as left by stopword removal); documents 25, 50 and 75 are empty
RANDOM_WORDS = ["alpha", "beta", "gamma", "delta", "epsilon", "zeta", "eta", "theta", "iota", "kappa"]
# the random words are not stopwords and should not be stemmed
RANDOM_OPT = dict(stopword_removal=False, stemming=False)
SEED = 20260101


def make_random_corpus(seed: int = SEED, n_docs: int = 80):
    rng = random.Random(seed)
    corpus = []
    for d in range(1, n_docs + 1):
        pos, terms = 0, []
        for _ in range(0 if d % 25 == 0 else rng.randint(1, 40)):
            terms.append((rng.choice(RANDOM_WORDS), pos))
            pos += rng.choice([1, 1, 1, 2, 3])
        corpus.append((d, f"doc-{d}", terms))
    return corpus


def build_index_variants(docs, out_dir: str, **cfg_kw) -> Dict[str, Any]:
    """Build the same documents in every supported way; returns {variant: index}.

    memory: in-memory; noskip: in-memory without skip pointers; spimi: SPIMI blocks, merged into memory;
    disk: streaming SPIMI + trie of byte offsets; notrie: streaming, lookups scan postings.txt;
    multiline: streaming with the slide's multi-line postings layout.
    Streaming variants write {variant}_dict.txt, _post.txt, _map.txt, _terms.txt into out_dir."""
    docs = list(docs)
    small = dict(cfg_kw, spimi_block_limit=25)

    def stream(name, multiline=False, **extra):
        ix, _ = PositionalIndex.build_and_save_streaming(
            iter(docs), *(os.path.join(out_dir, f"{name}_{k}.txt") for k in ("dict", "post", "map", "terms")),
            cfg=Config(**dict(small, **extra)), multiline_postings=multiline)
        return ix

    return {
        "memory": PositionalIndex.build_from_iter(iter(docs), Config(**cfg_kw)),
        "noskip": PositionalIndex.build_from_iter(iter(docs), Config(**dict(cfg_kw, use_skip_pointers=False))),
        "spimi": PositionalIndex.build_from_iter(iter(docs), Config(**small), use_spimi=True),
        "disk": stream("disk"),
        "notrie": stream("notrie", use_trie=False),
        "multiline": stream("multiline", multiline=True),
    }


# ---------- traceability: homework requirement -> checks (printed with results in Section 8.5) ----------
REQUIREMENTS = [
    ("Input: ClueWeb09 WARC records (p.2, p.4)", "P01-P08 P17 P18 G01 G11"),
    ("Preprocessing: HTTP header, charset, HTML -> text (p.6)", "P09-P16 P19-P27 G02-G06 G10 X01-X05"),
    ("Tokenizer: digits, hyphens, punctuation (p.6)", "T01-T25 A05 A06"),
    ("pos = token position in the document body (p.6)", "T09 T18 L04 A03-A05 G07 G08"),
    ("Linguistic: case folding, stopwords, lemmatization, stemming (p.7)", "L01-L17 A07 O30-O36 G09"),
    ("Optional features switched on/off by a parameter (p.7)", "A01 A02 A23 L09 L10 L12 C01-C22"),
    ("Dictionary: sorted vocabulary, one term per line, with df (p.5, p.6)", "I13 I14 I16 O04-O12"),
    ("Postings: term, df: <doc, tf: <pos, ...>; ...> (p.5, p.6)", "I01-I03 I11 I13 I19 O13-O29"),
    ("Index correctness on the test collection (p.9)", "A08-A22 I04-I07 I12 I15 I17 I20 G01-G08 D03 M01-M03"),
    ("Large corpus: SPIMI blocks, streaming output (p.8 efficiency)", "I05-I09 I18 D01-D05"),
    ("Bonus: trie dictionary (p.7)", "I10 B01-B05 M04"),
    ("Bonus: skip pointers (p.7)", "B06-B10 M05"),
    ("Bonus: phrase query and proximity search (p.6)", "B11-B19 M06"),
]


def _expand_ids(spec: str) -> List[str]:
    """'P01-P03 G05' -> ['P01', 'P02', 'P03', 'G05']"""
    ids = []
    for part in spec.split():
        if "-" in part:
            a, b = part.split("-")
            ids += [f"{a[0]}{n:02d}" for n in range(int(a[1:]), int(b[1:]) + 1)]
        else:
            ids.append(part)
    return ids


print(f"test files go to {TEST_TMP} (removed when the kernel exits)")

test files go to C:\Users\Lab1323\AppData\Local\Temp\ir_hw1_tests_1rf17lgf (removed when the kernel exits)


### 6.1 Parser（P01–P27）

- **WARC reader**：每個 record 都讀到且順序正確、content 長度等於 `Content-Length`、`.warc` 與 `.warc.gz` 結果相同；容錯：缺少或錯誤（非數字、負數）的 `Content-Length`、被截斷的 `.warc.gz`、空檔案、record 之間多餘的空行。
- **HTTP 與編碼**：`split_http` 取最早的空行（regression）、沒有空行時整段都是 payload；`_detect_charset` 的優先順序（HTTP header → 前 4 KB 的 `<meta>` → UTF-8）、別名（`big5` → `cp950` 等）、略過不認得的編碼名稱。
- **HTML 轉文字**：刪除 script / style / noscript / 註解、解碼 entity、壓縮空白、提早出現的 `</body>`（regression）、`<?xml?>` 宣告、regex 備援、無法解碼的 byte。
- **`parse_warc`**：只有 `response` 變成文件、docID 連續、空頁面也有 docID；測試文件 1–3 的 title 與內文逐字比對（包括 Big5 頁面）。
- **評分報告指出的問題（regression）**：NUL 不會截斷頁面（P22，ClueWeb09 doc 762）、沒有 `;` 的 `&nbsp`（P23，doc 145）、`&#146;`（P24，doc 9）、每個 tag 都分隔字（P25：選單連結 doc 4623 不黏在一起，代價是 doc 15 的 `C ORPORATE`）、UTF-8 頁面夾雜 cp1252 byte（P26）、低位元組引號亂碼（P27，8.5 找到）。

In [13]:
ts = TestSuite("6.1 Parser")

# ------------------------------------------------------------------
ts.section("WARC reader: iter_warc_records")
recs = list(iter_warc_records(FIXTURE_WARC))
ts.eq("P01 every record is read, in file order", [r.headers.get("WARC-Type") for r in recs],
      ["warcinfo"] + ["response"] * 5)
ts.ok("P02 content is exactly Content-Length bytes",
      all(len(r.content) == int(r.headers["Content-Length"]) for r in recs))
plain_warc = tmp_path("parser", "sample.warc")
with gzip.open(FIXTURE_WARC, "rb") as fin, open(plain_warc, "wb") as fout:
    fout.write(fin.read())
ts.eq("P03 gzip is detected from the magic bytes: .warc and .warc.gz give the same records",
      [(r.headers, r.content) for r in iter_warc_records(plain_warc)], [(r.headers, r.content) for r in recs])

# hand-made records with CRLF line endings
_body = b"HTTP/1.1 200 OK\r\n\r\n<html><body>Robustness test</body></html>"
_good = (b"WARC/0.18\r\nWARC-Type: response\r\nWARC-TREC-ID: doc-valid\r\nContent-Length: "
         + str(len(_body)).encode() + b"\r\n\r\n" + _body + b"\r\n\r\n")


def _write_bytes(name: str, data: bytes) -> str:
    p = tmp_path("parser", name)
    Path(p).write_bytes(data)
    return p


p = _write_bytes("no_length.warc", b"WARC/0.18\r\nWARC-Type: response\r\nWARC-TREC-ID: doc-empty\r\n\r\n\r\n" + _good)
ts.eq("P04 missing Content-Length -> empty document; the next record is still read",
      [(d.trec_id, d.body) for d in parse_warc(p)], [("doc-empty", ""), ("doc-valid", "Robustness test")])
p = _write_bytes("bad_length.warc",
                 b"WARC/0.18\r\nWARC-Type: response\r\nWARC-TREC-ID: doc-bad\r\nContent-Length: abc\r\n\r\n"
                 + _body + b"\r\n\r\n"
                 + b"WARC/0.18\r\nWARC-Type: response\r\nWARC-TREC-ID: doc-neg\r\nContent-Length: -5\r\n\r\n"
                 + _body + b"\r\n\r\n" + _good)
ts.eq("P05 non-numeric / negative Content-Length -> empty document, the reader resyncs on the next record",
      [(d.trec_id, d.body) for d in parse_warc(p)], [("doc-bad", ""), ("doc-neg", ""), ("doc-valid", "Robustness test")])
p = tmp_path("parser", "truncated.warc.gz")
with gzip.open(p, "wb") as f:
    f.write(_good * 3)
Path(p).write_bytes(Path(p).read_bytes()[:-10])
try:
    _trunc = list(parse_warc(p))
    ts.ok("P06 truncated .warc.gz: complete records are returned, then the reader stops without an exception",
          1 <= len(_trunc) <= 3 and all(d.body == "Robustness test" for d in _trunc), f"{_trunc}")
except Exception as e:
    ts.ok("P06 truncated .warc.gz: complete records are returned, then the reader stops without an exception",
          False, f"raised {type(e).__name__}: {e}")
ts.eq("P07 empty file -> no records", list(parse_warc(_write_bytes("empty.warc", b""))), [])
p = _write_bytes("blank_lines.warc", b"\r\n\n" + _good + b"\n\n\n" + _good)
ts.eq("P08 extra blank lines before and between records are skipped",
      [(d.doc_id, d.body) for d in parse_warc(p)], [(1, "Robustness test"), (2, "Robustness test")])

# ------------------------------------------------------------------
ts.section("HTTP header, charset, HTML -> text")
ts.eq("P09 split_http cuts at the EARLIEST blank line (\\n\\n header, \\r\\n\\r\\n inside the HTML)",
      split_http(b"HTTP/1.1 200 OK\nA: b\n\n<p>x</p>\r\n\r\n</html>"),
      (b"HTTP/1.1 200 OK\nA: b", b"<p>x</p>\r\n\r\n</html>"))
ts.eq("P10 split_http: CRLF header; no blank line at all -> everything is payload",
      [split_http(b"HTTP/1.1 200 OK\r\nA: b\r\n\r\n<p>y</p>"), split_http(b"<p>no header</p>")],
      [(b"HTTP/1.1 200 OK\r\nA: b", b"<p>y</p>"), (b"", b"<p>no header</p>")])
ts.eq("P11 charset: HTTP header > <meta> in the first 4 KB > utf-8; aliases; unknown codecs are skipped",
      [_detect_charset(b"Content-Type: text/html; charset=big5", b"<meta charset='utf-8'>"),
       _detect_charset(b"Content-Type: text/html", b"<meta http-equiv='Content-Type' content='text/html; charset=GB2312'>"),
       _detect_charset(b"Content-Type: text/html; charset=x-bogus", b'<meta charset="ISO-8859-1">'),
       _detect_charset(b"", b"<html>"),
       _detect_charset(b"", b" " * 5000 + b"<meta charset='big5'>")],
      ["cp950", "gb18030", "cp1252", "utf-8", "utf-8"])
ts.eq("P12 html_to_text: <script>/<style>/<noscript>/comments dropped, entities decoded, whitespace collapsed",
      html_to_text("<html><head><title> A &amp;  B </title><style>p{x}</style></head><body><script>var s;</script>"
                   "<noscript>no</noscript><!-- c --><p>Hello\n\n  world</p></body></html>"),
      ("A & B", "Hello world"))
ts.eq("P13 text after a premature </body> is kept (regression)",
      html_to_text("<html><head><title>T</title></head><body></body><p>after body</p></html>"), ("T", "after body"))
ts.eq("P14 <?xml?> declaration is removed before lxml parses the page",
      html_to_text("<?xml version='1.0' encoding='utf-8'?><html><body><p>XHTML pages also work.</p></body></html>"),
      ("", "XHTML pages also work."))
_rt = _regex_text("<html><head><title>A &amp; B</title><style>x</style></head>"
                  "<body><!--c--><script>s</script><p>Hi</p></body></html>")
ts.eq("P15 regex fallback: same cleaning rules; pages lxml rejects (comment only, empty) give ('', '')",
      [(_rt[0], " ".join(_rt[1].split())), html_to_text("<!-- only a comment -->"), html_to_text("")],
      [("A & B", "Hi"), ("", ""), ("", "")])


def _page_body(name: str, html_bytes: bytes, content_type: str = "text/html") -> str:
    """Body text of a one-record WARC file holding html_bytes."""
    http = f"HTTP/1.1 200 OK\r\nContent-Type: {content_type}\r\n\r\n".encode("ascii") + html_bytes
    p = _write_bytes(name, b"WARC/0.18\r\nWARC-Type: response\r\nWARC-TREC-ID: x\r\nContent-Length: "
                     + str(len(http)).encode() + b"\r\n\r\n" + http + b"\r\n\r\n")
    return [d.body for d in parse_warc(p)][0]


try:
    # \x81 is invalid UTF-8 and undefined in cp1252 too; \x82 is not valid Shift-JIS here
    _got = [_page_body("undecodable.warc", b"<html><body>caf\x81 x</body></html>"),
            _page_body("undecodable_sjis.warc", b"<p>a\x82 b</p>", "text/html; charset=shift_jis")]
    ts.eq("P16 bytes that cannot be decoded become U+FFFD; nothing raises", _got, ["caf� x", "a� b"])
except Exception as e:
    ts.ok("P16 bytes that cannot be decoded become U+FFFD; nothing raises", False, f"raised {type(e).__name__}: {e}")

# ------------------------------------------------------------------
ts.section("parse_warc on the sample collection")
docs = list(parse_warc(FIXTURE_WARC))
ts.eq("P17 only 'response' records become documents, numbered 1..N with their TREC id and URL",
      [(d.doc_id, d.trec_id, d.url) for d in docs],
      [(1, "clueweb09-en0000-00-00000", "http://example.com/ir"),
       (2, "clueweb09-en0000-00-00001", "http://example.com/messy"),
       (3, "clueweb09-zh0000-00-00000", "http://example.com.tw/"),
       (4, "clueweb09-en0000-00-00002", "http://example.com/xhtml"),
       (5, "clueweb09-en0000-00-00003", "http://example.com/empty")])
ts.eq("P18 a response without HTML still gets a doc_id (doc_id <-> TREC id stay aligned)",
      (docs[4].doc_id, docs[4].title, docs[4].body), (5, "", ""))
ts.eq("P19 doc 2: title and visible text exactly (style, script and comment gone, entities decoded)",
      (docs[1].title, docs[1].body),
      ("Messy & Tricky",
       "The U.S.A. spent $1,000,000 on state-of-the-art e-mail systems in 2009. "
       "John's mp3 player doesn't support version 2.0.1; pi is 3.14 & IP is 192.168.0.1. "
       "Call 555-1234 (e.g. after 5pm). Café naïve résumé"))
ts.eq("P20 doc 3: Big5 page decoded through the <meta> charset", (docs[2].title, docs[2].body),
      ("資訊檢索", "資訊檢索（IR）課程 HW1：建立 inverted index！"))
ts.eq("P21 doc 1: <h1> and <p> text, title not repeated in the body", (docs[0].title, docs[0].body),
      ("Information Retrieval",
       "Information Retrieval Information retrieval (IR) is finding material of an unstructured nature "
       "that satisfies an information need from within large collections."))

# ------------------------------------------------------------------
ts.section("Text clean-up (regressions from the grading report; ClueWeb09 doc ids in brackets)")
ts.eq("P22 NUL and other control characters are dropped; the text after them is kept [762]",
      html_to_text("<h1>Human Ecology</h1>\x00\r\n<p>more text\x07 here</p>"), ("", "Human Ecology more text here"))
ts.eq("P23 legacy entities without ';' are decoded, so 'nbsp' is not indexed as a word [145]",
      html_to_text("<p>Email: &nbsp<input> &copy 2008 AT&T &amp;</p>")[1], "Email: © 2008 AT&T &")
ts.eq("P24 C1 references / characters are read as cp1252: &#146; -> U+2019, possessive kept whole [9]",
      [html_to_text("<p>the client&#146;s view &#147;quoted&#148; it\x92s</p>")[1],
       toks(html_to_text("<p>the client&#146;s view</p>")[1])],
      ["the client’s view “quoted” it’s", ["the", "client", "view"]])
# Every tag separates words. Joining across inline tags was tried and reverted: it fixes rare split words
# ("<big>C</big>ORPORATE" [15]) but glues far more words that CSS shows apart (menus [4623], badges [12072]).
_tags = "<a href=x>Home</a><a href=y>About</a> Hotels<em>beta</em> <big>C</big>ORPORATE<td>x</td><td>y</td>"
ts.eq("P25 every tag separates words, in the lxml path and the regex fallback (links, badges, drop caps)",
      [html_to_text(_tags)[1], " ".join(_regex_text(_tags)[1].split())], ["Home About Hotels beta C ORPORATE x y"] * 2)
ts.eq("P26 a UTF-8 page with stray cp1252 bytes: only the invalid bytes are read as cp1252",
      # cp1252 \xe9 / \xef next to valid UTF-8 sequences for an em dash and a CJK character
      _page_body("mixed_utf8_cp1252.warc", b"<p>caf\xe9 na\xefve \xe2\x80\x94 \xe5\xa4\xa7</p>"),
      "café naïve — 大")
# found by the independent check of 8.5: "don\x19t" is U+2019 with only its low byte left
ts.eq("P27 low-byte punctuation mojibake (\\x19 = ’, \\x1c\\x1d = “ ”, \\x13 = –) is restored, not deleted",
      [html_to_text("<p>don\x19t \x1cquoted\x1d a\x13b</p>")[1], toks(html_to_text("<p>don\x19t stop</p>")[1])],
      ["don’t “quoted” a–b", ["don't", "stop"]])
ts.done()

===== 6.1 Parser =====

--- WARC reader: iter_warc_records ---
[PASS] P01 every record is read, in file order
[PASS] P02 content is exactly Content-Length bytes
[PASS] P03 gzip is detected from the magic bytes: .warc and .warc.gz give the same records
[PASS] P04 missing Content-Length -> empty document; the next record is still read
[PASS] P05 non-numeric / negative Content-Length -> empty document, the reader resyncs on the next record
[PASS] P06 truncated .warc.gz: complete records are returned, then the reader stops without an exception
[PASS] P07 empty file -> no records
[PASS] P08 extra blank lines before and between records are skipped

--- HTTP header, charset, HTML -> text ---
[PASS] P09 split_http cuts at the EARLIEST blank line (\n\n header, \r\n\r\n inside the HTML)
[PASS] P10 split_http: CRLF header; no blank line at all -> everything is payload
[PASS] P11 charset: HTTP header > <meta> in the first 4 KB > utf-8; aliases; unknown codecs are skipped
[PASS] P12 html_to_text: <

### 6.2 Tokenizer（T01–T25）

投影片 p.6 要求說明的數字、連字號、標點處理，每條規則（見第 2 節的表）各有一項檢查，另外包括 Unicode 正規化、組合符號（T21）、數字接單位與型號（T22、T23）、apostrophe 替代字元（T24）、看不見的字元（T25）、position 計算、`trace=True` 的明細。T20 把測試文件 2（含所有規則）的每個 token 與 position 和人工推導的結果逐一比對。

In [14]:
ts = TestSuite("6.2 Tokenizer")

ts.section("Digits")
ts.eq("T01 digits and years are kept", toks("room 101 in 2009"), ["room", "101", "in", "2009"])
ts.eq("T02 thousands separators are removed", toks("$1,000,000 and 1,234.5"), ["1000000", "and", "1234.5"])
ts.eq("T03 decimals, version numbers and IP addresses stay whole", toks("3.14 2.0.1 192.168.0.1"),
      ["3.14", "2.0.1", "192.168.0.1"])
ts.eq("T04 a sentence-final '.' is not part of a number", toks("in 2009. Then 3.14."), ["in", "2009", "Then", "3.14"])
ts.eq("T05 letter+digit tokens are kept whole", toks("mp3 covid19 5pm"), ["mp3", "covid19", "5pm"])
ts.eq("T22 a number glued to a unit is split from it, and the number stays whole",
      toks("3.5mm 2.4GHz 1,450cc 1,000th 2.5-inch"), ["3.5", "mm", "2.4", "GHz", "1450", "cc", "1000", "th", "2.5", "inch"])
ts.eq("T23 version / model numbers stay whole; lists, abbreviations and glued text are not joined",
      toks("v2.0.1 PM2.5 web2.0. 8,1995 No.1 Fig.3 SP2.Then v2.0.1beta x3.5mm"),
      ["v2.0.1", "PM2.5", "web2.0", "8", "1995", "No", "1", "Fig", "3", "SP2", "Then", "v2", "0.1", "beta", "x3", "5mm"])

ts.section("Hyphens")
ts.eq("T06 hyphen_mode='split' (default): one token per part, also for digits",
      toks("state-of-the-art 555-1234"), ["state", "of", "the", "art", "555", "1234"])
ts.eq("T07 hyphen_mode='join'", toks("state-of-the-art e-mail", hyphen_mode="join"), ["stateoftheart", "email"])
ts.eq("T08 hyphen_mode='keep'", toks("state-of-the-art e-mail", hyphen_mode="keep"), ["state-of-the-art", "e-mail"])
ts.eq("T09 every split part takes its own position", [(x.text, x.pos) for x in tokenize("a b-c d")],
      [("a", 0), ("b", 1), ("c", 2), ("d", 3)])

ts.section("Punctuation, acronyms, apostrophes")
ts.eq("T10 punctuation and '_' are separators", toks("hello, world! (foo) a/b; c?d under_score ..."),
      ["hello", "world", "foo", "a", "b", "c", "d", "under", "score"])
ts.eq("T11 acronyms lose their dots", toks("U.S.A. e.g."), ["USA", "eg"])
ts.eq("T12 possessive 's is removed (any case, curly apostrophe too); other apostrophes are kept",
      toks("John's JOHN'S don't O'Neill John’s"), ["John", "JOHN", "don't", "O'Neill", "John"])
ts.eq("T24 ` ´ ʼ between two letters count as an apostrophe; elsewhere they are separators",
      toks("Let`s don\u00b4t can\u02bct O\u00b4Neill `code` a \u00b4 b"),
      ["Let", "don't", "can't", "O'Neill", "code", "a", "b"])

ts.section("Unicode")
ts.eq("T13 CJK characters are single-character tokens", toks("資訊檢索IR"), ["資", "訊", "檢", "索", "IR"])
ts.eq("T14 NFKC: full-width forms and ligatures are normalized", toks("ＩＲ　ｈｗ１ ﬁle"), ["IR", "hw1", "file"])
ts.eq("T15 the Unicode hyphen U+2010 is treated as '-'", toks("state‐of"), ["state", "of"])
ts.eq("T16 accented letters stay inside words", toks("Café naïve résumé"), ["Café", "naïve", "résumé"])
ts.eq("T25 invisible characters (zero-width space, word joiner, BOM, soft hyphen) do not split words",
      toks("Shuff\u200ble infor\u00admation zero\u2060width \ufeffBOM"), ["Shuffle", "information", "zerowidth", "BOM"])
ts.eq("T17 empty or punctuation-only text gives no tokens", toks("") + toks("!!! ... --- '' $"), [])
# q + U+0303 has no precomposed form, so NFKC leaves the combining tilde in place
ts.eq("T21 combining marks stay inside words (Devanagari vowel signs and virama, a combining tilde)",
      toks("हिन्दी भाषा q̃x"), ["हिन्दी", "भाषा", "q̃x"])

ts.section("Positions and trace")
ts.eq("T18 positions are consecutive token numbers starting at pos_base",
      [(x.text, x.pos) for x in tokenize("a b-c", Config(pos_base=1))], [("a", 1), ("b", 2), ("c", 3)])
_, _rows = tokenize("U.S.A. state-of-the-art 3.14!", trace=True)
ts.eq("T19 trace=True reports (raw match, rule, tokens, positions) per match", _rows,
      [("U.S.A.", "ACRONYM", ["USA"], [0]),
       ("state-of-the-art", "WORD", ["state", "of", "the", "art"], [1, 2, 3, 4]),
       ("3.14", "NUMBER", ["3.14"], [5])])

# doc 2 of the sample collection (title + body), every token and position worked out by hand
_d2 = list(parse_warc(FIXTURE_WARC))[1]
_golden = ("Messy Tricky The USA spent 1000000 on state of the art e mail systems in 2009 John mp3 player doesn't "
           "support version 2.0.1 pi is 3.14 IP is 192.168.0.1 Call 555 1234 eg after 5pm Café naïve résumé").split()
ts.eq("T20 sample doc 2: every token and position (hand-worked)",
      [(x.text, x.pos) for x in tokenize(f"{_d2.title} {_d2.body}")], [(w, i) for i, w in enumerate(_golden)])
ts.done()

===== 6.2 Tokenizer =====

--- Digits ---
[PASS] T01 digits and years are kept
[PASS] T02 thousands separators are removed
[PASS] T03 decimals, version numbers and IP addresses stay whole
[PASS] T04 a sentence-final '.' is not part of a number
[PASS] T05 letter+digit tokens are kept whole
[PASS] T22 a number glued to a unit is split from it, and the number stays whole
[PASS] T23 version / model numbers stay whole; lists, abbreviations and glued text are not joined

--- Hyphens ---
[PASS] T06 hyphen_mode='split' (default): one token per part, also for digits
[PASS] T07 hyphen_mode='join'
[PASS] T08 hyphen_mode='keep'
[PASS] T09 every split part takes its own position

--- Punctuation, acronyms, apostrophes ---
[PASS] T10 punctuation and '_' are separators
[PASS] T11 acronyms lose their dots
[PASS] T12 possessive 's is removed (any case, curly apostrophe too); other apostrophes are kept
[PASS] T24 ` ´ ʼ between two letters count as an apostrophe; elsewhere they are separators

--- Unicod

### 6.3 Linguistic module（L01–L17）

投影片 p.7 的四個步驟各自的開與關、stopword 不分大小寫且在 stemming 之前比對（`others` / `owned` / `DOS` 不會被誤刪）、移除後保留 gap、lemmatization 在 stemming 之前、數字與中文不處理、關閉 case folding 時保留大小寫樣式、`Config.baseline()`、`trace=True` 的各步驟輸出。

需要 NLTK WordNet 的 lemmatization 檢查集中在 L13–L17；沒有安裝 WordNet 時這幾項標成 `SKIP`，其他檢查照常執行。L16、L17 檢查 WordNet lemmatizer 的品質；L17 預期是 `WARN`（WordNet 把 `species` 還原成 `specie`），lemmatization 預設關閉，不影響索引。

In [15]:
ts = TestSuite("6.3 Linguistic module")

ts.section("Case folding")
ts.eq("L01 case_folding=True lower-cases; False keeps the case",
      [words(ling("Hello WORLD", stopword_removal=False, stemming=False)),
       words(ling("Hello WORLD", case_folding=False, stopword_removal=False, stemming=False))],
      [["hello", "world"], ["Hello", "WORLD"]])

ts.section("Stopword removal")
ts.eq("L02 stopword_removal=True drops NLTK stopwords; False keeps them",
      [words(ling("the cat and the dog", stemming=False)), words(ling("the cat", stopword_removal=False, stemming=False))],
      [["cat", "dog"], ["the", "cat"]])
ts.eq("L03 stopwords are matched case-insensitively, also with case_folding=False",
      ling("The Cat", case_folding=False, stemming=False), [("Cat", 1)])
ts.eq("L04 removed tokens leave gaps: positions are never renumbered (pos_base 0 and 1)",
      [ling("the cat and the dog", stemming=False), ling("The cat", pos_base=1, stemming=False)],
      [[("cat", 1), ("dog", 4)], [("cat", 2)]])
ts.eq("L05 stopwords are matched on the surface word before stemming: others/owned/DOS/willing stay indexed",
      ling("the others owned DOS willing will"), [("other", 1), ("own", 2), ("do", 3), ("will", 4)])

ts.section("Stemming")
ts.eq("L06 stemming=True: Porter stems; False keeps the word",
      [words(ling("running dogs retrieval")), words(ling("running dogs retrieval", stemming=False))],
      [["run", "dog", "retriev"], ["running", "dogs", "retrieval"]])
ts.eq("L07 numbers, CJK and letter+digit tokens are not stemmed",
      ling("The 2009 資訊 mp3 running"), [("2009", 1), ("資", 2), ("訊", 3), ("mp3", 4), ("run", 5)])
ts.eq("L08 case_folding=False: stems keep the casing pattern, all-caps acronyms survive",
      [ling("The Running USA", case_folding=False), [_stem("running"), _stem("RUNNING"), _stem("Running")]],
      [[("Run", 1), ("USA", 2)], ["run", "RUN", "Run"]])

ts.section("Switches and trace")
_base = Config.baseline()
ts.eq("L09 Config.baseline() leaves every token untouched",
      linguistic(tokenize("To be Running", _base), _base), [("To", 0), ("be", 1), ("Running", 2)])
_over = Config.baseline(case_folding=True)
ts.eq("L10 Config.baseline(**overrides) changes single fields", words(linguistic(tokenize("To Be", _over), _over)),
      ["to", "be"])
_, _stages = linguistic(tokenize("The Running Dogs"), trace=True)
ts.eq("L11 trace=True: the output of every enabled step, in execution order",
      list(_stages.items()),
      [("input", [("The", 0), ("Running", 1), ("Dogs", 2)]),
       ("case_folding", [("the", 0), ("running", 1), ("dogs", 2)]),
       ("stopword_removal", [("running", 1), ("dogs", 2)]),
       ("stemming", [("run", 1), ("dog", 2)])])
ts.eq("L12 lemmatization is off by default", words(ling("children mice", stemming=False)), ["children", "mice"])

ts.section("Lemmatization (optional, needs NLTK WordNet)")
_lemma_checks = ["L13 lemmatization=True: WordNet lemma, noun first, then verb",
                 "L14 lemmatization runs before stemming (mice -> mouse -> mous)",
                 "L15 lemmas keep the casing pattern; numbers / CJK are not lemmatized",
                 "L16 plurals, irregular forms and acronyms",
                 "L17 words ending in -s that are not plurals"]
if not HAVE_WORDNET:
    for _name in _lemma_checks:
        ts.skip(_name, "NLTK WordNet is not installed")
else:
    ts.eq(_lemma_checks[0],
          words(ling("children mice went studies", lemmatization=True, stemming=False, stopword_removal=False)),
          ["child", "mouse", "go", "study"])
    ts.eq(_lemma_checks[1], words(ling("mice", lemmatization=True)), ["mous"])
    ts.eq(_lemma_checks[2],
          [ling("The Running Mice USA 2009 資", case_folding=False, lemmatization=True, stemming=False),
           [_lemmatize("MICE"), _lemmatize("USA")]],
          [[("Run", 1), ("Mouse", 2), ("USA", 3), ("2009", 4), ("資", 5)], ["MOUSE", "USA"]])
    _lem = {"children": "child", "Mice": "Mouse", "USA": "USA", "studies": "study", "cities": "city",
            "boxes": "box", "dogs": "dog", "status": "status", "class": "class", "analysis": "analysis"}
    ts.eq(_lemma_checks[3], {w: _lemmatize(w) for w in _lem}, _lem)
    _weak = {"movies": "movie", "series": "series", "species": "species", "news": "news", "lens": "lens",
             "always": "always"}
    ts.ok(_lemma_checks[4], {w: _lemmatize(w) for w in _weak} == _weak,
          "WordNet gives " + str({w: _lemmatize(w) for w in _weak if _lemmatize(w) != _weak[w]})
          + " (only matters with lemmatization=True)", level="WARN")
ts.done()

===== 6.3 Linguistic module =====

--- Case folding ---
[PASS] L01 case_folding=True lower-cases; False keeps the case

--- Stopword removal ---
[PASS] L02 stopword_removal=True drops NLTK stopwords; False keeps them
[PASS] L03 stopwords are matched case-insensitively, also with case_folding=False
[PASS] L04 removed tokens leave gaps: positions are never renumbered (pos_base 0 and 1)
[PASS] L05 stopwords are matched on the surface word before stemming: others/owned/DOS/willing stay indexed

--- Stemming ---
[PASS] L06 stemming=True: Porter stems; False keeps the word
[PASS] L07 numbers, CJK and letter+digit tokens are not stemmed
[PASS] L08 case_folding=False: stems keep the casing pattern, all-caps acronyms survive

--- Switches and trace ---
[PASS] L09 Config.baseline() leaves every token untouched
[PASS] L10 Config.baseline(**overrides) changes single fields
[PASS] L11 trace=True: the output of every enabled step, in execution order
[PASS] L12 lemmatization is off by default

--- Le

### 6.4 Pipeline API 與中間檔（A01–A23）

- `Config()` 的預設值與第 0 節的表一致；`Config.baseline()` 只關閉四個語言處理步驟。
- `process_document`：`include_title` 的開關；一份人工推導的網頁，Baseline 與 Enhanced 兩種設定下每個 (term, position) 都正確，並用一個獨立的 regex tokenizer 比對字彙。
- `iter_processed_docs`：每個 response 一筆、結果等於逐篇呼叫 `process_document`、`max_docs`。
- `iter_processed_file`（第 4 節的兩種模式）：照 7.3 的方式寫出中間檔再讀回，結果必須與直接處理 WARC 相同；檔名含 `tok` 會再處理、含 `ling` 不會（**不會重複 stemming**）；檔名都不含時自動判斷；position 平移；純文字與自由文字欄位的備援；`raw_tokens` 明確指定時不再依檔名猜測（A23）。

In [16]:
ts = TestSuite("6.4 Pipeline API & intermediate files")

ts.section("Config")
_c = Config()
ts.eq("A01 Config() defaults match the table in Section 0",
      (_c.include_title, _c.hyphen_mode, _c.pos_base, _c.case_folding, _c.stopword_removal, _c.lemmatization,
       _c.stemming, _c.use_trie, _c.use_skip_pointers, _c.spimi_block_limit),
      (True, "split", 0, True, True, False, True, True, True, 2_500_000))
_c = Config.baseline()
ts.eq("A02 Config.baseline() switches off the four linguistic steps and nothing else",
      (_c.case_folding, _c.stopword_removal, _c.lemmatization, _c.stemming,
       _c.include_title, _c.hyphen_mode, _c.pos_base, _c.use_trie, _c.use_skip_pointers),
      (False, False, False, False, True, "split", 0, True, True))

ts.section("process_document")
_doc = Document(1, "x", "u", "Title Words", "body text here")
ts.eq("A03 include_title=True: the title comes first and takes the first positions",
      process_document(_doc, Config(stemming=False)), [("title", 0), ("words", 1), ("body", 2), ("text", 3)])
ts.eq("A04 include_title=False: only the body is indexed",
      process_document(_doc, Config(include_title=False, stemming=False)), [("body", 0), ("text", 1)])
# hand-worked page: script in <head>, text after a premature </body>
_html = ("<html><head><title>To Be Or Not To Be</title><script>var secret_bug = 999;</script></head>"
         "<body><p>Welcome to Information Retrieval 2009!</p></body><p>Trailing text after premature body tag.</p></html>")
_title, _body = html_to_text(_html)
_g = Document(1, "golden", "", _title, _body)
_base_terms = process_document(_g, Config.baseline(case_folding=True))
ts.eq("A05 hand-worked page, Baseline + case folding: every (term, position)", _base_terms,
      [(w, i) for i, w in enumerate("to be or not to be welcome to information retrieval 2009 "
                                    "trailing text after premature body tag".split())])
ts.eq("A06 an independent regex tokenizer finds the same vocabulary on that page",
      set(re.findall(r"[a-z0-9]+", f"{_title} {_body}".lower())), set(words(_base_terms)))
ts.eq("A07 the same page with the default (Enhanced) Config: stopwords gone, gaps kept, Porter stems",
      process_document(_g),
      [("welcom", 6), ("inform", 8), ("retriev", 9), ("2009", 10), ("trail", 11), ("text", 12),
       ("prematur", 14), ("bodi", 15), ("tag", 16)])

ts.section("iter_processed_docs on the sample collection")
docs = list(parse_warc(FIXTURE_WARC))
out = list(iter_processed_docs(FIXTURE_WARC))
ts.eq("A08 one (doc_id, trec_id, terms) per response record", [(d, tr) for d, tr, _ in out],
      [(d.doc_id, d.trec_id) for d in docs])
ts.eq("A09 terms == process_document(parse_warc(...))", [terms for _, _, terms in out], [process_document(d) for d in docs])
ts.eq("A10 doc 1 starts with its title 'Information Retrieval'", out[0][2][:5],
      [("inform", 0), ("retriev", 1), ("inform", 2), ("retriev", 3), ("inform", 4)])
ts.eq("A11 a page without text gives an empty term list", out[4][2], [])
ts.eq("A12 max_docs stops after that doc_id", [d for d, _, _ in iter_processed_docs(FIXTURE_WARC, max_docs=2)], [1, 2])

ts.section("Intermediate files: doc_id<TAB>trec_id<TAB>term:pos ... (iter_processed_file)")
# written exactly like Section 7.3 does; the file names contain 'tok' / 'ling'
_tok_file = tmp_path("api", "sample_tokenizer_output.txt")
_ling_file = tmp_path("api", "sample_linguistic_output.txt")
_raw = []
with open(_tok_file, "w", encoding="utf-8", newline="\n") as f_tok, \
     open(_ling_file, "w", encoding="utf-8", newline="\n") as f_ling:
    for d in docs:
        _tokens = tokenize(f"{d.title} {d.body}")
        _raw.append((d.doc_id, d.trec_id, [(x.text, x.pos) for x in _tokens]))
        f_tok.write(f"{d.doc_id}\t{d.trec_id}\t{_fmt((x.text, x.pos) for x in _tokens)}\n")
        f_ling.write(f"{d.doc_id}\t{d.trec_id}\t{_fmt(linguistic(_tokens))}\n")
ts.eq("A13 cfg=None streams a linguistic file verbatim (Enhanced index in 7.4) == iter_processed_docs",
      list(iter_processed_file(_ling_file)), out)
ts.eq("A14 cfg=None streams a tokenizer file verbatim (Baseline index in 7.4) == raw tokens",
      list(iter_processed_file(_tok_file)), _raw)
ts.eq("A15 with cfg, a 'tok' file goes through the linguistic module == iter_processed_docs",
      list(iter_processed_file(_tok_file, Config())), out)


def _write_text(name: str, text: str) -> str:
    p = tmp_path("api", name)
    with open(p, "w", encoding="utf-8", newline="\n") as f:
        f.write(text)
    return p


# 'pleas' would become 'plea' and 'other' / 'own' would be dropped if these terms were processed twice
_stems = _write_text("custom_ling.txt", "1\tdoc-1\tpleas:0 other:1 own:2\n")
ts.eq("A16 with cfg, a 'ling' file is not processed again (no double stemming)",
      list(iter_processed_file(_stems, Config())), [(1, "doc-1", [("pleas", 0), ("other", 1), ("own", 2)])])
ts.eq("A17 file name without 'tok'/'ling': raw tokens are recognized (capitals) and processed",
      list(iter_processed_file(_write_text("unnamed_a.txt", "1\tdoc-1\tThe:0 Others:1 Pleased:2\n"), Config())),
      [(1, "doc-1", [("other", 1), ("pleas", 2)])])
ts.eq("A18 file name without 'tok'/'ling': processed terms are recognized (lower case, < 20% stopwords) and kept",
      list(iter_processed_file(_write_text("unnamed_b.txt", "1\tdoc-1\tpleas:0 retriev:1 system:2 index:3 other:4 invert:5\n"),
                               Config())),
      [(1, "doc-1", [("pleas", 0), ("retriev", 1), ("system", 2), ("index", 3), ("other", 4), ("invert", 5)])])
ts.eq("A19 positions are shifted to cfg.pos_base", list(iter_processed_file(_stems, Config(pos_base=1))),
      [(1, "doc-1", [("pleas", 1), ("other", 2), ("own", 3)])])
ts.eq("A20 plain text: one document per line, doc_id = line number",
      list(iter_processed_file(_write_text("plain_docs.txt", "Information retrieval system\nSearch engine index\n"), Config())),
      [(1, "txt-00001", [("inform", 0), ("retriev", 1), ("system", 2)]),
       (2, "txt-00002", [("search", 0), ("engin", 1), ("index", 2)])])
ts.eq("A21 a tab-separated line whose third column is free text is processed as the document body",
      list(iter_processed_file(_write_text("free_text.txt", "7\tdoc-7\tThe Running Dogs\n"), Config())),
      [(7, "doc-7", [("run", 1), ("dog", 2)])])
ts.eq("A22 max_docs is honored", [d for d, _, _ in iter_processed_file(_ling_file, Config(), max_docs=2)], [1, 2])
# a processed file whose name happens to contain 'tok': the name-based guess is wrong, raw_tokens=False fixes it
_stokes = _write_text("stokes_ling.txt", "1\tdoc-1\tpleas:0 other:1\n")
ts.eq("A23 raw_tokens=True / False overrides the guess from the file name",
      [list(iter_processed_file(_stokes, Config()))[0][2], list(iter_processed_file(_stokes, Config(), raw_tokens=False))[0][2],
       list(iter_processed_file(_stems, Config(), raw_tokens=True))[0][2]],
      [[("plea", 0)], [("pleas", 0), ("other", 1)], [("plea", 0)]])
ts.done()

===== 6.4 Pipeline API & intermediate files =====

--- Config ---
[PASS] A01 Config() defaults match the table in Section 0
[PASS] A02 Config.baseline() switches off the four linguistic steps and nothing else

--- process_document ---
[PASS] A03 include_title=True: the title comes first and takes the first positions
[PASS] A04 include_title=False: only the body is indexed
[PASS] A05 hand-worked page, Baseline + case folding: every (term, position)
[PASS] A06 an independent regex tokenizer finds the same vocabulary on that page
[PASS] A07 the same page with the default (Enhanced) Config: stopwords gone, gaps kept, Porter stems

--- iter_processed_docs on the sample collection ---
[PASS] A08 one (doc_id, trec_id, terms) per response record
[PASS] A09 terms == process_document(parse_warc(...))
[PASS] A10 doc 1 starts with its title 'Information Retrieval'
[PASS] A11 a page without text gives an empty term list
[PASS] A12 max_docs stops after that doc_id

--- Intermediate files: doc_id<TAB

### 6.5 Indexer（I01–I20）

- **格式**：投影片 p.5 的範例（`to, 2: <1, 6: <7, 18, ...>; 2, 5: <...>>`）逐字比對單行與多行兩種排版，並能讀回。
- **與暴力法比對**（隨機語料）：記憶體內、SPIMI、串流 SPIMI 三種建法的 postings 都等於暴力法；串流索引透過 trie 的 byte offset、無 trie 的檔案掃描、多行格式讀回也都相同；不同區塊大小、`save_files`、`use_trie=False` 寫出的檔案逐 byte 相同；trie 的 byte offset 指向正確的行；空文件有 docID 但沒有 postings；四個輸出檔的內容逐行比對。
- **測試 WARC**：真實網頁上三種建法與暴力法一致、字典依 Unicode code point 排序、Baseline 保留 `to` / `retrieval` 而 Enhanced 不保留、兩次建索引輸出相同（決定性）；docID 沒有嚴格遞增時，三種建法都丟出 `ValueError`（I20）。

In [17]:
ts = TestSuite("6.5 Indexer")

ts.section("Postings record format (Slide 5)")
_slide = [PostingEntry(1, 6, [7, 18, 33, 72, 86, 231]), PostingEntry(2, 5, [1, 17, 74, 222, 255])]
ts.eq("I01 single-line record of the Slide 5 example",
      format_posting_standard("to", _slide), "to, 2: <1, 6: <7, 18, 33, 72, 86, 231>; 2, 5: <1, 17, 74, 222, 255>>")
ts.eq("I02 multi-line record (multiline_postings=True), laid out as on the slide",
      format_posting_standard("to", _slide, multiline=True),
      "to, 2:\n<1, 6: <7, 18, 33, 72, 86, 231>;\n2, 5: <1, 17, 74, 222, 255>>")
ts.eq("I03 parse_posting_line reads both layouts back",
      [(term, df, entries(pl)) for term, df, pl in
       (parse_posting_line(format_posting_standard("to", _slide, multiline=m)) for m in (False, True))],
      [("to", 2, entries(_slide))] * 2)

# ------------------------------------------------------------------
ts.section("Random corpus: every build method vs. brute force")
corpus = make_random_corpus()
truth = brute_force_index(corpus)
_dir = tempfile.mkdtemp(dir=TEST_TMP)
P = lambda name: os.path.join(_dir, name)
V = build_index_variants(corpus, _dir, **RANDOM_OPT)
ts.ok("I04 in-memory index == brute force", flat(V["memory"]) == truth)
ts.ok("I05 SPIMI (many blocks, merged in memory) == brute force", flat(V["spimi"]) == truth)
ts.ok("I06 streaming SPIMI: postings file == brute force (independent parser)",
      all(parse_post_file(P(f"{v}_post.txt")) == truth for v in ("disk", "notrie", "multiline")))
ts.ok("I07 streaming SPIMI: get_postings() == brute force (trie byte offset / sorted file scan / multi-line)",
      all(entries(V[v].get_postings(term)) == truth[term] for v in ("disk", "notrie", "multiline") for term in truth))
_, _st = PositionalIndex.build_and_save_streaming(iter(corpus), P("a_dict.txt"), P("a_post.txt"), P("a_map.txt"),
                                                  P("a_terms.txt"), cfg=Config(spimi_block_limit=25, **RANDOM_OPT))
ts.eq("I08 streaming stats: docs, tokens, vocabulary, several blocks",
      (_st["num_docs"], _st["total_tokens"], _st["vocab_size"], _st["num_blocks"] > 1),
      (len(corpus), sum(len(x[2]) for x in corpus), len(truth), True))
_, _ = PositionalIndex.build_and_save_streaming(iter(corpus), P("b_dict.txt"), P("b_post.txt"), P("b_map.txt"),
                                                P("b_terms.txt"), cfg=Config(spimi_block_limit=10**9, **RANDOM_OPT))
V["memory"].save_files(P("m_dict.txt"), P("m_post.txt"), P("m_map.txt"), P("m_terms.txt"))
_same = lambda x, y: all(sha256_file(P(f"{x}_{k}.txt")) == sha256_file(P(f"{y}_{k}.txt"))
                         for k in ("dict", "post", "map", "terms"))
ts.ok("I09 identical files for block limit 25 vs one block, in-memory save_files, and use_trie=False",
      _same("a", "b") and _same("a", "m") and _same("a", "notrie"))
_E = ErrBook()
with open(P("disk_post.txt"), "rb") as fb:
    for term in truth:
        df, off = V["disk"].trie.search(term)
        fb.seek(off)
        if not fb.readline().decode("utf-8").startswith(f"{term}, {df}: <"):
            _E.add("off", term)
ts.ok("I10 the trie's byte offsets point at the term's line in postings.txt", _E.ok("off"), _E.detail("off"))
ts.ok("I11 parse_posting_line round trip on every record (single-line and multi-line files)",
      all(entries(parse_posting_line(rec)[2]) == truth[parse_posting_line(rec)[0]]
          for v in ("disk", "multiline") for _, rec in iter_records(P(f"{v}_post.txt"))))
_empty = [d for d, _, terms in corpus if not terms]
ts.ok("I12 empty documents are counted and mapped but have no postings",
      _empty and all(V[v].num_docs == len(corpus) and all(d in V[v].doc_map for d in _empty) for v in ("memory", "disk"))
      and not any(e[0] in _empty for pl in truth.values() for e in pl), f"empty docs {_empty}")
ts.eq("I13 output files: dictionary 'term, df', terms-only, doc map, one postings line per term",
      [read_lines(P("disk_dict.txt")), read_lines(P("disk_terms.txt")), read_lines(P("disk_map.txt")),
       len(read_lines(P("disk_post.txt")))],
      [[f"{term}, {len(truth[term])}" for term in sorted(truth)], sorted(truth),
       [f"{d}\t{trec}" for d, trec, _ in corpus], len(truth)])
_, _ = PositionalIndex.build_and_save_streaming(iter(corpus), P("c_dict.txt"), P("c_post.txt"),
                                                cfg=Config(**RANDOM_OPT), dict_include_df=False)
ts.eq("I14 dict_include_df=False writes one bare term per line", read_lines(P("c_dict.txt")), sorted(truth))

# ------------------------------------------------------------------
ts.section("Sample collection (all documents processed with the default Config)")
fx = list(iter_processed_docs(FIXTURE_WARC))
fx_truth = brute_force_index(fx)
_fx_dir = tempfile.mkdtemp(dir=TEST_TMP)
VX = build_index_variants(fx, _fx_dir)
ts.ok("I15 in-memory, SPIMI and streaming SPIMI == brute force on real pages",
      flat(VX["memory"]) == flat(VX["spimi"]) == fx_truth
      and all(entries(VX["disk"].get_postings(term)) == fx_truth[term] for term in fx_truth))
_terms = read_lines(os.path.join(_fx_dir, "disk_terms.txt"))
ts.ok("I16 dictionary is sorted by Unicode code point: terms starting with a digit first, CJK last",
      _terms == sorted(fx_truth) and _terms[0] == "1000000" and _terms[-1] == "資",
      f"first {_terms[:3]}, last {_terms[-3:]}")
_mini = [(d.doc_id, d.trec_id, process_document(d)) for d in QUERY_DOCS]
_mini_idx = PositionalIndex.build_from_iter(iter(_mini), Config())
_bcfg = Config.baseline(case_folding=True)
_mini_base = PositionalIndex.build_from_iter(((d.doc_id, d.trec_id, process_document(d, _bcfg)) for d in QUERY_DOCS), _bcfg)
ts.eq("I17 Baseline keeps 'to' and 'retrieval' (Slide 5 terms); Enhanced drops 'to' and stems 'retriev'",
      [len(_mini_base.get_postings("to")), len(_mini_base.get_postings("retrieval")),
       len(_mini_idx.get_postings("to")), len(_mini_idx.get_postings("retriev"))], [1, 2, 0, 2])
_fx_dir2 = tempfile.mkdtemp(dir=TEST_TMP)
build_index_variants(fx, _fx_dir2)
ts.eq("I18 two streaming builds of the same input are byte-identical (deterministic output)",
      [sha256_file(os.path.join(_fx_dir2, f"disk_{k}.txt")) for k in ("dict", "post", "map", "terms")],
      [sha256_file(os.path.join(_fx_dir, f"disk_{k}.txt")) for k in ("dict", "post", "map", "terms")])
ts.eq("I19 the postings line of 'inform' in the query corpus (hand-worked)",
      format_posting_standard("inform", _mini_idx.get_postings("inform")), "inform, 2: <1, 2: <0, 2>; 2, 1: <12>>")
# postings are concatenated in arrival order, so unsorted or repeated doc_ids would give an unsorted index
_unsorted = [(3, "c", [("x", 0)]), (1, "a", [("x", 0)]), (2, "b", [("x", 0)])]
_repeated = [(1, "a", [("x", 0)]), (1, "a", [("x", 1)])]
_n_raised = 0
for _docs in (_unsorted, _repeated):
    for _build in (lambda d: PositionalIndex.build_from_iter(iter(d), Config()),
                   lambda d: PositionalIndex.build_from_iter(iter(d), Config(), use_spimi=True),
                   lambda d: PositionalIndex.build_and_save_streaming(iter(d), tmp_path("order", "d.txt"),
                                                                       tmp_path("order", "p.txt"))):
        try:
            _build(_docs)
        except ValueError:
            _n_raised += 1
ts.eq("I20 doc_ids that are not strictly increasing raise ValueError (in-memory, SPIMI, streaming)", _n_raised, 6)
ts.done()

===== 6.5 Indexer =====

--- Postings record format (Slide 5) ---
[PASS] I01 single-line record of the Slide 5 example
[PASS] I02 multi-line record (multiline_postings=True), laid out as on the slide
[PASS] I03 parse_posting_line reads both layouts back

--- Random corpus: every build method vs. brute force ---
[PASS] I04 in-memory index == brute force
[PASS] I05 SPIMI (many blocks, merged in memory) == brute force
[PASS] I06 streaming SPIMI: postings file == brute force (independent parser)
[PASS] I07 streaming SPIMI: get_postings() == brute force (trie byte offset / sorted file scan / multi-line)
[PASS] I08 streaming stats: docs, tokens, vocabulary, several blocks
[PASS] I09 identical files for block limit 25 vs one block, in-memory save_files, and use_trie=False
[PASS] I10 the trie's byte offsets point at the term's line in postings.txt
[PASS] I11 parse_posting_line round trip on every record (single-line and multi-line files)
[PASS] I12 empty documents are counted and mapped but ha

### 6.6 加分功能（B01–B19）

- **Trie**：精確查詢、前綴查詢的結果與順序、重複插入、5000 字元的 term 不會 `RecursionError`、`use_trie=False`。
- **Skip pointers**：位置與目標符合 ⌊√L⌋、L < 4 時不建、關閉時沒有、從檔案讀回時重新計算；同樣的 AND 結果但比較次數較少。
- **查詢**：AND、phrase、proximity 在六種索引上都與暴力法相同（100 組 AND、150 個隨機片語、200 組隨機 proximity）；再用 `QUERY_DOCS` 的人工推導結果檢查含停用字的片語、`pos_base=1`、查詢詞正規化（同一個字的各種變化都查到同一個 term）、不存在的詞與停用字；B19 是評分報告 M1 的 regression：原始查詢詞剛好也是罕見 stem 時，AND / proximity / phrase 的結果必須一致。

In [18]:
ts = TestSuite("6.6 Bonus features")
corpus = make_random_corpus()
truth = brute_force_index(corpus)
V = build_index_variants(corpus, tempfile.mkdtemp(dir=TEST_TMP), **RANDOM_OPT)
# per word: doc_id -> positions, for the brute-force query answers
ref = {w: {d: ps for d, _, ps in truth.get(w, [])} for w in RANDOM_WORDS}
rng = random.Random(SEED)

# ------------------------------------------------------------------
ts.section("Trie dictionary")
_trie = V["memory"].trie
ts.ok("B01 exact lookup returns the df of every term; unknown terms and bare prefixes are rejected",
      all(_trie.search(term)[0] == len(truth[term]) for term in truth)
      and _trie.search("nonexistentword") is None and _trie.search("alph") is None)
_prefixes = ["", "a", "e", "ep", "z", "q"]
ts.eq("B02 prefix lookup returns every matching term in lexicographical order",
      [[term for term, _, _ in _trie.starts_with(pre)] for pre in _prefixes],
      [sorted(term for term in truth if term.startswith(pre)) for pre in _prefixes])
_t = VocabularyTrie()
_t.insert("ab", 1)
_t.insert("abc", 2)
_t.insert("ab", 5, meta="x")
ts.eq("B03 inserting an existing term updates df / meta without growing the vocabulary",
      (_t.size, _t.search("ab"), _t.search("abc")), (2, (5, "x"), (2, None)))
_t = VocabularyTrie()
_long = "x" * 5000
_t.insert(_long, 1)
_t.insert("xy", 2)
try:
    _ok = [term for term, _, _ in _t.starts_with("x")] == [_long, "xy"]
except RecursionError:
    _ok = False
ts.ok("B04 prefix scan is iterative: a 5000-character term causes no RecursionError", _ok)
ts.ok("B05 use_trie=False keeps no trie in memory; lookups scan the sorted postings file",
      V["notrie"].trie is None and entries(V["notrie"].get_postings("alpha")) == truth["alpha"]
      and V["notrie"].get_postings("nonexistent") == [])

# ------------------------------------------------------------------
ts.section("Skip pointers")
_E = ErrBook()
for term, pl in V["memory"].postings.items():
    L, step = len(pl), math.isqrt(len(pl))
    for i, p in enumerate(pl):
        want = (i + step, pl[i + step].doc_id) if (L >= 4 and i % step == 0 and i + step < L) else None
        if p.skip != want:
            _E.add("skip", f"{term}[{i}] {p.skip} != {want}")
_short_list = compute_skip_pointers([PostingEntry(d, 1, [0]) for d in (1, 2, 3)])
ts.ok("B06 a skip every floor(sqrt(L)) entries pointing step entries ahead; none when L < 4",
      _E.ok("skip") and all(p.skip is None for p in _short_list), _E.detail("skip"))
ts.ok("B07 use_skip_pointers=False -> no skip pointers",
      all(p.skip is None for pl in V["noskip"].postings.values() for p in pl))
ts.ok("B08 skip pointers are recomputed when postings are read back from postings.txt",
      all([p.skip for p in V["disk"].get_postings(term)] == [p.skip for p in V["memory"].postings[term]]
          for term in truth))
# 'common' is in all 16 documents, 'rare' only in the first and the last
_docs16 = [(d, f"doc-{d}", [("common", 0)] + ([("rare", 1)] if d in (1, 16) else [])) for d in range(1, 17)]
_on = PositionalIndex.build_from_iter(iter(_docs16), Config(use_skip_pointers=True))
_off = PositionalIndex.build_from_iter(iter(_docs16), Config(use_skip_pointers=False))
(_r_on, _s_on), (_r_off, _s_off) = _on.intersect_with_skips("common", "rare"), _off.intersect_with_skips("common", "rare")
ts.ok("B09 AND with skip pointers: same answer, skips used, fewer comparisons",
      _r_on == _r_off == [1, 16] and _s_on["skips_used"] > 0 and _s_on["comparisons"] < _s_off["comparisons"],
      f"with skips {_s_on}, without {_s_off}", info=f"with skips {_s_on}, without {_s_off}")
_E = ErrBook()
for t1 in RANDOM_WORDS:
    for t2 in RANDOM_WORDS:
        want = sorted(set(ref[t1]) & set(ref[t2]))
        for name, ix in V.items():
            if ix.intersect_with_skips(t1, t2)[0] != want:
                _E.add("and", f"{name}: {t1} AND {t2}")
ts.ok("B10 AND == set intersection for all 100 word pairs, on every index variant", _E.ok("and"), _E.detail("and"))

# ------------------------------------------------------------------
ts.section("Phrase query and proximity search")
_positions = {d: dict((p, w) for w, p in terms) for d, _, terms in corpus}


def brute_phrase(q: List[str]) -> List[Tuple[int, List[int]]]:
    """Documents and start positions where the words of q occur at consecutive positions."""
    out = []
    for d, at in _positions.items():
        starts = sorted(p for p in at if all(at.get(p + i) == w for i, w in enumerate(q)))
        if starts:
            out.append((d, starts))
    return out


_E = ErrBook()
for _ in range(150):
    q = [rng.choice(RANDOM_WORDS) for _ in range(rng.randint(1, 3))]
    want = brute_phrase(q)
    for name, ix in V.items():
        if ix.phrase_query(" ".join(q)) != want:
            _E.add("phrase", f"{name}: {' '.join(q)}")
ts.ok("B11 phrase query == brute force (150 random 1-3 word phrases, every index variant)", _E.ok("phrase"),
      _E.detail("phrase"))
_E = ErrBook()
for _ in range(200):
    t1, t2, k = rng.choice(RANDOM_WORDS), rng.choice(RANDOM_WORDS), rng.randint(0, 6)
    want = []
    for d in sorted(set(ref[t1]) & set(ref[t2])):
        pairs = [(a, b) for a in ref[t1][d] for b in ref[t2][d] if abs(a - b) <= k]
        if pairs:
            want.append((d, pairs))
    for name, ix in V.items():
        if ix.proximity_query(t1, t2, k) != want:
            _E.add("prox", f"{name}: {t1},{t2},{k}")
ts.ok("B12 proximity search == brute force (200 random word pairs and distances, every index variant)",
      _E.ok("prox"), _E.detail("prox"))

# QUERY_DOCS, positions worked out by hand:
#   doc 1: information 0, retrieval 1, information 2, retrieval 3, system 4, builds 5, (an 6), inverted 7, index 8,
#          (to 9), search 10, engines 11, please 12, come 13, home 14
#   doc 2: messy 0, tricky 1, café 2, naïve 3, résumé 4, (with 5), usa 6, (and 7), state 8, (of 9), (the 10), art 11,
#          information 12, retrieval 13, please 14, stay 15, home 16
q_idx = PositionalIndex.build_from_iter(((d.doc_id, d.trec_id, process_document(d)) for d in QUERY_DOCS), Config())
# "Please come home": its stem 'pleas' would become 'plea' if the processed phrase terms were stemmed again
ts.eq("B13 phrase queries on text; stopwords inside a phrase keep their gaps",
      [q_idx.phrase_query("Information retrieval"), q_idx.phrase_query("state-of-the-art information"),
       q_idx.phrase_query("retrieval information retrieval"), q_idx.phrase_query("retrieval system builds"),
       q_idx.phrase_query("information system"), q_idx.phrase_query("Please come home")],
      [[(1, [0, 2]), (2, [12])], [(2, [8])], [(1, [1])], [(1, [3])], [], [(1, [12])]])
_cfg1 = Config(pos_base=1)
_idx1 = PositionalIndex.build_from_iter(((d.doc_id, d.trec_id, process_document(d, _cfg1)) for d in QUERY_DOCS), _cfg1)
ts.eq("B14 phrase query on a pos_base=1 index returns 1-based positions",
      _idx1.phrase_query("Information retrieval"), [(1, [1, 3]), (2, [13])])
ts.eq("B15 proximity search: |pos1 - pos2| <= k, both orders",
      [q_idx.proximity_query("invert", "search", 4), q_idx.proximity_query("search", "invert", 3),
       q_idx.proximity_query("invert", "search", 2), q_idx.proximity_query("please", "home", 2)],
      [[(1, [(7, 10)])], [(1, [(10, 7)])], [], [(1, [(12, 14)]), (2, [(14, 16)])]])
ts.eq("B16 raw query words are normalized with the index Config ('Inverted' -> 'invert')",
      [q_idx.proximity_query("Inverted", "Searching", 4), q_idx.intersect_with_skips("Information", "Retrieval")[0]],
      [[(1, [(7, 10)])], [1, 2]])
ts.eq("B17 every inflection of a query word reaches the same term ('Please', 'pleases', 'PLEASE' -> 'pleas')",
      [q_idx.intersect_with_skips(w, "Home")[0] for w in ("Please", "pleases", "PLEASE")], [[1, 2]] * 3)
ts.eq("B18 unknown words and stopwords give empty answers",
      [q_idx.intersect_with_skips("zzz", "home")[0], q_idx.phrase_query("zzz home"),
       q_idx.proximity_query("home", "zzz", 5), q_idx.phrase_query("the of and"), q_idx.intersect_with_skips("the", "home")[0]],
      [[], [], [], [], []])
# regression (grading report M1, ClueWeb09 doc 1218): "Informationes" is stemmed to the term "information" (df 1).
# The query word "information" must still mean "inform", as in phrase_query, not that rare term.
_shadow_docs = [Document(1, "d1", "", "", "Information retrieval systems"),
                Document(2, "d2", "", "", "Informationes retrieval"),
                Document(3, "d3", "", "", "information and more retrieval")]
_shadow = PositionalIndex.build_from_iter(((d.doc_id, d.trec_id, process_document(d)) for d in _shadow_docs), Config())
ts.eq("B19 a raw query word that is also a rare stem does not hide its real postings (AND, proximity, phrase agree)",
      [_shadow.trie.search("information") is not None,
       _shadow.intersect_with_skips("information", "retrieval")[0],
       [d for d, _ in _shadow.proximity_query("information", "retrieval", 3)],
       [d for d, _ in _shadow.phrase_query("information retrieval")]],
      [True, [1, 3], [1, 3], [1]])
ts.done()

===== 6.6 Bonus features =====

--- Trie dictionary ---
[PASS] B01 exact lookup returns the df of every term; unknown terms and bare prefixes are rejected
[PASS] B02 prefix lookup returns every matching term in lexicographical order
[PASS] B03 inserting an existing term updates df / meta without growing the vocabulary
[PASS] B04 prefix scan is iterative: a 5000-character term causes no RecursionError
[PASS] B05 use_trie=False keeps no trie in memory; lookups scan the sorted postings file

--- Skip pointers ---
[PASS] B06 a skip every floor(sqrt(L)) entries pointing step entries ahead; none when L < 4
[PASS] B07 use_skip_pointers=False -> no skip pointers
[PASS] B08 skip pointers are recomputed when postings are read back from postings.txt
[PASS] B09 AND with skip pointers: same answer, skips used, fewer comparisons  -> with skips {'comparisons': 11, 'skips_used': 2}, without {'comparisons': 16, 'skips_used': 0}
[PASS] B10 AND == set intersection for all 100 word pairs, on every index v

### 6.7 命令列開關 `run_cli`（C01–C22）

5.3 表中的每個參數都在測試 WARC 上跑一次，輸出寫到暫存目錄，檢查該參數造成的效果；並確認預設參數的輸出與直接呼叫 Python API 逐 byte 相同、只影響記憶體結構的參數（`--no-trie`、`--no-skip-pointers`、`--block-limit`）不會改變輸出檔；`.txt` 輸入與 `--txt-input`（C22）；`--baseline --lemmatize` 會套用 lemmatization（C21）；錯誤參數會被拒絕。

In [19]:
ts = TestSuite("6.7 Command line (run_cli)")
OUT_FILES = ("dictionary.txt", "dictionary_terms_only.txt", "postings.txt", "doc_map.txt")


def cli(*args: str, input_path: str = FIXTURE_WARC) -> Dict[str, Any]:
    """run_cli on input_path into a fresh temporary directory; returns the stats and the parsed output files."""
    out_dir = tempfile.mkdtemp(dir=TEST_TMP)
    stats = run_cli(["--input", input_path, "--output-dir", out_dir, *args])
    f = lambda name: os.path.join(out_dir, name)
    return dict(stats=stats, terms=read_lines(f("dictionary_terms_only.txt")), dict=read_lines(f("dictionary.txt")),
                post=parse_post_file(f("postings.txt")), raw=Path(f("postings.txt")).read_text(encoding="utf-8"),
                doc_map=read_lines(f("doc_map.txt")), sha={n: sha256_file(f(n)) for n in OUT_FILES})


ts.section("Default run and the linguistic switches")
R = cli()
ts.ok("C01 default run: 4 output files, 5 documents, 'term, df' dictionary aligned with the terms-only file",
      R["stats"]["num_docs"] == 5 and len(R["doc_map"]) == 5
      and [line.rsplit(", ", 1)[0] for line in R["dict"]] == R["terms"], _short(R["stats"]))
ts.ok("C02 the default is the Enhanced index: lower case, stopwords removed, Porter stems",
      "inform" in R["terms"] and "information" not in R["terms"] and not {"is", "the", "of"} & set(R["terms"])
      and all(term == term.lower() for term in R["terms"]))
_api_dir = tempfile.mkdtemp(dir=TEST_TMP)
PositionalIndex.build_and_save_streaming(
    iter_processed_docs(FIXTURE_WARC, Config()),
    *(os.path.join(_api_dir, n) for n in ("dictionary.txt", "postings.txt", "doc_map.txt", "dictionary_terms_only.txt")),
    cfg=Config())
ts.eq("C03 default run == the Python API with Config() (byte-identical files)",
      {n: sha256_file(os.path.join(_api_dir, n)) for n in OUT_FILES}, R["sha"])
_terms = set(cli("--baseline")["terms"])
ts.ok("C04 --baseline keeps case, stopwords and unstemmed words",
      {"The", "is", "Information", "Retrieval", "retrieval"} <= _terms, _short(sorted(_terms)))
_terms = set(cli("--no-case-folding")["terms"])
ts.ok("C05 --no-case-folding (stemming still on) keeps capitalized stems", {"Inform", "Retriev", "Messi"} <= _terms,
      _short(sorted(_terms)))
_terms = set(cli("--no-stopwords")["terms"])
ts.ok("C06 --no-stopwords keeps stopwords", {"is", "the", "of"} <= _terms)
_no_stem = set(cli("--no-stem")["terms"])
ts.ok("C07 --no-stem keeps whole words", "information" in _no_stem and "inform" not in _no_stem)
if HAVE_WORDNET:
    _terms = set(cli("--lemmatize", "--no-stem")["terms"])
    ts.ok("C08 --lemmatize uses WordNet lemmas",
          {"collection", "system"} <= _terms and not {"collections", "systems"} & _terms)
    _terms = set(cli("--baseline", "--lemmatize")["terms"])
    ts.ok("C21 --baseline --lemmatize: lemmas on top of the Baseline (case and stopwords kept)",
          # 'is' -> 'be' (WordNet verb lemma) shows the stopword was kept and lemmatized
          {"collection", "system", "Information", "The", "be"} <= _terms and not {"collections", "systems", "is"} & _terms,
          _short(sorted(_terms)))
else:
    ts.skip("C08 --lemmatize uses WordNet lemmas", "NLTK WordNet is not installed")
    ts.skip("C21 --baseline --lemmatize: lemmas on top of the Baseline", "NLTK WordNet is not installed")
_terms = set(cli("--no-title", "--no-stem")["terms"])
ts.ok("C09 --no-title indexes the body only (title-only words 'messy', 'tricky' disappear)",
      {"messy", "tricky"} <= _no_stem and not {"messy", "tricky"} & _terms and "information" in _terms)

ts.section("Tokenizer switches and output formats")
_join, _keep = set(cli("--baseline", "--hyphen-mode", "join")["terms"]), set(cli("--baseline", "--hyphen-mode", "keep")["terms"])
ts.ok("C10 --hyphen-mode join / keep",
      {"stateoftheart", "email"} <= _join and "state" not in _join and {"state-of-the-art", "e-mail", "555-1234"} <= _keep)
_mins = defaultdict(lambda: 1 << 30)
for _pl in cli("--baseline", "--pos-base", "1")["post"].values():
    for d, _, ps in _pl:
        _mins[d] = min(_mins[d], ps[0])
ts.eq("C11 --pos-base 1: every non-empty document starts at position 1", dict(_mins), {1: 1, 2: 1, 3: 1, 4: 1})
_R = cli("--dict-terms-only")
ts.ok("C12 --dict-terms-only: dictionary.txt has one bare term per line", _R["dict"] == _R["terms"] == R["terms"])
_R = cli("--multiline-postings")
ts.ok("C13 --multiline-postings: slide layout, same postings", ":\n<" in _R["raw"] and _R["post"] == R["post"])
ts.eq("C14 --no-trie / --no-skip-pointers change only in-memory structures, not the files",
      [cli("--no-trie")["sha"], cli("--no-skip-pointers")["sha"]], [R["sha"], R["sha"]])
_R = cli("--block-limit", "3")
ts.ok("C15 --block-limit 3: several SPIMI blocks, identical files", _R["stats"]["num_blocks"] > 1 and _R["sha"] == R["sha"],
      _short(_R["stats"]))
_R = cli("--max-docs", "2")
ts.ok("C16 --max-docs 2 indexes only the first two documents",
      _R["stats"]["num_docs"] == 2 and _R["doc_map"] == R["doc_map"][:2])

ts.section(".txt inputs and errors")
_tok_txt = tmp_path("cli", "tokenizer_sample.txt")
with open(_tok_txt, "w", encoding="utf-8", newline="\n") as f:
    f.write("1\tclueweb09-en0000-00-00000\tThe:0 Running:1 Dogs:2\n")
    f.write("2\tclueweb09-en0000-00-00001\tInformation:0 Retrieval:1\n")
_R = cli("--max-docs", "1", "--pos-base", "1", input_path=_tok_txt)
# 'The'(0 -> 1) is removed, 'Running'(1 -> 2) -> 'run', 'Dogs'(2 -> 3) -> 'dog'
ts.ok("C17 tokenizer .txt input: --max-docs, --pos-base and the linguistic module are applied",
      _R["stats"]["num_docs"] == 1 and _R["terms"] == ["dog", "run"] and _R["post"]["dog"] == [(1, 1, [3])],
      _short((_R["stats"], _R["post"])))
_plain_txt = tmp_path("cli", "plain_docs.txt")
with open(_plain_txt, "w", encoding="utf-8", newline="\n") as f:
    f.write("Information retrieval system\nSearch engine index\n")
_R = cli("--no-trie", "--multiline-postings", input_path=_plain_txt)
ts.ok("C18 plain-text .txt input: one document per line",
      _R["stats"]["num_docs"] == 2 and _R["terms"] == ["engin", "index", "inform", "retriev", "search", "system"])
# the name contains 'tok', so 'auto' would run the linguistic module again ('pleas' -> 'plea', 'other' dropped)
_terms_txt = tmp_path("cli", "stokes_ling.txt")
with open(_terms_txt, "w", encoding="utf-8", newline="\n") as f:
    f.write("1\tdoc-1\tpleas:0 other:1\n")
ts.eq("C22 --txt-input terms / tokens overrides the guess from the file name",
      [cli(input_path=_terms_txt)["terms"], cli("--txt-input", "terms", input_path=_terms_txt)["terms"],
       cli("--txt-input", "tokens", input_path=_terms_txt)["terms"]],
      [["plea"], ["other", "pleas"], ["plea"]])
ts.raises("C19 missing --input raises ValueError", ValueError, run_cli, [])
with contextlib.redirect_stderr(io.StringIO()):
    ts.raises("C20 an invalid --hyphen-mode is rejected by argparse", SystemExit, run_cli,
              ["--input", FIXTURE_WARC, "--hyphen-mode", "nope"])
ts.done()

===== 6.7 Command line (run_cli) =====

--- Default run and the linguistic switches ---
[PASS] C01 default run: 4 output files, 5 documents, 'term, df' dictionary aligned with the terms-only file
[PASS] C02 the default is the Enhanced index: lower case, stopwords removed, Porter stems
[PASS] C03 default run == the Python API with Config() (byte-identical files)
[PASS] C04 --baseline keeps case, stopwords and unstemmed words
[PASS] C05 --no-case-folding (stemming still on) keeps capitalized stems
[PASS] C06 --no-stopwords keeps stopwords
[PASS] C07 --no-stem keeps whole words
[PASS] C08 --lemmatize uses WordNet lemmas
[PASS] C21 --baseline --lemmatize: lemmas on top of the Baseline (case and stopwords kept)
[PASS] C09 --no-title indexes the body only (title-only words 'messy', 'tricky' disappear)

--- Tokenizer switches and output formats ---
[PASS] C10 --hyphen-mode join / keep
[PASS] C11 --pos-base 1: every non-empty document starts at position 1
[PASS] C12 --dict-terms-only: dictiona

## 7. 處理 ClueWeb09 並建立索引

| 小節 | 內容 | 時間 |
|---|---|---|
| 7.1 | 小型範例：用 5 篇的 `test.warc.gz` 建索引，看輸出檔的實際內容與查詢結果 | 數秒 |
| 7.2 | 檢視單一 ClueWeb09 文件每個 token 的處理過程 | 數秒 |
| 7.3 | 處理完整的 `07.warc.gz`，存下 tokenizer 與 linguistic module 的中間檔 | 約 3 分鐘 |
| 7.4 | 由中間檔建立 Enhanced 與 Baseline 兩套索引 | 約 3 分鐘 |

路徑與設定都取自第 0 節。

### 7.1 小型範例：`sample_index/`

用 `SAMPLE_PATH`（`test.warc.gz`，5 篇文件）以串流 SPIMI 建立索引，輸出到 `sample_index/`。`spimi_block_limit=5` 很小，所以這 5 篇也會分成好幾個區塊再合併。若 `test.warc.gz` 不存在，會先用第 6 節的測試資料產生（內容相同）。

**輸出**：索引統計、四個輸出檔各自的前 6 行，以及幾個查詢的結果（前綴查詢、AND、phrase query、proximity search）。

In [20]:
# Index the 5-document sample file with streaming SPIMI and show what the output files look like.
# SAMPLE_PATH is created from the Section 6 test data if it does not exist yet.
if not os.path.exists(SAMPLE_PATH):
    write_sample_warc(SAMPLE_PATH)
    print(f"created {SAMPLE_PATH}")

sample_idx, sample_stats = PositionalIndex.build_and_save_streaming(
    iter_processed_docs(SAMPLE_PATH, CFG),
    dict_path=DICT_OUT,
    postings_path=POSTINGS_OUT,
    doc_map_path=DOC_MAP_OUT,
    dict_terms_only_path=DICT_TERMS_OUT,
    # a tiny block limit, so even 5 documents go through several SPIMI blocks and the K-way merge
    cfg=Config(spimi_block_limit=5),
)
print("Sample index:", {k: v for k, v in sample_stats.items() if not k.endswith("_path")})
for label, path in [("doc_map.txt ('docID<TAB>TREC-ID')", DOC_MAP_OUT),
                    ("dictionary.txt ('term, df')", DICT_OUT),
                    ("dictionary_terms_only.txt ('term')", DICT_TERMS_OUT),
                    ("postings.txt (one record per line)", POSTINGS_OUT)]:
    print(f"\n--- {label}, first 6 lines ---")
    with open(path, encoding="utf-8") as f:
        for _ in range(6):
            print(f.readline().rstrip("\n"))

# the query text goes through the same tokenizer and linguistic module as the documents
print("\n--- queries on the sample index ---")
print("terms starting with 'in'      :", [(term, df) for term, df, _ in sample_idx.trie.starts_with("in")])
print("information AND retrieval     :", sample_idx.intersect_with_skips("information", "retrieval")[0])
print("phrase 'Information Retrieval':", sample_idx.phrase_query("Information Retrieval"))
print("'inverted' within 1 of 'index':", sample_idx.proximity_query("inverted", "index", 1))

Sample index: {'num_docs': 5, 'total_tokens': 66, 'vocab_size': 56, 'num_blocks': 4}

--- doc_map.txt ('docID<TAB>TREC-ID'), first 6 lines ---
1	clueweb09-en0000-00-00000
2	clueweb09-en0000-00-00001
3	clueweb09-zh0000-00-00000
4	clueweb09-en0000-00-00002
5	clueweb09-en0000-00-00003


--- dictionary.txt ('term, df'), first 6 lines ---
1000000, 1
1234, 1
192.168.0.1, 1
2.0.1, 1
2009, 1
3.14, 1

--- dictionary_terms_only.txt ('term'), first 6 lines ---
1000000
1234
192.168.0.1
2.0.1
2009
3.14

--- postings.txt (one record per line), first 6 lines ---
1000000, 1: <2, 1: <5>>
1234, 1: <2, 1: <31>>
192.168.0.1, 1: <2, 1: <28>>
2.0.1, 1: <2, 1: <22>>
2009, 1: <2, 1: <15>>
3.14, 1: <2, 1: <25>>

--- queries on the sample index ---
terms starting with 'in'      : [('index', 1), ('inform', 1), ('invert', 1)]
information AND retrieval     : [1]
phrase 'Information Retrieval': [(1, [0, 2, 4])]
'inverted' within 1 of 'index': [(3, [(14, 15)])]


### 7.2 檢視單一文件的處理過程

把第 `PICK` 篇文件（預設第 1 篇）的每個 token 處理過程印出來，方便人工檢查規則是否正確。改 `PICK` 就能看別篇。

**輸出分三段**

1. **Tokenizer 表**：每一次 regex 比對一行，欄位是「原始字串、使用的規則、輸出的 token、分到的 position」，最後印出 token 總數。
2. **Linguistic module 的變化**：每個比對一行，格式為

   ```text
   (原始字串, 規則, tokenizer 輸出, positions)  [實際生效的步驟]  (原始字串, 規則, 最終 terms, 保留的 positions)
   ```

   例如 `(The, WORD, ['The'], [0])  [case_folding, stopword_removal]  (The, WORD, [], [])` 表示 `The` 先轉小寫、再被當成停用字移除。方括號內只列出真的改變了這個 token 的步驟。最後兩行是各步驟之後的 token 數（`#tok`）與不重複 term 數（`#uniq`）。
3. **`processed:`**：`process_document` 交給 indexer 的最終 `[(term, pos), ...]`，與 `linguistic_output.txt` 中該篇的內容相同。

In [21]:
def show_tokenize(text: str, cfg: Config = Config()):
    """Print one row per regex match: (raw match, rule, tokens, positions); return (tokens, rows)."""
    tokens, rows = tokenize(text, cfg, trace=True)
    print(f"{'raw match':<22}{'rule':<10}{'tokens':<36}positions")
    print("-" * 80)
    for raw, kind, outs, poss in rows:
        print(f"{raw:<22}{kind:<10}{str(outs):<36}{poss}")
    print(f"\n=> {len(tokens)} tokens")
    return tokens, rows


def show_linguistic(tokens, rows, cfg: Config = Config()):
    """Print (before) [steps that changed it] (after) for every tokenizer match; return the final terms."""
    terms, stages = linguistic(tokens, cfg, trace=True)
    # stage names in execution order, e.g. ["input", "case_folding", "stopword_removal", "stemming"]
    names = list(stages)
    # stage -> {position: term}
    by_pos = {n: dict((p, w) for w, p in stages[n]) for n in names}
    final = by_pos[names[-1]]
    for raw, kind, outs, poss in rows:
        # a step "applied" if it removed or changed at least one token of this match
        applied = []
        for prev, cur in zip(names, names[1:]):
            if any(p in by_pos[prev] and by_pos[cur].get(p) != by_pos[prev][p] for p in poss):
                applied.append(cur)
        # positions (and their terms) that survived every step
        new_poss = [p for p in poss if p in final]
        new_outs = [final[p] for p in new_poss]
        print(f"({raw}, {kind}, {outs}, {poss})  [{', '.join(applied)}]  ({raw}, {kind}, {new_outs}, {new_poss})")
    # token count and distinct-term count after each step
    print("\n=> #tok : " + " -> ".join(str(len(stages[n])) for n in names))
    print("=> #uniq: " + " -> ".join(str(len({w for w, _ in stages[n]})) for n in names)
          + f"   ({' -> '.join(names)})")
    return terms


# doc_id of the document to inspect
PICK = 1
doc = next(d for d in parse_warc(WARC_PATH) if d.doc_id == PICK)
# same text selection as the pipeline, so the result matches the intermediate files
text = f"{doc.title} {doc.body}" if CFG.include_title else doc.body
tokens, rows = show_tokenize(text, CFG)
print()
_ = show_linguistic(tokens, rows, CFG)
print()
# exactly what iter_processed_docs hands to the indexer
print("processed:", process_document(doc, CFG))

raw match             rule      tokens                              positions
--------------------------------------------------------------------------------
Access                WORD      ['Access']                          [0]
Denied                WORD      ['Denied']                          [1]
Accesso               WORD      ['Accesso']                         [2]
Negato                WORD      ['Negato']                          [3]
Siamo                 WORD      ['Siamo']                           [4]
spiacenti             WORD      ['spiacenti']                       [5]
Il                    WORD      ['Il']                              [6]
software              WORD      ['software']                        [7]
utilizzato            WORD      ['utilizzato']                      [8]
per                   WORD      ['per']                             [9]
accedere              WORD      ['accedere']                        [10]
a                     WORD      ['a']           

### 7.3 處理完整的 ClueWeb09（`07.warc.gz`）

執行前請確認第 0 節的 `WARC_PATH` 指向 `07.warc.gz`。`MAX_DOCS = None` 會處理整個檔案（31,480 篇，最後一篇是 `clueweb09-en0000-07-31479`）；設成小數字可以快速試跑。

逐篇串流處理（不會把整個檔案讀進記憶體），每篇文件同時寫出兩個中間檔，格式見第 4 節：

| 檔案 | 內容 | 用途 |
|---|---|---|
| `tokenizer_output.txt` | tokenizer 的輸出（未經語言處理） | 建立 Baseline index |
| `linguistic_output.txt` | linguistic module 的輸出 | 建立 Enhanced index |

7.4 直接讀這兩個檔建索引，不必重新解析 WARC。

**輸出**

- 第一行：`docs`（文件數）、`empty_docs`（沒有任何 term 的文件數）、`terms`（term 總數，等於 postings 中 position 的總數）、`vocab`（不重複的 term 數，等於 dictionary 大小）、耗時與每秒處理篇數。
- 第二行：出現次數最多的 20 個 term。正常應看到數字、年份、`com`、`new`、`use` 這類常見字；若出現大量 HTML 或 JavaScript 字詞（`div`、`var`、`function`），代表 HTML 清理有問題。

In [22]:
# Uses WARC_PATH, TOKEN_OUT, LING_OUT, MAX_DOCS and CFG from Section 0, and _fmt from Section 4
assert os.path.exists(WARC_PATH), f"cannot find {WARC_PATH}"


t0 = time.time()
n_docs = 0
# total (term, pos) pairs = total positions in the index
n_terms = 0
# documents that produced no term at all
empty_docs = 0
# term -> total occurrences in the collection
vocab = Counter()
# LF line endings on every OS; one line per document in each file
with open(TOKEN_OUT, "w", encoding="utf-8", newline="\n") as f_tok, \
     open(LING_OUT, "w", encoding="utf-8", newline="\n") as f_ling:
    for doc in parse_warc(WARC_PATH):
        if MAX_DOCS is not None and doc.doc_id > MAX_DOCS:
            break
        # same text selection as process_document()
        text = f"{doc.title} {doc.body}" if CFG.include_title else doc.body
        tokens = tokenize(text, CFG)
        f_tok.write(f"{doc.doc_id}\t{doc.trec_id}\t{_fmt((t.text, t.pos) for t in tokens)}\n")
        terms = linguistic(tokens, CFG)
        f_ling.write(f"{doc.doc_id}\t{doc.trec_id}\t{_fmt(terms)}\n")
        n_docs += 1
        n_terms += len(terms)
        empty_docs += not terms
        vocab.update(w for w, _ in terms)
dt = time.time() - t0
speed = n_docs / max(dt, 1e-9)
print(f"docs={n_docs}  empty_docs={empty_docs}  terms={n_terms}  vocab={len(vocab)}  "
      f"time={dt:.1f}s ({speed:.0f} docs/s)")
print("top 20 terms:", vocab.most_common(20))
print("wrote", TOKEN_OUT, "and", LING_OUT)

docs=31480  empty_docs=0  terms=17803162  vocab=345701  time=232.3s (135 docs/s)
top 20 terms: [('1', 91142), ('2008', 83862), ('new', 66608), ('2', 57946), ('use', 55020), ('com', 54559), ('review', 54395), ('us', 51273), ('3', 46991), ('home', 45477), ('4', 45025), ('one', 43127), ('post', 43095), ('blog', 42529), ('car', 42092), ('comment', 40713), ('5', 40140), ('page', 39380), ('time', 38282), ('site', 38039)]
wrote e:\ir_hw\claude_hw1\tokenizer_output.txt and e:\ir_hw\claude_hw1\linguistic_output.txt


### 7.4 建立 ClueWeb09 的完整索引

使用第 0 節的 `CLUEWEB_*` 與 `BASELINE_*` 路徑，以串流 SPIMI（`build_and_save_streaming`）建立兩套索引：

| 索引 | 輸入 | 輸出 |
|---|---|---|
| **Enhanced**（主要輸出） | `linguistic_output.txt`；若不存在，直接從 `07.warc.gz` 用 `CFG` 處理 | `dictionary.txt`、`dictionary_terms_only.txt`、`postings.txt`、`doc_map.txt` |
| **Baseline** | `tokenizer_output.txt`（存在時才建立） | `baseline_dictionary.txt`、`baseline_dictionary_terms_only.txt`、`baseline_postings.txt`（docID 與 Enhanced 相同，共用 `doc_map.txt`） |

讀取中間檔可以省下重新解析 WARC 的時間，**所以 7.3 的程式或設定有修改時，必須先重新執行 7.3**，否則這裡會用到舊的中間檔。`USE_SPIMI = False` 時改用記憶體內建索引再存檔，輸出檔案相同。

**輸出**：兩套索引各印一行耗時、文件數、token 總數與字典大小。

In [23]:
# Build the Enhanced and Baseline ClueWeb09 indexes with streaming SPIMI (paths from Section 0)
if os.path.exists(CLUEWEB_LING_PATH) or os.path.exists(CLUEWEB_WARC_PATH):
    t_start = time.time()
    # prefer the Section 5.2 output (already processed with CFG) over re-parsing the WARC file
    if os.path.exists(CLUEWEB_LING_PATH):
        print(f"[Enhanced Index] Streaming from pre-processed {CLUEWEB_LING_PATH} ...")
        stream = iter_processed_file(CLUEWEB_LING_PATH)
    else:
        print(f"[Enhanced Index] Streaming directly from {CLUEWEB_WARC_PATH} ...")
        stream = iter_processed_docs(CLUEWEB_WARC_PATH, CFG, max_docs=MAX_DOCS)

    if USE_SPIMI:
        clueweb_idx, cw_stats = PositionalIndex.build_and_save_streaming(
            stream,
            dict_path=CLUEWEB_DICT_OUT,
            postings_path=CLUEWEB_POST_OUT,
            doc_map_path=CLUEWEB_MAP_OUT,
            dict_terms_only_path=CLUEWEB_DICT_TERMS_OUT,
            cfg=CFG,
        )
    else:
        clueweb_idx = PositionalIndex.build_from_iter(stream, CFG, use_spimi=False)
        cw_stats = clueweb_idx.save_files(
            dict_path=CLUEWEB_DICT_OUT,
            postings_path=CLUEWEB_POST_OUT,
            doc_map_path=CLUEWEB_MAP_OUT,
            dict_terms_only_path=CLUEWEB_DICT_TERMS_OUT,
        )
    dt_enh = time.time() - t_start
    print(f"=> Enhanced Index done in {dt_enh:.1f}s | docs={cw_stats['num_docs']:,} | "
          f"tokens={cw_stats['total_tokens']:,} | vocab={cw_stats['vocab_size']:,}")

    # Baseline index from the raw tokenizer output (no linguistic processing), if that file exists
    if os.path.exists(CLUEWEB_TOK_PATH):
        t_base = time.time()
        print(f"[Baseline Index] Streaming from raw tokens in {CLUEWEB_TOK_PATH} ...")
        base_idx, base_stats = PositionalIndex.build_and_save_streaming(
            iter_processed_file(CLUEWEB_TOK_PATH),
            dict_path=BASELINE_DICT_OUT,
            postings_path=BASELINE_POST_OUT,
            doc_map_path=None,
            dict_terms_only_path=BASELINE_DICT_TERMS_OUT,
            cfg=Config.baseline(),
        )
        print(f"=> Baseline Index done in {time.time() - t_base:.1f}s | "
              f"tokens={base_stats['total_tokens']:,} | vocab={base_stats['vocab_size']:,}")
else:
    print(f"ClueWeb09 files ({CLUEWEB_LING_PATH} or {CLUEWEB_WARC_PATH}) not found in {BASE_DIR}; "
          "place them in BASE_DIR to run the full 31,480-doc indexing cell.")

[Enhanced Index] Streaming from pre-processed e:\ir_hw\claude_hw1\linguistic_output.txt ...
=> Enhanced Index done in 79.8s | docs=31,480 | tokens=17,803,162 | vocab=345,701
[Baseline Index] Streaming from raw tokens in e:\ir_hw\claude_hw1\tokenizer_output.txt ...
=> Baseline Index done in 103.0s | tokens=25,090,585 | vocab=500,394


## 8. 輸出驗證

檢查 7.4 產生的 ClueWeb09 索引檔；若它們不存在，改為檢查 7.1 的 `sample_index/`。這一節只讀取輸出檔與原始語料，不會修改任何檔案（暫存檔放在系統暫存目錄）。請在 7.4 之後執行。

| 小節 | 檢查內容 | 編號 | 時間 |
|---|---|---|---|
| 8.1 | 輸出檔格式與內部一致性，Enhanced 與 Baseline 的差異 | O01–O36 | 約 1.5 分鐘 |
| 8.2 | 重新解析原始 WARC，逐一比對每個 (term, docID, position) | G01–G11 | 約 4 分鐘 |
| 8.3 | 語料前 300 篇：決定性、區塊大小、三種建法一致、記憶體用量 | D01–D05 | 約 30 秒 |
| 8.4 | 7.4 留在記憶體中的索引物件：統計、trie、AND、phrase query | M01–M06 | 約 1 分鐘 |
| 8.5 | 隨機 500 篇文件，與 Python 內建 `html.parser` 寫的獨立抽取器比對：漏字、黏字、拆字、entity 外洩 | X01–X05 | 約 10 秒 |
| 8.6 | 所有測試的統計，以及「作業要求 → 檢查」對照表的執行結果 | — | — |

### 8.1 輸出檔格式與一致性

Enhanced 與 Baseline 兩套索引各自檢查（檢查名稱前面標 `[enhanced]` / `[baseline]`），每個檔案都完整讀過一次，不是抽查：

| 編號 | 檢查內容 |
|---|---|
| O01–O03 | 檔案存在且非空、沒有 BOM、整個檔案都沒有 `\r`（LF 換行） |
| O04–O12 | Dictionary：UTF-8、每行 `term` 或 `term, df`、df ≥ 1、不重複、嚴格排序、沒有控制字元、`U+FFFD` 很少；`dictionary_terms_only.txt` 與它完全一致，且每行只有一個 term（投影片 p.5 "a sorted list of vocabularies"） |
| O13–O18 | Doc map：每行 `docID<TAB>TREC-ID`、docID 從 1 到 N 連續、TREC-ID 不重複且格式正確；處理完整 `07.warc.gz` 時，文件數必須是 31,480、最後一篇是 `clueweb09-en0000-07-31479` |
| O19–O29 | Postings：每筆符合 `term, df: <doc, tf: <pos, ...>; ...>`、與 dictionary 逐行對齊、df 與 dictionary 相同且等於文件數、docID 遞增且存在於 doc map、tf 等於 position 數、position 嚴格遞增且 ≥ `pos_base` |
| O30–O36 | Enhanced 全為小寫；Baseline 保留停用字與 `retrieval`、每篇從 `pos_base` 開始；每篇文件 Enhanced 的 position 數與最後位置都不超過 Baseline |

開啟 stemming 時，字典中有些 stem 剛好和停用字同形（例如 `other`、`own`），這是刻意保留的（見第 3 節），只列為 INFO；8.2 的 G09 會確認它們都不是來自原文的停用字。

In [24]:
ts = TestSuite("8.1 Output files")

# ---------- what is checked ----------
# the Baseline index of 7.4: same tokenizer settings as CFG, no linguistic processing
BASE_CFG = Config.baseline(hyphen_mode=CFG.hyphen_mode, pos_base=CFG.pos_base, include_title=CFG.include_title)
# known facts about 07.warc.gz (checked only when the whole file was processed)
_full_07 = os.path.basename(WARC_PATH) == "07.warc.gz" and MAX_DOCS is None
EXPECTED_DOCS = 31480 if _full_07 else None
EXPECTED_LAST_TREC = "clueweb09-en0000-07-31479" if _full_07 else None


def _have(*paths) -> bool:
    return all(p and os.path.isfile(p) for p in paths)


SETS = []
if _have(CLUEWEB_DICT_OUT, CLUEWEB_POST_OUT, CLUEWEB_MAP_OUT):
    SETS.append(dict(name="enhanced", kind="enhanced", cfg=CFG, dpath=CLUEWEB_DICT_OUT, tpath=CLUEWEB_DICT_TERMS_OUT,
                     ppath=CLUEWEB_POST_OUT, mpath=CLUEWEB_MAP_OUT))
if _have(BASELINE_DICT_OUT, BASELINE_POST_OUT):
    # the Baseline index shares doc_map.txt with the Enhanced index
    SETS.append(dict(name="baseline", kind="baseline", cfg=BASE_CFG, dpath=BASELINE_DICT_OUT,
                     tpath=BASELINE_DICT_TERMS_OUT, ppath=BASELINE_POST_OUT,
                     mpath=CLUEWEB_MAP_OUT if _have(CLUEWEB_MAP_OUT) else None))
if SETS:
    CORPUS = WARC_PATH
else:
    # no ClueWeb09 index yet: check the sample index of 7.1 instead
    SETS.append(dict(name="sample", kind="enhanced", cfg=CFG, dpath=DICT_OUT, tpath=DICT_TERMS_OUT,
                     ppath=POSTINGS_OUT, mpath=DOC_MAP_OUT))
    CORPUS = SAMPLE_PATH
    EXPECTED_DOCS = EXPECTED_LAST_TREC = None
HAVE_CORPUS = os.path.isfile(CORPUS)
ts.note(f"index sets: {[S['name'] for S in SETS]} | corpus: {CORPUS} ({'found' if HAVE_CORPUS else 'MISSING'})")
ts.note(f"CFG: include_title={CFG.include_title}, pos_base={CFG.pos_base}, hyphen_mode={CFG.hyphen_mode}, "
        f"case_folding={CFG.case_folding}, stopwords={CFG.stopword_removal}, "
        f"lemmatization={CFG.lemmatization}, stemming={CFG.stemming}")


def chk(S, name, ok, detail="", level="FAIL", info=""):
    """ts.ok() with the index-set name as prefix."""
    return ts.ok(f"[{S['name']}] {name}", ok, detail, level, info)


def _has_cr(path: str) -> bool:
    """True if the file contains a '\\r' anywhere (whole file, read in 1 MB chunks)."""
    with open(path, "rb") as f:
        return any(b"\r" in chunk for chunk in iter(lambda: f.read(1 << 20), b""))


def load_set(S) -> Dict[str, Any]:
    """Check one index set's files; returns what Sections 8.2 and 8.4 need (terms, df, doc map, fingerprints, ...).

    fp[term] = (df, total tf, sum of hash((doc_id, pos))) is an order-independent fingerprint of the postings."""
    R = dict(ok=True, dterms=[], ddf={}, has_df=False, doc_map={}, fp={}, offs={}, ntok=Counter(),
             maxpos={}, minpos={}, n_rec=0)
    ts.section(f"[{S['name']}] files")
    files = [("dictionary", S["dpath"]), ("dictionary (terms only)", S["tpath"]), ("postings", S["ppath"])]
    if S["mpath"]:
        files.append(("doc map", S["mpath"]))
    for label, p in files:
        ok = os.path.isfile(p) and os.path.getsize(p) > 0
        chk(S, f"O01 {label} exists and is not empty", ok, f"missing or empty: {p}",
            info=f"{os.path.getsize(p):,} bytes" if os.path.isfile(p) else "")
        R["ok"] &= ok
    if not R["ok"]:
        return R
    boms = [p for _, p in files if open(p, "rb").read(3) == b"\xef\xbb\xbf"]
    chk(S, "O02 no UTF-8 BOM", not boms, str(boms))
    crlf = [p for _, p in files if _has_cr(p)]
    chk(S, "O03 LF line endings only (no '\\r' anywhere)", not crlf, str(crlf))

    # ---- dictionary ----
    ts.section(f"[{S['name']}] dictionary")
    E, lines_with_df = ErrBook(), 0
    try:
        with open(S["dpath"], encoding="utf-8") as f:
            for ln, line in enumerate(f, 1):
                line = line.rstrip("\n")
                if not line:
                    E.add("blank", f"line {ln}")
                    continue
                m = re.fullmatch(r"(\S+), (\d+)", line)
                if m:
                    term, df = m.group(1), int(m.group(2))
                    lines_with_df += 1
                    if df < 1:
                        E.add("df", f"line {ln}")
                    R["ddf"][term] = df
                elif re.search(r"\s", line):
                    E.add("fmt", f"line {ln}: {line!r}")
                    continue
                else:
                    term = line
                R["dterms"].append(term)
        chk(S, "O04 dictionary is valid UTF-8", True)
    except UnicodeDecodeError as e:
        chk(S, "O04 dictionary is valid UTF-8", False, str(e))
        R["ok"] = False
        return R
    dt = R["dterms"]
    R["has_df"] = lines_with_df == len(dt) and len(dt) > 0
    chk(S, "O05 no blank lines; each line is `term` or `term, df`", E.ok("blank") and E.ok("fmt"),
        E.detail("blank") or E.detail("fmt"))
    chk(S, "O06 df values are positive", E.ok("df"), E.detail("df"))
    chk(S, "O07 no duplicate terms", len(set(dt)) == len(dt), f"{len(dt) - len(set(dt))} duplicates")
    bad = [(a, b) for a, b in zip(dt, dt[1:]) if not a < b]
    chk(S, "O08 terms strictly sorted", not bad, f"first violations: {bad[:3]}")
    chk(S, "O09 no control characters in terms", not any(re.search(r"[\x00-\x1f\x7f]", x) for x in dt))
    fffd = sum("�" in x for x in dt)
    chk(S, "O10 few terms with U+FFFD (mis-decoded pages)", fffd <= max(5, 0.001 * len(dt)), f"{fffd} terms", "WARN",
        info=f"{fffd} terms")
    ts.note(f"[{S['name']}] vocabulary size = {len(dt):,}; dictionary.txt has a df column = {R['has_df']}")
    try:
        tl = read_lines(S["tpath"])
        chk(S, "O11 terms-only dictionary == dictionary terms, same order", tl == dt, f"{len(tl)} vs {len(dt)} lines")
        chk(S, "O12 terms-only dictionary is one bare term per line (Slide 5: a sorted list of vocabularies)",
            not any(re.search(r"\s", x) for x in tl))
    except UnicodeDecodeError as e:
        chk(S, "O11 terms-only dictionary is valid UTF-8", False, str(e))

    # ---- doc map ----
    if S["mpath"]:
        ts.section(f"[{S['name']}] doc map")
        E = ErrBook()
        with open(S["mpath"], encoding="utf-8") as f:
            for ln, line in enumerate(f, 1):
                parts = line.rstrip("\n").split("\t")
                if len(parts) != 2 or not parts[0].isdigit() or not parts[1]:
                    E.add("fmt", f"line {ln}: {line!r}")
                    continue
                R["doc_map"][int(parts[0])] = parts[1]
        dm = R["doc_map"]
        chk(S, "O13 each line is `docID<TAB>TREC-ID`", E.ok("fmt"), E.detail("fmt"))
        chk(S, "O14 docIDs are exactly 1..N", sorted(dm) == list(range(1, len(dm) + 1)))
        chk(S, "O15 TREC ids are unique", len(set(dm.values())) == len(dm))
        badt = [x for x in dm.values() if not re.fullmatch(r"clueweb09-[a-z]{2}\d{4}-\d{2}-\d{5}", x)]
        chk(S, "O16 TREC ids look like clueweb09-enXXXX-YY-ZZZZZ", not badt, str(badt[:3]), "WARN")
        ts.note(f"[{S['name']}] documents = {len(dm):,}")
        if EXPECTED_DOCS is not None:
            chk(S, f"O17 document count == {EXPECTED_DOCS:,}", len(dm) == EXPECTED_DOCS, f"got {len(dm)}")
        if EXPECTED_LAST_TREC is not None and dm:
            chk(S, f"O18 last TREC id == {EXPECTED_LAST_TREC}", dm[max(dm)] == EXPECTED_LAST_TREC, dm[max(dm)])

    # ---- postings ----
    ts.section(f"[{S['name']}] postings: `term, df: <doc, tf: <pos, ...>; ...>`")
    t0, E, single = time.time(), ErrBook(), True
    max_doc = max(R["doc_map"]) if R["doc_map"] else 0
    MASK = (1 << 64) - 1
    try:
        for off, rec in iter_records(S["ppath"]):
            R["n_rec"] += 1
            m = REC_RE.fullmatch(rec)
            if not m or not BODY_RE.fullmatch(m.group(3)):
                E.add("fmt", rec[:80])
                continue
            single &= "\n" not in rec
            term, df, body = m.group(1), int(m.group(2)), m.group(3)
            n = R["n_rec"]
            if n <= len(dt) and dt[n - 1] != term:
                E.add("order", f"record {n}: postings {term!r} vs dictionary {dt[n - 1]!r}")
            if term in R["ddf"] and R["ddf"][term] != df:
                E.add("df_dict", f"{term}: postings {df}, dictionary {R['ddf'][term]}")
            if term in R["fp"]:
                E.add("dup", term)
            ents = ENT_RE.findall(body)
            if len(ents) != df:
                E.add("df", f"{term}: header {df}, found {len(ents)}")
            prev, h, npos = 0, 0, 0
            for d_s, tf_s, p_s in ents:
                d, tf = int(d_s), int(tf_s)
                pos = [int(x) for x in p_s.split(", ")]
                if d <= prev:
                    E.add("doc_order", f"{term}: doc {d} after {prev}")
                if max_doc and d > max_doc:
                    E.add("doc_range", f"{term}: doc {d} > {max_doc}")
                if tf != len(pos):
                    E.add("tf", f"{term} doc {d}: tf={tf}, #pos={len(pos)}")
                if any(b <= a for a, b in zip(pos, pos[1:])):
                    E.add("pos_order", f"{term} doc {d}")
                for p in pos:
                    h += hash((d, p))
                npos += len(pos)
                R["ntok"][d] += len(pos)
                if pos[-1] > R["maxpos"].get(d, -1):
                    R["maxpos"][d] = pos[-1]
                if pos[0] < R["minpos"].get(d, 1 << 60):
                    R["minpos"][d] = pos[0]
                prev = d
            R["fp"][term] = (df, npos, h & MASK)
            R["offs"][term] = off
        chk(S, "O19 postings are valid UTF-8", True)
    except UnicodeDecodeError as e:
        chk(S, "O19 postings are valid UTF-8", False, str(e))
    chk(S, "O20 every record matches `term, df: <doc, tf: <pos, ...>; ...>`", E.ok("fmt"), E.detail("fmt"))
    chk(S, "O21 postings terms == dictionary terms, same order", E.ok("order") and R["n_rec"] == len(dt),
        E.detail("order") or f"{R['n_rec']} records vs {len(dt)} dictionary lines")
    chk(S, "O22 no term repeated in postings", E.ok("dup"), E.detail("dup"))
    chk(S, "O23 record df == dictionary df", E.ok("df_dict"), E.detail("df_dict"))
    chk(S, "O24 df == number of document entries", E.ok("df"), E.detail("df"))
    chk(S, "O25 docIDs strictly increasing within a record", E.ok("doc_order"), E.detail("doc_order"))
    chk(S, "O26 every docID exists in the doc map", E.ok("doc_range"), E.detail("doc_range"))
    chk(S, "O27 tf == number of positions", E.ok("tf"), E.detail("tf"))
    chk(S, "O28 positions strictly increasing", E.ok("pos_order"), E.detail("pos_order"))
    chk(S, f"O29 every position >= pos_base ({S['cfg'].pos_base})",
        all(v >= S["cfg"].pos_base for v in R["minpos"].values()),
        f"min seen {min(R['minpos'].values(), default=None)}")
    ts.note(f"[{S['name']}] {R['n_rec']:,} records, {'single-line' if single else 'multi-line'}, "
            f"{sum(R['ntok'].values()):,} positions, {time.time() - t0:.0f}s")
    return R


LOADED = {S["name"]: load_set(S) for S in SETS}
GOOD_SETS = [S for S in SETS if LOADED[S["name"]]["ok"]]

# ------------------------------------------------------------------
ts.section("Enhanced vs. Baseline: the linguistic switches are visible in the vocabulary")
for S in GOOD_SETS:
    R, cfg = LOADED[S["name"]], S["cfg"]
    vocab = set(R["dterms"])
    if S["kind"] == "enhanced":
        if cfg.case_folding:
            up = [x for x in vocab if x != x.lower()]
            chk(S, "O30 vocabulary is lower case (case_folding=True)", not up, f"{len(up)} found, e.g. {up[:5]}")
        if cfg.stopword_removal:
            sw = sorted(vocab & STOPWORDS)
            if cfg.stemming or cfg.lemmatization:
                # stopwords are removed from surface tokens only; a content word may still stem to a stopword
                # string (others -> other, owned -> own, DOS -> do). 8.2 checks the surface word of each one.
                ts.note(f"[{S['name']}] {len(sw)} stems equal a stopword string (kept on purpose), e.g. {sw[:8]}")
            else:
                chk(S, "O31 no stopword in the vocabulary (stopword_removal=True)", not sw,
                    f"{len(sw)} found, e.g. {sw[:5]}")
    else:
        common = {"to", "the", "of", "and", "in", "a"}
        chk(S, "O32 stopwords are kept (Slide 5 example term `to`)", bool(vocab & common), f"none of {sorted(common)}")
        chk(S, "O33 the unstemmed word `retrieval` is present", "retrieval" in vocab, "not in the vocabulary", "WARN")
        late = [d for d, p in R["minpos"].items() if p != cfg.pos_base]
        chk(S, f"O34 every non-empty document starts at pos_base={cfg.pos_base} (nothing filtered)", not late,
            f"{len(late)} docs, e.g. {late[:5]}")
if "enhanced" in LOADED and "baseline" in LOADED and all(LOADED[k]["ok"] for k in ("enhanced", "baseline")):
    _E, _B = LOADED["enhanced"], LOADED["baseline"]
    ts.ok("[cross] O35 the Baseline index has at least as many positions as the Enhanced index",
          sum(_B["ntok"].values()) >= sum(_E["ntok"].values()))
    bad = [d for d in _E["ntok"] if _B["ntok"].get(d, 0) < _E["ntok"][d] or _B["maxpos"].get(d, -1) < _E["maxpos"][d]]
    ts.ok("[cross] O36 per document: Enhanced positions and last position never exceed Baseline", not bad,
          f"{len(bad)} docs, e.g. {bad[:5]}")
ts.done()

===== 8.1 Output files =====
[INFO] index sets: ['enhanced', 'baseline'] | corpus: e:\ir_hw\claude_hw1\07.warc.gz (found)
[INFO] CFG: include_title=True, pos_base=0, hyphen_mode=split, case_folding=True, stopwords=True, lemmatization=False, stemming=True

--- [enhanced] files ---
[PASS] [enhanced] O01 dictionary exists and is not empty  -> 3,950,621 bytes
[PASS] [enhanced] O01 dictionary (terms only) exists and is not empty  -> 2,862,439 bytes
[PASS] [enhanced] O01 postings exists and is not empty  -> 200,342,775 bytes
[PASS] [enhanced] O01 doc map exists and is not empty  -> 996,254 bytes
[PASS] [enhanced] O02 no UTF-8 BOM
[PASS] [enhanced] O03 LF line endings only (no '\r' anywhere)

--- [enhanced] dictionary ---
[PASS] [enhanced] O04 dictionary is valid UTF-8
[PASS] [enhanced] O05 no blank lines; each line is `term` or `term, df`
[PASS] [enhanced] O06 df values are positive
[PASS] [enhanced] O07 no duplicate terms
[PASS] [enhanced] O08 terms strictly sorted
[PASS] [enhanced] O09 no 

### 8.2 與原始語料比對

不使用 7.3 的中間檔，而是重新解析 `07.warc.gz`、重新做 tokenizer 與 linguistic 處理，算出每個 term 的 df、總 tf，以及所有 (docID, position) 的指紋，與 8.1 讀到的索引檔逐一比對（G01–G08）。另外確認沒有任何索引詞來自原文的停用字（G09）、高 df 的詞裡沒有 HTML / JavaScript 字詞（G10，WARN），以及 doc map 的大小等於原始檔中 `WARC-Type: response` 的行數（G11，與 parser 無關的獨立計數）。

完整語料約需 4 分鐘；設 `RUN_GROUND_TRUTH = False` 可略過。

In [25]:
ts = TestSuite("8.2 Index vs. raw corpus")
# re-parse the raw WARC and compare every posting (about 4 minutes on 07.warc.gz); False skips this cell
RUN_GROUND_TRUTH = True

if RUN_GROUND_TRUTH and HAVE_CORPUS and GOOD_SETS:
    t0 = time.time()
    kinds = {S["kind"]: S["cfg"] for S in GOOD_SETS}
    # per kind: term -> [df, total tf, sum of hash((doc_id, pos)), last doc_id seen]
    GT = {k: dict(gt={}, ntok={}, maxpos={}, minpos={}, sw_leak=ErrBook()) for k in kinds}
    gt_trec = {}
    for doc in parse_warc(CORPUS):
        if MAX_DOCS is not None and doc.doc_id > MAX_DOCS:
            break
        # the tokenizer runs once; each index kind applies its own linguistic Config
        text = f"{doc.title} {doc.body}" if CFG.include_title else doc.body
        tokens = tokenize(text, CFG)
        gt_trec[doc.doc_id] = doc.trec_id
        for k, cfg in kinds.items():
            terms = linguistic(tokens, cfg)
            G = GT[k]
            gt = G["gt"]
            G["ntok"][doc.doc_id] = len(terms)
            G["maxpos"][doc.doc_id] = terms[-1][1] if terms else -1
            G["minpos"][doc.doc_id] = terms[0][1] if terms else -1
            if cfg.stopword_removal:
                # a term equal to a stopword string must come from a content word, never from a surface stopword
                for term, pos in terms:
                    if term.lower() in STOPWORDS:
                        surface = tokens[pos - CFG.pos_base].text
                        if surface.lower() in STOPWORDS:
                            G["sw_leak"].add("sw", f"doc {doc.doc_id} pos {pos}: {surface!r} -> {term!r}")
            for term, pos in terms:
                r = gt.get(term)
                if r is None:
                    r = gt[term] = [0, 0, 0, 0]
                r[1] += 1
                r[2] += hash((doc.doc_id, pos))
                if r[3] != doc.doc_id:
                    r[0] += 1
                    r[3] = doc.doc_id
    ts.note(f"re-parsed {CORPUS}: {len(gt_trec):,} docs in {time.time() - t0:.0f}s")
    MASK = (1 << 64) - 1
    for S in GOOD_SETS:
        R, G = LOADED[S["name"]], GT[S["kind"]]
        gt, fp = G["gt"], R["fp"]
        ts.section(f"[{S['name']}]")
        if R["doc_map"]:
            chk(S, "G01 doc map == corpus (docID -> TREC id)", R["doc_map"] == gt_trec,
                f"doc map {len(R['doc_map'])} vs corpus {len(gt_trec)}")
        miss = [x for x in gt if x not in fp]
        extra = [x for x in fp if x not in gt]
        chk(S, "G02 no term of the corpus is missing", not miss, f"{len(miss)} missing, e.g. {miss[:5]}")
        chk(S, "G03 no term that is not in the corpus", not extra, f"{len(extra)} extra, e.g. {extra[:5]}")
        b_df = [x for x in fp if x in gt and fp[x][0] != gt[x][0]]
        b_n = [x for x in fp if x in gt and fp[x][1] != gt[x][1]]
        b_h = [x for x in fp if x in gt and fp[x][2] != (gt[x][2] & MASK)]
        chk(S, "G04 df correct for every term", not b_df, f"{len(b_df)} wrong, e.g. {b_df[:5]}")
        chk(S, "G05 total tf correct for every term", not b_n, f"{len(b_n)} wrong, e.g. {b_n[:5]}")
        chk(S, "G06 every (term, docID, position) triple is exact", not b_h, f"{len(b_h)} terms differ, e.g. {b_h[:5]}")
        chk(S, "G07 per-document token counts match", dict(R["ntok"]) == {d: n for d, n in G["ntok"].items() if n})
        chk(S, "G08 per-document first / last position match (gaps kept)",
            R["maxpos"] == {d: p for d, p in G["maxpos"].items() if p >= 0}
            and R["minpos"] == {d: p for d, p in G["minpos"].items() if p >= 0})
        if S["cfg"].stopword_removal:
            chk(S, "G09 no indexed term comes from a surface stopword (stopwords removed before stemming)",
                G["sw_leak"].ok("sw"), G["sw_leak"].detail("sw"))
        top = sorted(gt, key=lambda x: -gt[x][0])
        ts.note(f"[{S['name']}] top-10 df: " + ", ".join(f"{x}({gt[x][0]})" for x in top[:10]))
        junk = {"var", "function", "div", "px", "css", "nbsp", "javascript", "href", "style"}
        hit = [x for x in top[:100] if x.lower() in junk]
        chk(S, "G10 no HTML / JavaScript words among the 100 highest-df terms", not hit, str(hit), "WARN")
    if MAX_DOCS is None:
        # independent count: lines that start a response record in the raw file
        opener = gzip.open if open(CORPUS, "rb").read(2) == b"\x1f\x8b" else open
        with opener(CORPUS, "rb") as f:
            n_resp = sum(1 for line in f if line.startswith(b"WARC-Type: response"))
        for S in GOOD_SETS:
            if LOADED[S["name"]]["doc_map"]:
                chk(S, "G11 doc map size == number of `WARC-Type: response` lines in the raw file",
                    len(LOADED[S["name"]]["doc_map"]) == n_resp,
                    f"doc map {len(LOADED[S['name']]['doc_map'])}, raw {n_resp}", info=f"{n_resp:,}")
else:
    ts.note("skipped (RUN_GROUND_TRUTH=False, corpus missing, or no complete index set)")
ts.done()

===== 8.2 Index vs. raw corpus =====
[INFO] re-parsed e:\ir_hw\claude_hw1\07.warc.gz: 31,480 docs in 249s

--- [enhanced] ---
[PASS] [enhanced] G01 doc map == corpus (docID -> TREC id)
[PASS] [enhanced] G02 no term of the corpus is missing
[PASS] [enhanced] G03 no term that is not in the corpus
[PASS] [enhanced] G04 df correct for every term
[PASS] [enhanced] G05 total tf correct for every term
[PASS] [enhanced] G06 every (term, docID, position) triple is exact
[PASS] [enhanced] G07 per-document token counts match
[PASS] [enhanced] G08 per-document first / last position match (gaps kept)
[PASS] [enhanced] G09 no indexed term comes from a surface stopword (stopwords removed before stemming)
[INFO] [enhanced] top-10 df: home(20755), 1(18714), contact(17485), us(17022), 2008(16996), new(16590), right(16092), 2(16062), use(15935), 3(15153)
[PASS] [enhanced] G10 no HTML / JavaScript words among the 100 highest-df terms

--- [baseline] ---
[PASS] [baseline] G01 doc map == corpus (docID -> TR

### 8.3 真實語料切片

用語料的前 `SLICE_DOCS = 300` 篇：兩次建索引的輸出 SHA-256 相同（D01）、SPIMI 區塊大小不影響輸出（D02）、記憶體內 / SPIMI / 串流 / 暴力法四者一致（D03）、`save_files` 與串流寫出的檔案相同（D04），以及串流 SPIMI 的記憶體峰值低於記憶體內建法（D05，WARN）。最後印出每個 position 在 `postings.txt` 中平均佔幾個 byte。

In [26]:
ts = TestSuite("8.3 Real-corpus slice")
# number of documents from the start of the corpus used here
SLICE_DOCS = 300

if HAVE_CORPUS:
    slice_docs = list(iter_processed_docs(CORPUS, CFG, max_docs=SLICE_DOCS))
    _dir = tempfile.mkdtemp(dir=TEST_TMP)
    P = lambda name: os.path.join(_dir, name)
    with_cfg = lambda **kw: Config(**{**CFG.__dict__, **kw})

    def build(tag: str, limit: int):
        return PositionalIndex.build_and_save_streaming(
            iter(slice_docs), P(f"{tag}_d.txt"), P(f"{tag}_p.txt"), P(f"{tag}_m.txt"), P(f"{tag}_t.txt"),
            cfg=with_cfg(spimi_block_limit=limit))

    _, s1 = build("a", 2000)
    build("b", 2000)
    build("c", 10**9)
    same = lambda x, y: all(sha256_file(P(f"{x}_{k}.txt")) == sha256_file(P(f"{y}_{k}.txt")) for k in "dpmt")
    ts.note(f"slice: {s1['num_docs']} docs, {s1['total_tokens']:,} tokens, {s1['vocab_size']:,} terms, "
            f"{s1['num_blocks']} SPIMI blocks")
    ts.ok("D01 two identical runs give byte-identical files (SHA-256)", same("a", "b"))
    ts.ok("D02 the SPIMI block size does not change the output (2000 pairs vs. a single block)", same("a", "c"))
    im = PositionalIndex.build_from_iter(iter(slice_docs), CFG)
    sp = PositionalIndex.build_from_iter(iter(slice_docs), with_cfg(spimi_block_limit=2000), use_spimi=True)
    ts.ok("D03 in-memory == SPIMI == streaming file == brute force",
          flat(im) == flat(sp) == parse_post_file(P("a_p.txt")) == brute_force_index(slice_docs))
    im.save_files(P("x_d.txt"), P("x_p.txt"), P("x_m.txt"), P("x_t.txt"))
    ts.ok("D04 in-memory save_files() writes the same bytes as streaming", same("a", "x"))

    def peak(fn):
        """(peak MB traced by tracemalloc, seconds) of fn()."""
        tracemalloc.start()
        t1 = time.time()
        fn()
        _, pk = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        return pk / 2**20, time.time() - t1

    pk_mem, t_mem = peak(lambda: PositionalIndex.build_from_iter(
        iter_processed_docs(CORPUS, CFG, max_docs=SLICE_DOCS), with_cfg(use_trie=False)))
    pk_str, t_str = peak(lambda: PositionalIndex.build_and_save_streaming(
        iter_processed_docs(CORPUS, CFG, max_docs=SLICE_DOCS), P("q_d.txt"), P("q_p.txt"),
        cfg=with_cfg(spimi_block_limit=2000, use_trie=False)))
    ts.note(f"peak RAM: in-memory {pk_mem:.1f} MB ({t_mem:.1f}s) vs streaming {pk_str:.1f} MB ({t_str:.1f}s)")
    ts.ok("D05 streaming SPIMI needs less peak RAM than the in-memory build", pk_str < pk_mem,
          f"{pk_str:.1f} MB vs {pk_mem:.1f} MB (not meaningful if the slice is tiny)", "WARN")
    ts.note(f"storage: postings.txt = {os.path.getsize(P('a_p.txt')) / max(s1['total_tokens'], 1):.1f} "
            f"bytes per indexed position")
else:
    ts.note("skipped (corpus not found)")
ts.done()

===== 8.3 Real-corpus slice =====
[INFO] slice: 300 docs, 181,137 tokens, 24,112 terms, 70 SPIMI blocks
[PASS] D01 two identical runs give byte-identical files (SHA-256)
[PASS] D02 the SPIMI block size does not change the output (2000 pairs vs. a single block)
[PASS] D03 in-memory == SPIMI == streaming file == brute force
[PASS] D04 in-memory save_files() writes the same bytes as streaming
[INFO] peak RAM: in-memory 27.6 MB (6.6s) vs streaming 10.6 MB (11.7s)
[PASS] D05 streaming SPIMI needs less peak RAM than the in-memory build
[INFO] storage: postings.txt = 11.5 bytes per indexed position

8.3 Real-corpus slice: PASS=5  WARN=0  FAIL=0


### 8.4 記憶體中的索引物件

7.4 執行後，`clueweb_idx`、`base_idx` 仍在記憶體中（trie 存的是 byte offset）。這裡確認它們的統計數字與檔案一致（M01–M03）、trie 的 df 與 byte offset 正確（M04，抽 2 萬個 term）、高 df 詞的 AND 查詢等於集合交集（M05，只用經過處理後不變的詞當查詢詞，例如 `home`），並從前 200 篇文件隨機取連續 3 個 token 當 phrase query，必須找回原本的位置（M06）。若 kernel 重新啟動過、沒有執行 7.4，這一格會略過。

In [27]:
ts = TestSuite("8.4 Index objects from 7.4")
# number of random phrase queries per index
N_PHRASES = 10
# the first 200 documents of the corpus, source of the phrase queries
_docs_200 = []
if HAVE_CORPUS:
    for _d in parse_warc(CORPUS):
        if _d.doc_id > 200:
            break
        _docs_200.append(_d)
_found = False
for var, stats_var, S_name in (("clueweb_idx", "cw_stats", "enhanced"), ("base_idx", "base_stats", "baseline")):
    if var not in globals() or S_name not in LOADED or not LOADED[S_name]["ok"]:
        continue
    _found = True
    ix, R = globals()[var], LOADED[S_name]
    S = next(s for s in SETS if s["name"] == S_name)
    ts.section(f"[{S_name}] `{var}` vs. the files on disk")
    stats = globals().get(stats_var, {})
    if stats:
        chk(S, "M01 stats: total_tokens == positions in postings.txt", stats["total_tokens"] == sum(R["ntok"].values()),
            f"{stats['total_tokens']} vs {sum(R['ntok'].values())}")
        chk(S, "M02 stats: vocab_size == dictionary lines", stats["vocab_size"] == len(R["dterms"]))
        if R["doc_map"]:
            chk(S, "M03 stats: num_docs == doc map lines", stats["num_docs"] == len(R["doc_map"]))
    rr = random.Random(SEED)
    if ix.trie is not None:
        E = ErrBook()
        for term in rr.sample(R["dterms"], min(20000, len(R["dterms"]))):
            res = ix.trie.search(term)
            if res is None or res[0] != R["fp"][term][0]:
                E.add("df", term)
            elif isinstance(res[1], int) and res[1] != R["offs"][term]:
                E.add("off", term)
        chk(S, "M04 trie: df and byte offset of 20,000 sampled terms", E.ok("df") and E.ok("off"),
            E.detail("df") or E.detail("off"))
        # AND of frequent terms (long postings lists, so the skip pointers are used). Query words are normalized,
        # so only terms that the pipeline maps to themselves (e.g. "home", not "inform") are used as query words.
        pool = [x for x in sorted(R["dterms"], key=lambda x: -R["fp"][x][0])[:300]
                if words(linguistic(tokenize(x, ix.cfg), ix.cfg)) == [x]][:100]
        E = ErrBook()
        for _ in range(20):
            a, b = rr.sample(pool, 2)
            pa, pb = ix.get_postings(a), ix.get_postings(b)
            if len(pa) != R["fp"][a][0]:
                E.add("len", a)
            if ix.intersect_with_skips(a, b)[0] != sorted({p.doc_id for p in pa} & {p.doc_id for p in pb}):
                E.add("and", f"{a} AND {b}")
        chk(S, "M05 AND with skip pointers on 20 pairs of high-df terms == set intersection",
            E.ok("and") and E.ok("len"), E.detail("and") or E.detail("len"))
    # phrase queries built from 3 consecutive tokens of random documents must find that place again
    E, n_q = ErrBook(), 0
    for _ in range(N_PHRASES * 5):
        if n_q == N_PHRASES or not _docs_200:
            break
        d = rr.choice(_docs_200)
        tokens = tokenize(f"{d.title} {d.body}" if CFG.include_title else d.body, CFG)
        if len(tokens) < 3:
            continue
        i = rr.randrange(len(tokens) - 2)
        window = tokens[i:i + 3]
        query = " ".join(x.text for x in window)
        # the query must tokenize back to the same 3 tokens and keep at least one term
        expect = linguistic(window, ix.cfg)
        if [x.text for x in tokenize(query, CFG)] != [x.text for x in window] or not expect:
            continue
        n_q += 1
        hits = dict(ix.phrase_query(query))
        if expect[0][1] not in hits.get(d.doc_id, []):
            E.add("phrase", f"{query!r} at doc {d.doc_id} pos {expect[0][1]}")
    chk(S, f"M06 {n_q} phrase queries taken from random documents find their own position", E.ok("phrase") and n_q > 0,
        E.detail("phrase") or "no query could be built")
if not _found:
    ts.note("skipped: run 7.4 in this kernel first (it keeps clueweb_idx and base_idx in memory)")
ts.done()

===== 8.4 Index objects from 7.4 =====

--- [enhanced] `clueweb_idx` vs. the files on disk ---
[PASS] [enhanced] M01 stats: total_tokens == positions in postings.txt
[PASS] [enhanced] M02 stats: vocab_size == dictionary lines
[PASS] [enhanced] M03 stats: num_docs == doc map lines
[PASS] [enhanced] M04 trie: df and byte offset of 20,000 sampled terms
[PASS] [enhanced] M05 AND with skip pointers on 20 pairs of high-df terms == set intersection
[PASS] [enhanced] M06 10 phrase queries taken from random documents find their own position

--- [baseline] `base_idx` vs. the files on disk ---
[PASS] [baseline] M01 stats: total_tokens == positions in postings.txt
[PASS] [baseline] M02 stats: vocab_size == dictionary lines
[PASS] [baseline] M03 stats: num_docs == doc map lines
[PASS] [baseline] M04 trie: df and byte offset of 20,000 sampled terms
[PASS] [baseline] M05 AND with skip pointers on 20 pairs of high-df terms == set intersection
[PASS] [baseline] M06 10 phrase queries taken from random 

### 8.5 與獨立的 HTML 解析器比對

8.2 重新解析語料時用的是同一套 `html_to_text`，所以抽字本身的錯（截斷、黏字、entity 外洩）在兩邊會一模一樣，8.2 看不出來。這一格從語料隨機抽 500 篇，用 Python 內建的 `html.parser` 另寫一個抽取器（與 lxml、`html_to_text` 沒有共用程式碼；同樣略過 `<head>`、script、style、noscript、註解，同樣每個 tag 都分隔字），再把兩邊的文字切成一樣的「字母 / 數字連續段」比較。解碼後的 HTML 兩邊共用，因為編碼判斷已由 6.1 的 P11、P16、P20、P26 測試。

| 編號 | 檢查內容 |
|---|---|
| X01 | 每篇的字回收率（獨立抽取器的字有多少也出現在我們的文字裡）的中位數 ≥ 0.95 |
| X02 | 字少掉一半以上的頁面（截斷、整段被吃掉）不超過 1%（WARN） |
| X03 | 黏字：我們的某個字剛好是對方兩個相鄰字接起來（`HomeAbout`、`dont`），不超過總字數的 0.05% |
| X04 | 拆字：對方的某個字在我們這邊被拆成兩個相鄰字，不超過 0.1%（WARN） |
| X05 | `nbsp`、`amp`、`quot`、`lt`、`gt` 這些 entity 名稱沒有變成字 |

這個檢查找到了兩個 8.2 看不出來的問題：曾經嘗試的「inline tag 內文字相連」產生大量黏字（`vacationsalpha`、`hotelsbeta`，已撤回），以及低位元組的引號亂碼讓 `don't` 變成 `dont`（已修正，見 P27）。

In [28]:
ts = TestSuite("8.5 Independent HTML extraction")
# random documents compared with an extractor that shares no code with html_to_text / lxml
N_SAMPLE = 500
from html.parser import HTMLParser


class IndependentExtractor(HTMLParser):
    """Visible text with Python's html.parser (HTML5 entity rules): <title> text, then everything outside
    <head>, <script>, <style>, <noscript> and comments. Every tag separates words."""
    SKIP = {"script", "style", "noscript"}

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.title, self.body, self.skip, self.in_head, self.in_title = [], [], 0, False, False

    def handle_starttag(self, tag, attrs):
        if tag in self.SKIP:
            self.skip += 1
        elif tag == "head":
            self.in_head = True
        elif tag == "title":
            self.in_title = True
        elif tag == "body":
            self.in_head = False
        (self.title if self.in_title else self.body).append(" ")

    def handle_endtag(self, tag):
        if tag in self.SKIP:
            self.skip = max(0, self.skip - 1)
        elif tag == "head":
            self.in_head = False
        elif tag == "title":
            self.in_title = False
        (self.title if self.in_title else self.body).append(" ")

    def handle_data(self, data):
        if self.skip:
            return
        if self.in_title:
            self.title.append(data)
        elif not self.in_head:
            self.body.append(data)


def independent_text(html_str: str) -> str:
    x = IndependentExtractor()
    try:
        x.feed(html_str)
        x.close()
    except Exception:
        pass
    return "".join(x.title) + " " + "".join(x.body)


# both texts are cut into the same simple word runs (letters / digits), so only the extraction is compared
_RUN_RE = re.compile(r"[^\W_]+")
runs = lambda text: _RUN_RE.findall(text.lower())

if HAVE_CORPUS:
    t0 = time.time()
    n_docs = len(LOADED[GOOD_SETS[0]["name"]]["doc_map"]) if GOOD_SETS and LOADED[GOOD_SETS[0]["name"]]["doc_map"] else 0
    rng_x = random.Random(SEED)
    sample = set(rng_x.sample(range(1, n_docs + 1), min(N_SAMPLE, n_docs))) if n_docs else set()
    recalls, low, glued, split, n_ours, n_ind, leaked = [], [], Counter(), Counter(), 0, 0, Counter()
    doc_id = 0
    for rec in iter_warc_records(CORPUS):
        if rec.headers.get("WARC-Type") != "response":
            continue
        doc_id += 1
        if doc_id not in sample:
            continue
        # the decoded HTML is shared on purpose: charset handling is tested in 6.1 (P11, P16, P20, P26)
        http_header, payload = split_http(rec.content)
        charset = _detect_charset(http_header, payload)
        html_str = payload.decode(charset, errors="cp1252_fallback" if codecs.lookup(charset).name == "utf-8" else "replace")
        title, body = html_to_text(html_str) if html_str.strip() else ("", "")
        ours, ind = runs(f"{title} {body}"), runs(independent_text(html_str))
        n_ours += len(ours)
        n_ind += len(ind)
        if ind:
            common = sum((Counter(ours) & Counter(ind)).values())
            recalls.append(common / len(ind))
            if recalls[-1] < 0.5 and len(ind) >= 20:
                low.append((doc_id, len(ours), len(ind)))
        ours_set, ind_set = set(ours), set(ind)
        ind_pairs = {a + b for a, b in zip(ind, ind[1:])}
        ours_pairs = {a + b for a, b in zip(ours, ours[1:])}
        # a word of ours that is two neighbouring words of the reference glued together, and the reverse
        for w in ours:
            if w not in ind_set and w in ind_pairs:
                glued[w] += 1
        for w in ind:
            if w not in ours_set and w in ours_pairs:
                split[w] += 1
        for w in ("nbsp", "amp", "quot", "lt", "gt"):
            leaked[w] += max(0, ours.count(w) - ind.count(w))
    recalls.sort()
    med = recalls[len(recalls) // 2] if recalls else 0.0
    ts.note(f"{len(sample)} random documents in {time.time() - t0:.0f}s: {n_ours:,} words (ours) vs {n_ind:,} "
            f"(independent); median word recall {med:.3f}, 5th percentile {recalls[len(recalls) // 20] if recalls else 0:.3f}")
    ts.ok("X01 median word recall against the independent extractor >= 0.95", med >= 0.95, f"{med:.3f}", info=f"{med:.3f}")
    ts.ok("X02 at most 1% of the pages lose more than half of their words (truncated or swallowed text)",
          len(low) <= 0.01 * len(sample), f"{len(low)} pages, e.g. (doc, ours, reference) {low[:5]}",
          level="WARN", info=f"{len(low)} pages")
    g, s = sum(glued.values()), sum(split.values())
    ts.ok("X03 glued words (two neighbouring words merged into one) <= 0.05% of the words", g <= 0.0005 * max(n_ind, 1),
          f"{g} occurrences, e.g. {glued.most_common(8)}", info=f"{g} occurrences, e.g. {glued.most_common(5)}")
    ts.ok("X04 split words (one word cut into two) <= 0.1% of the words", s <= 0.001 * max(n_ind, 1),
          f"{s} occurrences, e.g. {split.most_common(8)}", level="WARN", info=f"{s} occurrences, e.g. {split.most_common(5)}")
    ts.ok("X05 no HTML entity names leak into the text (nbsp, amp, quot, lt, gt)", sum(leaked.values()) <= 5,
          f"{dict(leaked)}", info=f"{dict(leaked)}")
else:
    ts.note("skipped (corpus not found)")
ts.done()

===== 8.5 Independent HTML extraction =====
[INFO] 500 random documents in 8s: 439,312 words (ours) vs 440,458 (independent); median word recall 1.000, 5th percentile 0.984
[PASS] X01 median word recall against the independent extractor >= 0.95  -> 1.000
[PASS] X02 at most 1% of the pages lose more than half of their words (truncated or swallowed text)  -> 0 pages
[PASS] X03 glued words (two neighbouring words merged into one) <= 0.05% of the words  -> 46 occurrences, e.g. [('¹psž', 4), ('ì5æ', 2), ('ˆô', 2), ('2008t99å', 1), ('fedoraïåç', 1)]
[PASS] X04 split words (one word cut into two) <= 0.1% of the words  -> 17 occurrences, e.g. [('bobmarley', 10), ('caribbeangraphics', 5), ('kmh', 1), ('marley1', 1)]
[PASS] X05 no HTML entity names leak into the text (nbsp, amp, quot, lt, gt)  -> {'nbsp': 0, 'amp': 0, 'quot': 0, 'lt': 0, 'gt': 0}

8.5 Independent HTML extraction: PASS=5  WARN=0  FAIL=0


### 8.6 測試總結

列出本次 kernel 執行過的每個測試格的 PASS / WARN / FAIL / SKIP 數量，並依第 6 節的「作業要求 → 檢查」表，印出每條作業要求的檢查結果（沒有執行的檢查顯示 `not run`）。

**目前預期的 WARN**：L17（WordNet 把 `species` 還原成 `specie`；lemmatization 預設關閉，不影響索引）。

In [29]:
# Summary of every test cell run in this kernel (Sections 6 and 8), then the requirement coverage table
suites = list(dict.fromkeys(r[0] for r in TEST_RESULTS))
print(f"{'suite':<40}{'PASS':>6}{'WARN':>6}{'FAIL':>6}{'SKIP':>6}")
print("-" * 64)
for name in suites:
    c = Counter(r[3] for r in TEST_RESULTS if r[0] == name)
    print(f"{name:<40}{c['PASS']:>6}{c['WARN']:>6}{c['FAIL']:>6}{c['SKIP']:>6}")
total = Counter(r[3] for r in TEST_RESULTS)
print("-" * 64)
print(f"{'total':<40}{total['PASS']:>6}{total['WARN']:>6}{total['FAIL']:>6}{total['SKIP']:>6}")

print("\nHomework requirement -> checks (status over every check with that id; 'not run' = cell not executed)")
for label, spec in REQUIREMENTS:
    ids = set(_expand_ids(spec))
    st = Counter(r[3] for r in TEST_RESULTS if r[1] in ids)
    ran = st["PASS"] + st["WARN"] + st["FAIL"]
    if not ran:
        status = "skipped" if st["SKIP"] else "not run"
    elif st["FAIL"]:
        status = f"FAIL ({st['FAIL']} of {ran})"
    else:
        extra = [f"{st[k]} {k}" for k in ("WARN", "SKIP") if st[k]]
        status = f"PASS ({ran} checks" + (", " + ", ".join(extra) if extra else "") + ")"
    print(f"  {label:<72}{status}")

for kind, label in (("WARN", "informational, not failures"), ("SKIP", "could not run in this environment")):
    rows = [(s, n) for s, _, n, st in TEST_RESULTS if st == kind]
    if rows:
        print(f"\n{kind} ({label}):")
        for s, n in rows:
            print(f"  {s}: {n}")
failed = [(s, n) for s, _, n, st in TEST_RESULTS if st == "FAIL"]
if failed:
    raise AssertionError(f"{len(failed)} check(s) failed: {failed[:5]}")
print("\nALL REQUIRED CHECKS PASSED")

suite                                     PASS  WARN  FAIL  SKIP
----------------------------------------------------------------
6.1 Parser                                  27     0     0     0
6.2 Tokenizer                               25     0     0     0
6.3 Linguistic module                       16     1     0     0
6.4 Pipeline API & intermediate files       23     0     0     0
6.5 Indexer                                 20     0     0     0
6.6 Bonus features                          19     0     0     0
6.7 Command line (run_cli)                  22     0     0     0
8.1 Output files                            70     0     0     0
8.2 Index vs. raw corpus                    21     0     0     0
8.3 Real-corpus slice                        5     0     0     0
8.4 Index objects from 7.4                  12     0     0     0
8.5 Independent HTML extraction              5     0     0     0
----------------------------------------------------------------
total                    